# VN QUANT RESEARCH ENGINE v2 — Phase 2: Data Architecture

Evolving from `Tool_CK_Claude_v1.ipynb` (frozen as `Tool_CK_Claude_v1_baseline.ipynb`)
per `correctness_report.md`.

**This notebook covers Phase 2 only** (per the implementation order in the spec):
data architecture + the two correctness fixes flagged as highest priority in Phase 1.
Phases 3+ (research dataset, triple-barrier labeling, factor engine, probability
model, walk-forward validation, ablation) are separate follow-up notebooks/cells —
see the roadmap at the bottom.

**What's in this notebook:**
1. `TradingCalendar` — data-driven trading-session calendar (fixes bug #9)
2. `MarketDataStore` — replaces `DataLayer`; arbitrary date-range fetch, parquet
   cache, and **no more ffill/bfill across missing sessions** (fixes bug #2)
3. `FillSimulator` + `simulate_trade_v2` — realistic order-fill simulation
   (fixes bug #1, the highest-priority finding from Phase 1)
4. API key handled via environment variable / Colab secret, never hardcoded
   (fixes bug #7 — **you should still rotate the key that was in v1**, since
   it already left your machine in that notebook)
5. Unit tests for all of the above — not just "it ran once," each fix has an
   explicit test proving the specific bug it's meant to close


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════

!pip install -q \
    vnstock \
    pandas-ta \
    scikit-learn \
    "pandas==2.2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 61.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install vnstock

  Using cached vnstock-4.0.7-py3-none-any.whl.metadata (20 kB)
  Using cached vnai-2.5.9-py3-none-any.whl.metadata (691 bytes)
  Using cached vnstock_ezchart-1.0.2-py3-none-any.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.8/275.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.2 MB/s eta 0:00:00


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — VERIFY ENVIRONMENT                               ║
# ╚══════════════════════════════════════════════════════════════╝

import sys
import numpy as np
import pandas as pd
import scipy
import sklearn

print("=" * 50)
print("ENVIRONMENT")
print("=" * 50)

print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"SciPy        : {scipy.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")

print("=" * 50)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

print("✅ NumPy OK")
print("✅ SciPy OK")
print("✅ Scikit-learn OK")

ENVIRONMENT
Python       : 3.13.15
NumPy        : 2.1.3
Pandas       : 2.2.3
SciPy        : 1.16.3
Scikit-learn : 1.6.1
✅ NumPy OK
✅ SciPy OK
✅ Scikit-learn OK


In [ ]:
import warnings, os, time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Optional
from dataclasses import dataclass
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

FEATURE_VERSION = "2.0.0-phase2"
CONFIG_VERSION  = "research_v1"


## Secrets / config

Fix for bug #7: the v1 notebook had `register_user('vnstock_11e0b7...')` hardcoded
in Cell 1. That key is compromised the moment it appears in a shared file —
**rotate it in your vnstock account regardless of whether you adopt this notebook.**

Going forward, set it as an environment variable before running this notebook,
or as a Colab secret (`from google.colab import userdata`).

In [ ]:
VNSTOCK_API_KEY = os.environ.get("VNSTOCK_API_KEY")

if VNSTOCK_API_KEY is None:
    try:
        from google.colab import userdata  # type: ignore
        VNSTOCK_API_KEY = userdata.get("VNSTOCK_API_KEY")
    except Exception:
        pass

if VNSTOCK_API_KEY:
    from vnstock import register_user
    register_user(VNSTOCK_API_KEY)
    print("vnstock registered from environment/secret (key not printed).")
else:
    print("⚠️  VNSTOCK_API_KEY not set. Set it as an env var or Colab secret before fetching data.")


⚠️  VNSTOCK_API_KEY not set. Set it as an env var or Colab secret before fetching data.


## `QuantConfig` (subset needed for Phase 2)

Kept minimal here — the full config from v1 (scoring weights, Ichimoku params, etc.)
still applies unchanged and isn't reproduced in this notebook; only the fields
Phase 2 code needs are shown.

In [ ]:
@dataclass
class QuantConfig:
    sources: list = None
    delay_sec: float = 1.0

    def __post_init__(self):
        if self.sources is None:
            self.sources = ["KBS", "VCI"]


## 1. `TradingCalendar`

**Design choice:** rather than hardcoding a VN public-holiday table (which needs
yearly maintenance and misses ad-hoc exchange closures), the calendar is *built
from an observed reference series of actual trading sessions* — VNINDEX prints
on every real HOSE trading day, so it's the natural source of truth. This also
means the calendar automatically self-corrects if vnstock's own holiday handling
changes.

`sessions_between`, `add_sessions`, and `slice_sessions` give every other
component (labeling, backtesting, days-held) a single place to ask "how many
real trading days between these two dates" instead of doing calendar-day math
that silently drifts across holidays.

In [ ]:
class TradingCalendar:
    """
    Data-driven trading calendar. Build with TradingCalendar.from_reference_ohlcv(vnindex_df).
    """
    def __init__(self, session_dates):
        s = pd.to_datetime(pd.Series(session_dates)).dt.normalize().sort_values().drop_duplicates()
        self._sessions = s.reset_index(drop=True)
        self._idx = pd.Series(range(len(self._sessions)), index=self._sessions.values)

    @classmethod
    def from_reference_ohlcv(cls, df_reference: pd.DataFrame, time_col: str = "time"):
        """df_reference should be VNINDEX (or VN30) OHLCV - every row is a real session."""
        return cls(df_reference[time_col])

    @property
    def sessions(self) -> pd.DatetimeIndex:
        return pd.DatetimeIndex(self._sessions)

    def is_session(self, date) -> bool:
        return pd.Timestamp(date).normalize() in self._idx.index

    def session_index(self, date) -> Optional[int]:
        d = pd.Timestamp(date).normalize()
        return int(self._idx.loc[d]) if d in self._idx.index else None

    def sessions_between(self, start, end) -> int:
        """Trading sessions strictly between start and end (exclusive-exclusive)."""
        i0, i1 = self.session_index(start), self.session_index(end)
        if i0 is None or i1 is None:
            mask = (self._sessions > pd.Timestamp(start)) & (self._sessions < pd.Timestamp(end))
            return int(mask.sum())
        return max(0, i1 - i0 - 1)

    def add_sessions(self, date, n: int):
        """Date that is n trading sessions after `date` (n can be negative)."""
        i = self.session_index(date)
        if i is None:
            after = self._sessions[self._sessions > pd.Timestamp(date)]
            if after.empty:
                return None
            i = self._idx.loc[after.iloc[0]] - 1
        j = i + n
        return self._sessions.iloc[j] if 0 <= j < len(self._sessions) else None

    def slice_sessions(self, df: pd.DataFrame, time_col: str, as_of, n_back: int) -> pd.DataFrame:
        """Last n_back REAL sessions up to and including as_of - can only ever
        select rows that already exist in df, so no synthetic rows are possible."""
        sub = df[df[time_col] <= pd.Timestamp(as_of)]
        return sub.tail(n_back).reset_index(drop=True)


In [ ]:
# ── Unit tests: TradingCalendar ──────────────────────────────────
def test_trading_calendar():
    sessions = pd.bdate_range("2024-01-01", "2024-01-31")
    sessions = sessions.drop(sessions[5])  # simulate a holiday mid-month
    cal = TradingCalendar(sessions)

    assert cal.is_session(sessions[0])
    assert not cal.is_session(sessions[0] + pd.Timedelta(days=200)), "should reject a non-session date"
    assert cal.session_index(sessions[3]) == 3
    assert cal.sessions_between(sessions[0], sessions[3]) == 2

    # slice_sessions can never fabricate a row - test with a df matching sessions exactly
    df = pd.DataFrame({"time": sessions, "close": range(len(sessions))})
    sl = cal.slice_sessions(df, "time", sessions[10], n_back=5)
    assert len(sl) == 5
    assert set(sl["time"]).issubset(set(sessions)), "slice_sessions must only return real sessions"
    print("✓ TradingCalendar tests passed")

test_trading_calendar()


✓ TradingCalendar tests passed


## 2. `MarketDataStore` (replaces `DataLayer`)

Kept as `LEGACY_DataLayer` reference in `Tool_CK_Claude_v1_baseline.ipynb` — not
deleted, per the spec's legacy-compatibility requirement. `MarketDataStore` is
the Phase 2 replacement:

- `fetch(symbol, start_date, end_date)` — arbitrary date ranges (v1 only supported
  "last N days"), which Phase 3's `ResearchDatasetBuilder` will need to pull
  historical snapshots.
- `_clean()` **no longer calls `.ffill().bfill()` across dates.** A missing
  session produces no row — full stop. (This is the fix for bug #2. If vnstock's
  raw response is already gap-free this is a no-op; if it isn't, v1 was silently
  manufacturing candles for non-trading days and feeding them into every
  indicator downstream.)
- Local parquet cache per symbol, so repeated research runs don't re-hit the API.
- `validate()` returns a report instead of silently passing bad data through.

In [ ]:
class MarketDataStore:
    def __init__(self, cfg: QuantConfig, cache_dir: str = "data/raw"):
        self.cfg = cfg
        self.cache_dir = cache_dir

    def fetch(self, symbol: str, start_date: str, end_date: str, use_cache: bool = True) -> Optional[pd.DataFrame]:
        cached = self._read_cache(symbol) if use_cache else None
        if cached is not None and not cached.empty:
            need_start, need_end = pd.Timestamp(start_date), pd.Timestamp(end_date)
            if cached["time"].min() <= need_start and cached["time"].max() >= need_end:
                sub = cached[(cached["time"] >= need_start) & (cached["time"] <= need_end)]
                return sub.reset_index(drop=True)

        raw = None
        for src in self.cfg.sources:
            for fn in (self._fetch_v4, self._fetch_v3):
                try:
                    raw = fn(symbol, start_date, end_date, src)
                    if raw is not None:
                        break
                except Exception:
                    continue
            if raw is not None:
                break
        if raw is None:
            return None

        clean = self._clean(raw)
        if clean is None or clean.empty:
            return None
        if use_cache:
            self._write_cache(symbol, clean)

        sub = clean[(clean["time"] >= pd.Timestamp(start_date)) & (clean["time"] <= pd.Timestamp(end_date))]
        return sub.reset_index(drop=True)

    def _fetch_v4(self, sym, start, end, src):
          from vnstock import Quote

          q = Quote(
              symbol=sym,
              source=src
          )

          # IMPORTANT:
          # vnstock Quote.history() treats the end boundary
          # effectively as exclusive in some versions/providers.
          #
          # Therefore:
          # requested end = 2026-08-07
          # API end       = 2026-08-08
          #
          # After MarketDataStore.fetch() applies
          # <= requested end, the final dataset still ends
          # exactly at 2026-08-07.
          api_end = (
              pd.Timestamp(end)
              + pd.Timedelta(days=1)
          ).strftime("%Y-%m-%d")

          api_start = pd.Timestamp(start).strftime(
              "%Y-%m-%d"
          )

          try:
              return q.history(
                  start=api_start,
                  end=api_end,
                  interval="1D"
              )

          except TypeError:

              return q.history(
                  start_date=api_start,
                  end_date=api_end,
                  interval="1D"
              )


    def _fetch_v3(self, sym, start, end, src):
          from vnstock import Vnstock

          stk = Vnstock().stock(
              symbol=sym,
              source=src
          )

          # Same end-date correction as _fetch_v4.
          api_end = (
              pd.Timestamp(end)
              + pd.Timedelta(days=1)
          ).strftime("%Y-%m-%d")

          api_start = pd.Timestamp(start).strftime(
              "%Y-%m-%d"
          )

          try:
              return stk.quote.history(
                  start=api_start,
                  end=api_end,
                  interval="1D"
              )

          except TypeError:

              return stk.quote.history(
                  start_date=api_start,
                  end_date=api_end,
                  interval="1D"
              )

    def _clean(self, df: pd.DataFrame) -> Optional[pd.DataFrame]:
        """No ffill/bfill across dates - a missing session is simply absent, not synthesized."""
        df = df.copy()
        df.columns = [c.lower().strip() for c in df.columns]
        df = df.rename(columns={
            "tradingdate": "time", "date": "time",
            "o": "open", "h": "high", "l": "low", "c": "close", "v": "volume",
        })
        required = {"time", "open", "high", "low", "close", "volume"}
        if not required.issubset(df.columns):
            return None

        df["time"] = pd.to_datetime(df["time"]).dt.normalize()
        for col in ["open", "high", "low", "close", "volume"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        before = len(df)
        df = df.dropna(subset=["open", "high", "low", "close"])
        df = df[(df["close"] > 0) & (df["high"] >= df["low"])]
        df["volume"] = df["volume"].fillna(0)
        df.attrs["rows_dropped_as_invalid"] = before - len(df)

        df = df.drop_duplicates(subset="time", keep="last")
        return df.sort_values("time").reset_index(drop=True)

    def validate(self, df: pd.DataFrame, calendar: "TradingCalendar" = None) -> dict:
        report = {"n_rows": 0 if df is None else len(df), "issues": []}
        if df is None or df.empty:
            report["issues"].append("empty_dataframe")
            report["ok"] = False
            return report
        if df["time"].duplicated().any():
            report["issues"].append("duplicate_dates")
        if not df["time"].is_monotonic_increasing:
            report["issues"].append("dates_not_sorted")
        if (df["high"] < df["low"]).any():
            report["issues"].append("invalid_ohlc_high_lt_low")
        if (df["close"] <= 0).any():
            report["issues"].append("non_positive_close")
        if calendar is not None:
            for i in range(1, len(df)):
                gap = calendar.sessions_between(df["time"].iloc[i-1], df["time"].iloc[i])
                if gap > 0:
                    report["issues"].append(
                        f"missing_{gap}_session(s)_between_{df['time'].iloc[i-1].date()}_and_{df['time'].iloc[i].date()}"
                    )
        report["ok"] = len(report["issues"]) == 0
        return report

    def _cache_path(self, symbol):
        os.makedirs(self.cache_dir, exist_ok=True)
        return f"{self.cache_dir}/{symbol}.parquet"

    def _read_cache(self, symbol):
        p = self._cache_path(symbol)
        if os.path.exists(p):
            try:
                return pd.read_parquet(p)
            except Exception:
                return None
        return None

    def _write_cache(self, symbol, df):
        try:
            df.to_parquet(self._cache_path(symbol), index=False)
        except Exception:
            pass  # cache is an optimization only, never block on it failing


In [ ]:
# ── Unit tests: MarketDataStore._clean (no network needed - synthetic raw input) ──
def test_clean_no_synthetic_candles():
    cfg = QuantConfig()
    store = MarketDataStore(cfg)

    # Raw input with a GAP (a holiday) - simulate what vnstock might return
    raw = pd.DataFrame({
        "time": ["2024-01-02", "2024-01-03", "2024-01-05"],  # 01-04 missing (holiday)
        "open": [10, 10.5, 11], "high": [10.2, 10.8, 11.2],
        "low": [9.8, 10.3, 10.9], "close": [10.1, 10.6, 11.1], "volume": [1000, 1200, 900],
    })
    clean = store._clean(raw)
    assert len(clean) == 3, "no row should be fabricated for the missing 01-04 session"
    assert clean["close"].tolist() == [10.1, 10.6, 11.1], "no ffill should alter real values"
    assert clean.attrs["rows_dropped_as_invalid"] == 0

    # Raw input with an invalid row (close=0, i.e. bad print, not a holiday) - must be dropped, not filled
    raw2 = pd.DataFrame({
        "time": ["2024-01-02", "2024-01-03"],
        "open": [10, 10.5], "high": [10.2, 10.8], "low": [9.8, 10.3],
        "close": [10.1, 0], "volume": [1000, 1200],
    })
    clean2 = store._clean(raw2)
    assert len(clean2) == 1, "the close=0 row must be dropped, not ffilled into a fake price"
    print("✓ MarketDataStore._clean tests passed (no synthetic candles)")

def test_validate_flags_gaps():
    sessions = pd.bdate_range("2024-01-01", "2024-01-10")
    cal = TradingCalendar(sessions)
    cfg = QuantConfig()
    store = MarketDataStore(cfg)
    df = pd.DataFrame({
        "time": [sessions[0], sessions[1], sessions[4]],  # skips sessions 2,3
        "open": [1,1,1], "high": [1,1,1], "low": [1,1,1], "close": [1,1,1], "volume": [1,1,1],
    })
    report = store.validate(df, calendar=cal)
    assert not report["ok"]
    assert any("missing_2_session" in i for i in report["issues"]), report["issues"]
    print("✓ MarketDataStore.validate gap-detection test passed")

test_clean_no_synthetic_candles()
test_validate_flags_gaps()


✓ MarketDataStore._clean tests passed (no synthetic candles)
✓ MarketDataStore.validate gap-detection test passed


## 3. `FillSimulator` — fixes bug #1 (the highest-priority finding)

`SignalBuilder.smart_entry` in v1 frequently plans a **resting limit price below
the current close** (pullback to Kijun-sen, VAL, EMA20, ATR dip). The v1 backtester
assumed this fills with certainty at the signal date, then scanned forward for
SL/TP. `FillSimulator` fixes that:

- **`chase`** entries (price at/above current close — divergence-confirmed or
  breakout entries meant to be bought immediately) fill at the **next session's
  open**, whatever that actually turns out to be.
- **`limit`** entries (price below current close) only fill on the first future
  session where `low <= entry_price`. On a gap-down through the limit, the fill
  price is the worse of (limit, that session's open) — you never get credited
  with a better fill than the market actually offered.
- If a limit order is **never touched within `max_wait_sessions`**, the result is
  `EXPIRED_UNFILLED` — a real, counted outcome, not a silently dropped one. Silently
  dropping unfilled signals would still bias the stats toward exactly the signals
  that happened to work.

In [ ]:
class FillSimulator:
    def __init__(self, max_wait_sessions: int = 10):
        self.max_wait_sessions = max_wait_sessions

    def classify(self, entry_price: float, close_at_signal: float) -> str:
        return "chase" if entry_price >= close_at_signal * 0.999 else "limit"

    def simulate_fill(self, entry_price: float, close_at_signal: float, df_future: pd.DataFrame):
        if df_future.empty:
            return None
        style = self.classify(entry_price, close_at_signal)
        window = df_future.iloc[: self.max_wait_sessions]

        if style == "chase":
            row = window.iloc[0]
            return {"fill_index": 0, "fill_date": row["time"], "fill_price": float(row["open"])}

        for i, row in window.iterrows():
            if row["low"] <= entry_price:
                fill_price = min(entry_price, row["open"])  # gap-down = worse fill, never better
                return {"fill_index": i - window.index[0], "fill_date": row["time"], "fill_price": float(fill_price)}
        return None  # never filled within the wait window


In [ ]:
# ── Unit tests: FillSimulator ────────────────────────────────────
def _make_future(prices, start="2024-02-01"):
    return pd.DataFrame({
        "time": pd.bdate_range(start, periods=len(prices)),
        "open": [p[0] for p in prices], "high": [p[1] for p in prices],
        "low": [p[2] for p in prices], "close": [p[3] for p in prices],
    })

def test_fill_simulator():
    fs = FillSimulator(max_wait_sessions=5)

    fut = _make_future([(100,101,99,100)]*5)
    assert fs.simulate_fill(90, 100, fut) is None, "limit never touched must return None"

    fut = _make_future([(100,101,99,100),(99,100,95,96),(96,97,93,94)])
    r = fs.simulate_fill(94, 100, fut)
    assert r is not None and r["fill_index"] == 2, r

    fut = _make_future([(85,86,84,85)])  # opens BELOW the limit entirely (gap down)
    r = fs.simulate_fill(90, 100, fut)
    assert r["fill_price"] == 85, "gap-down must fill at the worse open price, not the stale limit"

    fut = _make_future([(102,103,101,102)])
    r = fs.simulate_fill(100.5, 100, fut)
    assert r["fill_price"] == 102, "chase entry must fill at next open, not the theoretical price"

    print("✓ FillSimulator tests passed")

test_fill_simulator()


✓ FillSimulator tests passed


## Integration: `simulate_trade_v2`

Drop-in fix for `QuantBacktester._simulate_trade`. Same signature-level inputs
(entry/SL/TP, future price data), but:

1. Runs the order through `FillSimulator` first — no phantom fills.
2. Measures `days_held` and the T+2 settlement gate using `TradingCalendar`
   sessions from the **fill date**, not the signal date (you can't be in
   settlement for shares you don't own yet) — this also closes bug #9
   (days-held previously relied on `df_future` happening to be gap-free).

In [ ]:
def simulate_trade_v2(symbol, signal_date, entry_price, sl, tp1, close_at_signal,
                       df_future: pd.DataFrame, calendar: TradingCalendar,
                       max_hold_sessions: int = 20, settlement_sessions: int = 2,
                       max_wait_sessions: int = 10) -> dict:
    fs = FillSimulator(max_wait_sessions=max_wait_sessions)
    fill = fs.simulate_fill(entry_price, close_at_signal, df_future)
    if fill is None:
        return {"symbol": symbol, "signal_date": signal_date, "result": "EXPIRED_UNFILLED",
                "pnl_pct": 0.0, "days_held": None}

    fill_price, fill_date = fill["fill_price"], fill["fill_date"]
    after_fill = df_future[df_future["time"] >= fill_date].reset_index(drop=True)

    for _, row in after_fill.iterrows():
        days_held = calendar.sessions_between(fill_date, row["time"]) + 1
        can_exit = days_held >= settlement_sessions
        if can_exit and row["low"] <= sl:
            pnl = (sl - fill_price) / fill_price * 100
            return {"symbol": symbol, "signal_date": signal_date, "fill_date": fill_date,
                     "fill_price": fill_price, "result": "SL", "pnl_pct": pnl, "days_held": days_held}
        if can_exit and row["high"] >= tp1:
            pnl = (tp1 - fill_price) / fill_price * 100
            return {"symbol": symbol, "signal_date": signal_date, "fill_date": fill_date,
                     "fill_price": fill_price, "result": "TP1", "pnl_pct": pnl, "days_held": days_held}
        if days_held >= max_hold_sessions:
            pnl = (row["close"] - fill_price) / fill_price * 100
            return {"symbol": symbol, "signal_date": signal_date, "fill_date": fill_date,
                     "fill_price": fill_price, "result": "TIME_CLOSE", "pnl_pct": pnl, "days_held": days_held}

    if after_fill.empty:
        return {"symbol": symbol, "signal_date": signal_date, "fill_date": fill_date,
                 "fill_price": fill_price, "result": "OPEN_NO_FUTURE_DATA", "pnl_pct": 0.0, "days_held": 0}
    last = after_fill.iloc[-1]
    pnl = (last["close"] - fill_price) / fill_price * 100
    return {"symbol": symbol, "signal_date": signal_date, "fill_date": fill_date,
             "fill_price": fill_price, "result": "OPEN_NO_FUTURE_DATA", "pnl_pct": pnl,
             "days_held": calendar.sessions_between(fill_date, last["time"]) + 1}


In [ ]:
# ── Integration test ─────────────────────────────────────────────
def test_simulate_trade_v2_no_phantom_trade():
    sess = pd.bdate_range("2024-01-01", periods=40)
    cal = TradingCalendar(sess)
    fut = pd.DataFrame({"time": sess[10:20], "open": 100, "high": 101, "low": 99, "close": 100})
    out = simulate_trade_v2("TEST", sess[9], entry_price=90, sl=85, tp1=110,
                              close_at_signal=100, df_future=fut, calendar=cal)
    assert out["result"] == "EXPIRED_UNFILLED", out
    print("✓ simulate_trade_v2 integration test passed:", out)

test_simulate_trade_v2_no_phantom_trade()
print("\n" + "="*60)
print("ALL PHASE 2 TESTS PASSED")
print("="*60)


✓ simulate_trade_v2 integration test passed: {'symbol': 'TEST', 'signal_date': Timestamp('2024-01-12 00:00:00'), 'result': 'EXPIRED_UNFILLED', 'pnl_pct': 0.0, 'days_held': None}

ALL PHASE 2 TESTS PASSED


## Roadmap — what's still ahead

This notebook only covers **Phase 2 (Data Architecture)** plus the two
highest-priority correctness fixes. Per the spec's implementation order, still
to come as separate follow-ups:

- **Phase 3** — `ResearchDatasetBuilder`: historical feature snapshots + forward
  labels (`future_return_3d/5d/10d/20d`), with a look-ahead unit test proving
  features at T don't change when T+1-onward rows are mutated.
- **Phase 4** — `TripleBarrierLabeler` + `EmpiricalProbabilityEngine`: replaces
  the hardcoded `wr_base=45%` and `min_tp_pct=9%` with measured TP/SL/timeout
  frequencies per setup, with sample-size confidence bands.
- **Phase 5–8** — Factor engine (trend/momentum/flow/RS/volatility/structure),
  market breadth, sector engine, setup engine, exit engine — replacing the
  hand-tuned `BASE_WEIGHTS`/`sc_*` coefficients with ablation-tested factors.
- **Phase 9–12** — Probability model (logistic baseline), backtest v2 with fees/
  slippage/price-limits, walk-forward validation, ablation + parameter
  sensitivity.

Full integration of `MarketDataStore`/`TradingCalendar`/`simulate_trade_v2` back
into `QuantEngine`/`QuantBacktester` as drop-in replacements happens once Phase 3's
research dataset exists to validate against — wiring them in earlier would mean
re-touching the same integration point again once labeling changes what "a trade"
even means (triple-barrier vs. fixed SL/TP1).


---
# PHASE 3 — Research Dataset (Features + Forward Labels)

Builds the historical `(symbol, date) → features, labels` dataset the rest of
the research pipeline depends on. Two contracts, enforced by tests below, not
just asserted in comments:

- `feature_at(T)` — computed using information available **through T only**.
- `label_after(T)` — outcome observed **strictly after T**.

**Scope note:** the feature set here (`MinimalFeatureEngine`) is intentionally
small — SMA/EMA, ATR%, volatility, return, volume ratio. Porting the full v1
factor set (VSA, Ichimoku, DMI/Aroon, momentum divergence) into
`feature_at(T)`-safe form is Phase 5 work. Shipping a minimal-but-verified
feature engine now is more useful than a large one that hasn't been proven
leak-free yet — and the leak-free proof (the look-ahead test below) doesn't
get easier by having more features to check.

Labels here are plain forward returns (`future_return_Nd`,
`future_max_return_Nd`, `future_max_drawdown_Nd`) — **not** triple-barrier
TP/SL/timeout labels yet. That's Phase 4 (`TripleBarrierLabeler` +
`EmpiricalProbabilityEngine`), which is what actually replaces the hardcoded
`wr_base=45%` / `min_tp_pct=9%` from the Phase 1 audit.

In [ ]:
class MinimalFeatureEngine:
    """
    Computed only from a DataFrame slice that already ends at T - the caller
    (ResearchDatasetBuilder) is responsible for slicing via
    TradingCalendar.slice_sessions, which structurally cannot return rows
    with time > T.
    """
    MIN_BARS = 60  # need this much history for sma50 etc. to be meaningful

    def feature_at(self, df_slice: pd.DataFrame) -> Optional[dict]:
        if len(df_slice) < self.MIN_BARS:
            return None

        close, high, low, vol = df_slice["close"], df_slice["high"], df_slice["low"], df_slice["volume"]
        ret1 = close.pct_change()
        sma20 = close.rolling(20).mean().iloc[-1]
        sma50 = close.rolling(50).mean().iloc[-1]
        ema20 = close.ewm(span=20, adjust=False).mean().iloc[-1]

        tr = pd.concat([
            high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs(),
        ], axis=1).max(axis=1)
        atr14 = tr.rolling(14).mean().iloc[-1]

        c_now = close.iloc[-1]
        vol_ma20 = vol.rolling(20).mean().iloc[-1]

        return {
            "close": c_now,
            "return_5d": (c_now / close.iloc[-6] - 1) if len(close) > 5 else np.nan,
            "return_20d": (c_now / close.iloc[-21] - 1) if len(close) > 20 else np.nan,
            "volatility_20d": ret1.tail(20).std(),
            "sma20": sma20, "sma50": sma50, "ema20": ema20,
            "price_above_sma20": bool(c_now > sma20),
            "price_above_sma50": bool(c_now > sma50),
            "atr_pct": (atr14 / c_now) if c_now else np.nan,
            "volume_ratio": (vol.iloc[-1] / vol_ma20) if vol_ma20 else np.nan,
        }


In [ ]:
class LabelEngine:
    """
    Horizons counted in real trading sessions via TradingCalendar, not
    calendar days or raw row offsets - keeps this correct even if a symbol's
    dataframe ever has a gap MarketDataStore didn't catch.
    """
    HORIZONS = (3, 5, 10, 20)

    def __init__(self, calendar: "TradingCalendar"):
        self.calendar = calendar

    def label_after(self, df_full: pd.DataFrame, t_date) -> dict:
        row_t = df_full[df_full["time"] == pd.Timestamp(t_date)]
        if row_t.empty:
            return {}
        close_t = float(row_t["close"].iloc[0])
        future = df_full[df_full["time"] > pd.Timestamp(t_date)].reset_index(drop=True)

        out = {}
        for h in self.HORIZONS:
            target_date = self.calendar.add_sessions(t_date, h)
            window = future[future["time"] <= pd.Timestamp(target_date)] if target_date is not None else future.iloc[0:0]

            hit = future[future["time"] == pd.Timestamp(target_date)] if target_date is not None else future.iloc[0:0]
            if not hit.empty:
                out[f"future_return_{h}d"] = float(hit["close"].iloc[0]) / close_t - 1
            else:
                out[f"future_return_{h}d"] = np.nan  # not enough future data YET - NaN, never silently 0

            if not window.empty:
                out[f"future_max_return_{h}d"] = window["high"].max() / close_t - 1
                out[f"future_max_drawdown_{h}d"] = window["low"].min() / close_t - 1
            else:
                out[f"future_max_return_{h}d"] = np.nan
                out[f"future_max_drawdown_{h}d"] = np.nan
        return out


In [ ]:
class ResearchDatasetBuilder:
    """
    For every (symbol, date): features from data through date ONLY, labels
    from data strictly after date. The two steps never share a dataframe
    object with future rows visible to the feature step.
    """
    def __init__(self, calendar: "TradingCalendar", feature_engine=None, label_engine=None,
                 sample_stride: int = 1):
        self.calendar = calendar
        self.features = feature_engine or MinimalFeatureEngine()
        self.labels = label_engine or LabelEngine(calendar)
        self.sample_stride = sample_stride

    def build_for_symbol(self, symbol: str, df_full: pd.DataFrame,
                          start_date=None, end_date=None) -> pd.DataFrame:
        df_full = df_full.sort_values("time").reset_index(drop=True)
        dates = df_full["time"]
        if start_date is not None:
            dates = dates[dates >= pd.Timestamp(start_date)]
        if end_date is not None:
            dates = dates[dates <= pd.Timestamp(end_date)]
        dates = dates.iloc[::self.sample_stride]

        rows = []
        for t in dates:
            df_slice = self.calendar.slice_sessions(df_full, "time", t, n_back=len(df_full))
            feat = self.features.feature_at(df_slice)
            if feat is None:
                continue
            lbl = self.labels.label_after(df_full, t)
            rows.append({"date": t, "symbol": symbol, **feat, **lbl})
        return pd.DataFrame(rows)

    def build_for_universe(self, symbol_to_df: dict, start_date=None, end_date=None) -> pd.DataFrame:
        parts = [self.build_for_symbol(sym, df, start_date, end_date) for sym, df in symbol_to_df.items()]
        parts = [p for p in parts if not p.empty]
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


## Look-ahead test (spec #71)

This is the test, not a description of one:
1. Compute features at T from the real data.
2. Mutate every row after T (corrupt price/volume completely).
3. Recompute features at T through the exact same `feature_at`/slicing path
   `ResearchDatasetBuilder` uses.
4. Assert every feature is bit-identical.

If any feature changes, that's a look-ahead bug in `MinimalFeatureEngine` or
in how it's being sliced — this test is designed to catch either.

In [ ]:
def _synthetic_series(n=150, seed=0, drift=0.0003, vol=0.015):
    rng = np.random.default_rng(seed)
    sessions = pd.bdate_range("2024-01-01", periods=n)
    price = 100 * np.exp(np.cumsum(rng.normal(drift, vol, n)))
    high = price * (1 + np.abs(rng.normal(0, 0.005, n)))
    low = price * (1 - np.abs(rng.normal(0, 0.005, n)))
    volume = rng.integers(50_000, 200_000, n)
    return pd.DataFrame({"time": sessions, "open": price, "high": high, "low": low,
                          "close": price, "volume": volume})


def test_look_ahead():
    df = _synthetic_series(150, seed=1)
    cal = TradingCalendar(df["time"])
    fe = MinimalFeatureEngine()

    t = df["time"].iloc[100]
    feat_before = fe.feature_at(cal.slice_sessions(df, "time", t, n_back=len(df)))

    df_mutated = df.copy()
    mask_future = df_mutated["time"] > t
    df_mutated.loc[mask_future, ["open", "high", "low", "close"]] = 999999.0
    df_mutated.loc[mask_future, "volume"] = 0

    feat_after = fe.feature_at(cal.slice_sessions(df_mutated, "time", t, n_back=len(df_mutated)))

    for k in feat_before:
        a, b = feat_before[k], feat_after[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b), f"{k}: {a} vs {b}"
        else:
            assert a == b, f"LOOK-AHEAD BUG in feature '{k}': {a} vs {b}"
    print(f"✓ Look-ahead test passed ({len(feat_before)} features, all identical after mutating {mask_future.sum()} future rows)")

test_look_ahead()


✓ Look-ahead test passed (11 features, all identical after mutating 49 future rows)


In [ ]:
def test_labels_use_only_future():
    df = _synthetic_series(150, seed=2)
    cal = TradingCalendar(df["time"])
    le = LabelEngine(cal)
    t = df["time"].iloc[100]

    lbl_before = le.label_after(df, t)

    # corrupting PAST data must not change labels at all
    df_mut_past = df.copy()
    df_mut_past.loc[df_mut_past["time"] < t, "close"] = 1
    lbl_after_past_corrupt = le.label_after(df_mut_past, t)
    for k in lbl_before:
        a, b = lbl_before[k], lbl_after_past_corrupt[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b)
        else:
            assert a == b, f"label '{k}' changed when PAST data was corrupted: {a} vs {b}"
    print("✓ Label-isolation test passed (past-data corruption does not affect labels)")

    # sanity check the other direction: labels MUST react to the relevant future window,
    # otherwise label_after itself could be silently broken (e.g. always returning NaN)
    df_mut_future = df.copy()
    df_mut_future.loc[df_mut_future["time"] > t, ["high", "low", "close"]] = 1
    lbl_after_future_corrupt = le.label_after(df_mut_future, t)
    assert lbl_after_future_corrupt["future_return_5d"] != lbl_before["future_return_5d"], \
        "sanity check failed: label did not react to its own future window"
    print("✓ Label sanity check passed (labels DO react to the relevant future window)")

test_labels_use_only_future()


✓ Label-isolation test passed (past-data corruption does not affect labels)
✓ Label sanity check passed (labels DO react to the relevant future window)


In [ ]:
def test_dataset_builder_end_to_end():
    df = _synthetic_series(150, seed=3)
    cal = TradingCalendar(df["time"])
    builder = ResearchDatasetBuilder(cal, sample_stride=5)
    out = builder.build_for_symbol("TEST", df)

    assert len(out) > 0
    required_cols = {"date", "symbol", "close", "sma20", "future_return_5d", "future_return_20d"}
    assert required_cols.issubset(out.columns)
    # rows near the end of history should have NaN for horizons that don't have enough future data yet -
    # NaN, never a fabricated 0, which would silently look like "flat return"
    assert out["future_return_20d"].isna().sum() > 0, "expected some rows too close to the end to have a 20d label"
    print(f"✓ End-to-end dataset builder test passed ({len(out)} rows, {out.shape[1]} columns)")
    return out

_sample_dataset = test_dataset_builder_end_to_end()
_sample_dataset[["date","symbol","close","sma20","return_5d","future_return_5d","future_return_20d"]].tail(5)


✓ End-to-end dataset builder test passed (18 rows, 25 columns)


,date,symbol,close,sma20,return_5d,future_return_5d,future_return_20d
13,2024-06-24,TEST,99.096578,97.927986,0.027885,0.076164,0.067244
14,2024-07-01,TEST,106.644218,99.534616,0.076164,0.002687,NaN
15,2024-07-08,TEST,106.930781,100.742910,0.002687,-0.026712,NaN
16,2024-07-15,TEST,104.074422,102.746596,-0.026712,0.016199,NaN
17,2024-07-22,TEST,105.760277,105.251016,0.016199,NaN,NaN


## Usage (once you have real data via `MarketDataStore`)

```python
store = MarketDataStore(QuantConfig())
cal = TradingCalendar.from_reference_ohlcv(store.fetch("VNINDEX", "2018-01-01", "2026-08-01"))

universe = ["FPT", "VCB", "HPG"]  # start with a SMALL test universe first, per spec Phase 3
symbol_dfs = {s: store.fetch(s, "2018-01-01", "2026-08-01") for s in universe}
symbol_dfs = {s: df for s, df in symbol_dfs.items() if df is not None and len(df) > MinimalFeatureEngine.MIN_BARS}

builder = ResearchDatasetBuilder(cal, sample_stride=5)  # stride=5 for a fast first pass
dataset = builder.build_for_universe(symbol_dfs)

dataset.to_parquet("data/features/research_dataset_v1.parquet", index=False)
```

Keep `sample_stride=1` (every session) only once you're ready for the full
run — with a 250-symbol universe and daily sampling this is O(symbols ×
sessions), so start small and widen the universe/date range once the
pipeline and look-ahead test are both green on real data, not just synthetic.

## Roadmap update

Phase 3 delivered: `feature_at(T)` / `label_after(T)` contracts, a minimal
verified feature set, forward-return labels, and the look-ahead test proving
the contract holds under mutation — not just by construction.

Still ahead:
- **Phase 4** — `TripleBarrierLabeler` (TP/SL/timeout labels, replacing the
  plain forward returns here) + `EmpiricalProbabilityEngine` (replaces
  `wr_base=45%`/`min_tp_pct=9%` with measured P(TP_FIRST) per setup, with
  sample-size confidence bands).
- **Phase 5** — Factor engine: port the v1 VSA/Ichimoku/DMI/momentum-divergence
  logic into `feature_at(T)`-safe form and add it to `MinimalFeatureEngine`
  (or a successor), each addition covered by its own look-ahead test.
- **Phase 6–8** — Market breadth, sector engine, setup engine, exit engine.
- **Phase 9–12** — Probability model, backtest v2, walk-forward, ablation.

---
# PHASE 4 — Triple-Barrier Labeling + Empirical Probability

Replaces the two hardcoded numbers flagged in the Phase 1 audit:

- `QuantConfig.wr_base = 45.0` → `EmpiricalProbabilityEngine` measures
  P(TP_FIRST) from actual historical outcomes, conditional on setup/regime,
  with a sample-size confidence tag on every number.
- `QuantConfig.min_tp_pct = 9.0` (flat % for every stock) →
  `TripleBarrierLabeler`'s TP/SL barriers are **ATR-derived by default**, so
  the target scales with each stock's own volatility instead of one universal
  percentage.

## `TripleBarrierLabeler`

Entry is **T+1 open** — never the signal-day close, per the execution model
in Phase 1/spec §5. Barriers:

```
tp_price = entry_price × (1 + tp_mult × atr_pct_at_signal)
sl_price = entry_price × (1 - sl_mult × atr_pct_at_signal)
```

`atr_pct_at_signal` must come from `MinimalFeatureEngine.feature_at()` (Phase 3)
computed through the signal date only — the labeler itself never touches the
pre-entry data, so the look-ahead boundary stays in exactly one place.

If both barriers are technically touched within the same bar, **SL wins**
(risk-first — the conservative assumption absent real intrabar path data,
matching the philosophy already stated in v1's backtester comments).

If there isn't yet enough future data to know whether TIMEOUT would have been
reached, `label()` returns `None` rather than guessing — a signal near the end
of your available history should be excluded from statistics, not silently
mislabeled.

In [ ]:
class TripleBarrierLabeler:
    def __init__(self, calendar: "TradingCalendar", tp_mult: float = 2.0, sl_mult: float = 1.0,
                 horizon_sessions: int = 10):
        self.calendar = calendar
        self.tp_mult = tp_mult
        self.sl_mult = sl_mult
        self.horizon_sessions = horizon_sessions

    def label(self, df_full: pd.DataFrame, signal_date, atr_pct_at_signal: float) -> Optional[dict]:
        entry_date = self.calendar.add_sessions(signal_date, 1)
        if entry_date is None:
            return None
        entry_row = df_full[df_full["time"] == pd.Timestamp(entry_date)]
        if entry_row.empty or pd.isna(atr_pct_at_signal):
            return None
        entry_price = float(entry_row["open"].iloc[0])

        tp_price = entry_price * (1 + self.tp_mult * atr_pct_at_signal)
        sl_price = entry_price * (1 - self.sl_mult * atr_pct_at_signal)

        horizon_date = self.calendar.add_sessions(entry_date, self.horizon_sessions)
        future = df_full[df_full["time"] >= pd.Timestamp(entry_date)].reset_index(drop=True)
        window = future[future["time"] <= pd.Timestamp(horizon_date)] if horizon_date is not None else future

        mfe, mae = 0.0, 0.0
        for _, row in window.iterrows():
            mfe = max(mfe, row["high"] / entry_price - 1)
            mae = min(mae, row["low"] / entry_price - 1)
            if row["low"] <= sl_price:            # risk-first if both touched intrabar
                days_held = self.calendar.sessions_between(entry_date, row["time"]) + 1
                return {"signal_date": signal_date, "entry_date": entry_date, "entry_price": entry_price,
                        "tp_price": tp_price, "sl_price": sl_price, "exit_date": row["time"],
                        "outcome": "SL_FIRST", "days_held": days_held,
                        "forward_return": sl_price / entry_price - 1, "mfe": mfe, "mae": mae}
            if row["high"] >= tp_price:
                days_held = self.calendar.sessions_between(entry_date, row["time"]) + 1
                return {"signal_date": signal_date, "entry_date": entry_date, "entry_price": entry_price,
                        "tp_price": tp_price, "sl_price": sl_price, "exit_date": row["time"],
                        "outcome": "TP_FIRST", "days_held": days_held,
                        "forward_return": tp_price / entry_price - 1, "mfe": mfe, "mae": mae}

        if horizon_date is None or window.empty or window["time"].max() < pd.Timestamp(horizon_date):
            return None  # not enough future data yet - exclude, do not guess TIMEOUT

        last = window.iloc[-1]
        days_held = self.calendar.sessions_between(entry_date, last["time"]) + 1
        return {"signal_date": signal_date, "entry_date": entry_date, "entry_price": entry_price,
                "tp_price": tp_price, "sl_price": sl_price, "exit_date": last["time"],
                "outcome": "TIMEOUT", "days_held": days_held,
                "forward_return": last["close"] / entry_price - 1, "mfe": mfe, "mae": mae}


## `EmpiricalProbabilityEngine`

Turns a dataframe of `TripleBarrierLabeler` outcomes into `P(TP_FIRST)`,
`P(SL_FIRST)`, `P(TIMEOUT)` conditional on whatever setup/context columns you
group by (spec §16: conditional probability, not one flat win rate). Every
row carries `n` and a `confidence` tag — `LOW_CONFIDENCE` under 30 samples,
`MEDIUM_CONFIDENCE` 30–99, `HIGHER_CONFIDENCE` at 100+ (thresholds from spec
§17, configurable). Nothing is ever reported as a bare probability without
its sample size next to it.

In [ ]:
class EmpiricalProbabilityEngine:
    def __init__(self, n_low: int = 30, n_medium: int = 100):
        self.n_low = n_low
        self.n_medium = n_medium

    def _confidence(self, n: int) -> str:
        if n < self.n_low:
            return "LOW_CONFIDENCE"
        elif n < self.n_medium:
            return "MEDIUM_CONFIDENCE"
        return "HIGHER_CONFIDENCE"

    def compute(self, labeled_df: pd.DataFrame, group_cols: Optional[list] = None) -> pd.DataFrame:
        df = labeled_df.dropna(subset=["outcome"]).copy()
        if df.empty:
            return pd.DataFrame(columns=(group_cols or []) + ["n","p_tp_first","p_sl_first","p_timeout","expected_return","confidence"])
        keys = group_cols if group_cols else []
        grouped = df.groupby(keys) if keys else [((), df)]

        rows = []
        for key, g in grouped:
            n = len(g)
            row = {}
            if keys:
                key_tuple = key if isinstance(key, tuple) else (key,)
                row.update(dict(zip(keys, key_tuple)))
            row["n"] = n
            row["p_tp_first"] = (g["outcome"] == "TP_FIRST").mean()
            row["p_sl_first"] = (g["outcome"] == "SL_FIRST").mean()
            row["p_timeout"] = (g["outcome"] == "TIMEOUT").mean()
            row["expected_return"] = g["forward_return"].mean()
            row["confidence"] = self._confidence(n)
            rows.append(row)
        return pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)

    def expected_value(self, p_tp: float, p_sl: float, reward_pct: float, risk_pct: float) -> float:
        """EV = P(TP)*reward - P(SL)*risk (spec #42). For the timeout leg, use
        the `expected_return` column from compute() instead, which already
        return-weights all three outcomes rather than dropping TIMEOUT."""
        return p_tp * reward_pct - p_sl * risk_pct


## Tests

Four labeler tests target the specific failure modes a triple-barrier
implementation is prone to, plus one engine test checking the probability
engine actually recovers a real conditional effect (not just that it runs).

In [ ]:
def test_entry_is_t_plus_1_open():
    df = _synthetic_series(200, seed=101)
    cal = TradingCalendar(df["time"])
    tb = TripleBarrierLabeler(cal, tp_mult=2.0, sl_mult=1.0, horizon_sessions=10)
    signal_date = df["time"].iloc[100]
    t1_date = cal.add_sessions(signal_date, 1)
    expected_open = float(df[df["time"] == t1_date]["open"].iloc[0])

    out = tb.label(df, signal_date, atr_pct_at_signal=0.02)
    assert out is not None
    assert out["entry_date"] == t1_date, "entry must be T+1, never the signal date itself"
    assert out["entry_price"] == expected_open, "entry must be T+1 OPEN, not close or a theoretical price"
    print("✓ Entry-timing test passed: entry_date =", out["entry_date"].date(), "entry_price =", round(out["entry_price"],2))

test_entry_is_t_plus_1_open()


✓ Entry-timing test passed: entry_date = 2024-05-21 entry_price = 94.77


In [ ]:
def test_sl_wins_when_both_hit_same_bar():
    sessions = pd.bdate_range("2023-01-01", periods=10)
    df = pd.DataFrame({"time": sessions, "open": [100]*10, "close": [100]*10,
                        "high": [100]*10, "low": [100]*10})
    df.loc[1, "high"] = 130  # entry bar (T+1) touches both TP and SL extremes
    df.loc[1, "low"] = 50
    cal = TradingCalendar(sessions)
    tb = TripleBarrierLabeler(cal, tp_mult=1.0, sl_mult=1.0, horizon_sessions=5)
    out = tb.label(df, sessions[0], atr_pct_at_signal=0.10)  # entry=100, tp=110, sl=90
    assert out["outcome"] == "SL_FIRST", out
    print("✓ SL-priority test passed (both barriers touched intrabar -> SL wins, risk-first)")

test_sl_wins_when_both_hit_same_bar()


✓ SL-priority test passed (both barriers touched intrabar -> SL wins, risk-first)


In [ ]:
def test_timeout_when_insufficient_future_data():
    df = _synthetic_series(200, seed=102)
    cal = TradingCalendar(df["time"])
    tb = TripleBarrierLabeler(cal, tp_mult=5.0, sl_mult=5.0, horizon_sessions=10)  # wide, unlikely to hit
    signal_date = df["time"].iloc[195]  # too close to the end for a full horizon
    out = tb.label(df, signal_date, atr_pct_at_signal=0.02)
    assert out is None, "must return None instead of fabricating a TIMEOUT outcome"
    print("✓ Insufficient-future-data test passed (returns None rather than guessing)")

test_timeout_when_insufficient_future_data()


✓ Insufficient-future-data test passed (returns None rather than guessing)


In [ ]:
def test_no_leakage_in_labeler():
    df = _synthetic_series(200, seed=103)
    cal = TradingCalendar(df["time"])
    tb = TripleBarrierLabeler(cal, tp_mult=2.0, sl_mult=1.0, horizon_sessions=10)
    signal_date = df["time"].iloc[100]

    out_before = tb.label(df, signal_date, atr_pct_at_signal=0.02)
    df_corrupt = df.copy()
    df_corrupt.loc[df_corrupt["time"] < signal_date, ["open","high","low","close"]] = 1
    out_after = tb.label(df_corrupt, signal_date, atr_pct_at_signal=0.02)
    for k in ["entry_price","tp_price","sl_price","outcome","forward_return"]:
        assert out_before[k] == out_after[k], f"{k} changed when PAST data was corrupted"
    print("✓ No-leakage test passed (corrupting rows before the signal date does not change the label)")

test_no_leakage_in_labeler()


✓ No-leakage test passed (corrupting rows before the signal date does not change the label)


In [ ]:
def test_empirical_probability_engine():
    rng = np.random.default_rng(42)
    n = 500
    setup = rng.choice(["breakout", "pullback"], size=n)
    regime = rng.choice(["bull", "bear"], size=n, p=[0.6, 0.4])
    outcome, fwd_ret = [], []
    for s, r in zip(setup, regime):
        p = [0.55, 0.25, 0.20] if (s == "breakout" and r == "bull") else [0.30, 0.40, 0.30]
        o = rng.choice(["TP_FIRST","SL_FIRST","TIMEOUT"], p=p)
        outcome.append(o)
        fwd_ret.append({"TP_FIRST":0.08,"SL_FIRST":-0.04,"TIMEOUT":0.01}[o] + rng.normal(0,0.01))
    df = pd.DataFrame({"setup": setup, "regime": regime, "outcome": outcome, "forward_return": fwd_ret})

    eng = EmpiricalProbabilityEngine(n_low=30, n_medium=100)
    result = eng.compute(df, group_cols=["setup","regime"])

    bb = result[(result["setup"]=="breakout") & (result["regime"]=="bull")].iloc[0]
    other_avg = result[~((result["setup"]=="breakout") & (result["regime"]=="bull"))]["p_tp_first"].mean()
    assert bb["p_tp_first"] > other_avg, "engine should recover the genuinely stronger breakout+bull setup"
    print("✓ EmpiricalProbabilityEngine recovers the real conditional effect:")
    print(result.to_string())

    tiny_result = eng.compute(df.iloc[:10])
    assert tiny_result.iloc[0]["confidence"] == "LOW_CONFIDENCE"
    print("✓ Tiny-sample confidence tagging test passed (n=10 -> LOW_CONFIDENCE)")

test_empirical_probability_engine()
print("\nALL PHASE 4 TESTS PASSED")


✓ EmpiricalProbabilityEngine recovers the real conditional effect:
      setup regime    n  p_tp_first  p_sl_first  p_timeout  expected_return         confidence
0  pullback   bull  151    0.324503    0.317881   0.357616         0.017237  HIGHER_CONFIDENCE
1  breakout   bull  147    0.530612    0.231293   0.238095         0.034703  HIGHER_CONFIDENCE
2  breakout   bear  109    0.357798    0.339450   0.302752         0.017485  HIGHER_CONFIDENCE
3  pullback   bear   93    0.333333    0.408602   0.258065         0.011580  MEDIUM_CONFIDENCE
✓ Tiny-sample confidence tagging test passed (n=10 -> LOW_CONFIDENCE)

ALL PHASE 4 TESTS PASSED


## Usage — wiring Phase 3 + Phase 4 together

```python
fe = MinimalFeatureEngine()
cal = TradingCalendar.from_reference_ohlcv(vnindex_df)
tb = TripleBarrierLabeler(cal, tp_mult=2.0, sl_mult=1.0, horizon_sessions=10)

records = []
for sym, df in symbol_dfs.items():
    for t in df["time"].iloc[::5]:                       # sample every 5 sessions for a first pass
        df_slice = cal.slice_sessions(df, "time", t, n_back=len(df))
        feat = fe.feature_at(df_slice)
        if feat is None:
            continue
        outcome = tb.label(df, t, atr_pct_at_signal=feat["atr_pct"])
        if outcome is None:
            continue
        records.append({"symbol": sym, **feat, **outcome})

labeled = pd.DataFrame(records)
labeled.to_parquet("data/labels/triple_barrier_v1.parquet", index=False)

eng = EmpiricalProbabilityEngine()
stats = eng.compute(labeled, group_cols=["price_above_sma50"])  # replace with real setup/regime
                                                                  # columns once Phase 5-7 exist
```

Right now the only grouping columns available are the Phase-3 minimal
features (`price_above_sma20`, etc.) — genuinely useful setup/regime columns
(`breakout`, `bull market`, `sector strength`) don't exist until Phase 5–7's
factor/setup/regime engines are built. `EmpiricalProbabilityEngine` doesn't
care what the grouping columns are, so nothing here needs to change when
those are added later — you just pass richer `group_cols`.

## Roadmap update

Phase 4 delivered: ATR-derived triple-barrier labels (replacing the flat
`min_tp_pct=9%`) and empirical, sample-size-aware conditional probabilities
(replacing the flat `wr_base=45%`).

Still ahead:
- **Phase 5** — Factor engine: port v1's VSA/Ichimoku/DMI/momentum-divergence
  logic into `feature_at(T)`-safe features, each with its own look-ahead test,
  to give `EmpiricalProbabilityEngine` real setup columns to group by.
- **Phase 6–8** — Market breadth, sector engine, setup engine (breakout/
  pullback/accumulation/distribution), exit engine.
- **Phase 9** — Probability model: logistic regression baseline on top of the
  labeled dataset, calibration analysis (predicted P vs. observed frequency).
- **Phase 10–12** — Backtest v2 (fees/slippage/price-limits, built on
  `FillSimulator`), walk-forward validation, ablation + parameter sensitivity.

---
# PHASE 5 — Factor Engine (VSA, Ichimoku, DMI/Aroon, Momentum Divergence)

Ports four of v1's technical engines into `feature_at(T)`-safe form, each
with its own look-ahead test. **Not just re-typed** — auditing each one
surfaced a real, previously-unflagged bug (see Ichimoku and Divergence
below), fixed here rather than carried forward silently.

Per spec §18-21, this is still just the FEATURE layer — turning these into
interpretable 0-100 FACTOR scores with ablation-tested weights (replacing
`BASE_WEIGHTS`/`sc_*`) is separate work, once `EmpiricalProbabilityEngine`
(Phase 4) has enough labeled history to say which of these actually carries
alpha.

## 1. `VSAFeature`

Ported from v1 `VSAEngine`. Already look-ahead safe as written — every
input (`vol_ratio`, `spread_ratio`, `close_pos`, `roc`) uses `rolling()` or
`pct_change()`, both strictly backward-looking, no `.shift()` toward the
future anywhere. Confirmed with a test rather than assumed. The fixed VSA
scores from v1 (`NỔ VOL → 8.5`, etc.) are **not ported** — spec §22 flags
those as unvalidated; the categorical `vsa_state` is exposed instead, so
`EmpiricalProbabilityEngine` (Phase 4) can measure what each state is
actually worth.

In [ ]:
class VSAFeature:
    STATES = ["NO_VOL", "CAN_VOL", "RUT_CHAN", "PHAN_PHOI", "CAN_CAU", "NEUTRAL"]

    def __init__(self, lookback: int = 20):
        self.lookback = lookback

    def feature_at(self, df_slice: pd.DataFrame) -> Optional[dict]:
        if len(df_slice) < self.lookback + 2:
            return None
        c, h, l, v = df_slice["close"], df_slice["high"], df_slice["low"], df_slice["volume"]

        avg_vol = v.rolling(self.lookback).mean().replace(0, np.nan)
        vol_ratio = (v / avg_vol).iloc[-1]
        spread = h - l
        avg_spread = spread.rolling(self.lookback).mean().replace(0, np.nan)
        spread_ratio = (spread / avg_spread).iloc[-1]

        sp_now = float(spread.iloc[-1])
        close_pos = 0.5 if sp_now == 0 else float((c.iloc[-1] - l.iloc[-1]) / (sp_now + 1e-9))
        roc = float(c.pct_change().iloc[-1])

        if pd.isna(vol_ratio) or pd.isna(spread_ratio):
            return {"vol_ratio": np.nan, "spread_ratio": np.nan, "close_pos": close_pos,
                    "roc_1d": roc, "vsa_state": "NEUTRAL"}

        vr, sr, cp = float(vol_ratio), float(spread_ratio), close_pos
        if vr > 1.5 and sr > 1.2 and cp > 0.7 and roc > 0.015:      state = "NO_VOL"
        elif vr < 0.6 and sr < 0.85 and cp >= 0.4 and roc > -0.015: state = "CAN_VOL"
        elif vr > 1.3 and sr > 1.3 and cp > 0.65 and roc > -0.02:   state = "RUT_CHAN"
        elif vr > 1.5 and cp < 0.35 and sr > 1.0 and roc < 0.02:    state = "PHAN_PHOI"
        elif vr < 0.6 and cp < 0.4 and roc > 0:                     state = "CAN_CAU"
        else:                                                       state = "NEUTRAL"
        return {"vol_ratio": vr, "spread_ratio": sr, "close_pos": cp, "roc_1d": roc, "vsa_state": state}


def test_vsa_look_ahead():
    df = _synthetic_series(200, seed=11)
    cal = TradingCalendar(df["time"])
    fe = VSAFeature()
    t = df["time"].iloc[120]
    before = fe.feature_at(cal.slice_sessions(df, "time", t, n_back=len(df)))
    df_mut = df.copy()
    mask = df_mut["time"] > t
    df_mut.loc[mask, ["open","high","low","close"]] = 999999.0
    df_mut.loc[mask, "volume"] = 0
    after = fe.feature_at(cal.slice_sessions(df_mut, "time", t, n_back=len(df_mut)))
    for k in before:
        a, b = before[k], after[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b)
        else:
            assert a == b, f"VSA LOOK-AHEAD BUG in '{k}': {a} vs {b}"
    print("✓ VSAFeature look-ahead test passed:", before["vsa_state"])

test_vsa_look_ahead()


✓ VSAFeature look-ahead test passed: NEUTRAL


## 2. `IchimokuFeature`

Ported from v1 `IchimokuEngine`. The Daily calculation in v1 is genuinely
look-ahead safe *as written* — `tenkan_s`/`kijun_s`/`span_a_s`/`span_b_s`
are all rolling/backward-looking, and reading `target_idx = -k-1` pulls an
**older** value to reconstruct "today's cloud edge" the way classic
Ichimoku displaces the cloud forward by k periods. That's correct, not a
leak — confirmed by the look-ahead test below.

**Real bug found and fixed here:** v1's Weekly/Monthly used
`df.resample(...)` directly, which produces a **partial final bar** when the
slice ends mid-week or mid-month. That bar's OHLC — and therefore Kijun
Weekly, used as v1's T2 target — silently shifts every single day until the
week/month actually closes. This version always drops the final resampled
period and uses the last **closed** week/month, verified with a dedicated
stability test.

In [ ]:
class IchimokuFeature:
    def __init__(self, tenkan: int = 9, kijun: int = 26, senkou_b: int = 52):
        self.t, self.k, self.sb = tenkan, kijun, senkou_b

    def _calc(self, df: pd.DataFrame) -> dict:
        if df is None or len(df) < self.k + 2:
            return {}
        h, l, c = df["high"], df["low"], df["close"]
        tenkan_s = (h.rolling(self.t).max() + l.rolling(self.t).min()) / 2
        kijun_s  = (h.rolling(self.k).max() + l.rolling(self.k).min()) / 2
        span_a_s = (tenkan_s + kijun_s) / 2
        span_b_s = (h.rolling(self.sb).max() + l.rolling(self.sb).min()) / 2

        target_idx = -self.k - 1
        def _read_or_fallback(series):
            if len(series) >= abs(target_idx) and not pd.isna(series.iloc[target_idx]):
                return float(series.iloc[target_idx])
            valid = series.dropna()
            return float(valid.iloc[-1]) if not valid.empty else float(c.iloc[-1])

        span_a_now = _read_or_fallback(span_a_s)
        span_b_now = _read_or_fallback(span_b_s)
        tenkan_now = float(tenkan_s.iloc[-1]) if not pd.isna(tenkan_s.iloc[-1]) else float(c.iloc[-1])
        kijun_now  = float(kijun_s.iloc[-1])  if not pd.isna(kijun_s.iloc[-1])  else float(c.iloc[-1])
        close_now  = float(c.iloc[-1])

        kumo_top, kumo_bottom = max(span_a_now, span_b_now), min(span_a_now, span_b_now)
        return {
            "tenkan": tenkan_now, "kijun": kijun_now, "span_a": span_a_now, "span_b": span_b_now,
            "above_cloud": bool(close_now > kumo_top), "below_cloud": bool(close_now < kumo_bottom),
            "in_cloud": not (close_now > kumo_top) and not (close_now < kumo_bottom),
            "strong_cloud": bool(span_a_now > span_b_now),
            "cloud_thick_pct": abs(span_a_now - span_b_now) / (kumo_bottom + 1e-9) * 100,
        }

    def _last_closed_period(self, df_slice: pd.DataFrame, rule: str) -> pd.DataFrame:
        r = df_slice.set_index("time").resample(rule).agg(
            {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
        ).dropna()
        return r.iloc[:-1].reset_index()  # ALWAYS drop the final (possibly-incomplete) period

    def feature_at(self, df_slice: pd.DataFrame) -> Optional[dict]:
        if len(df_slice) < self.k + 2:
            return None
        out = {}
        d = self._calc(df_slice)
        out.update({f"ichi_{k}_d": v for k, v in d.items()})

        w_df = self._last_closed_period(df_slice, "W-FRI")
        w = self._calc(w_df) if len(w_df) >= self.k + 2 else {}
        out.update({f"ichi_{k}_w": v for k, v in w.items()})

        m_df = self._last_closed_period(df_slice, "ME")
        m = self._calc(m_df) if len(m_df) >= self.k + 2 else {}
        out.update({f"ichi_{k}_m": v for k, v in m.items()})

        out["ichi_3tf_bull"] = bool(d.get("above_cloud") and w.get("above_cloud") and m.get("above_cloud")) \
            if (d and w and m) else False
        return out


In [ ]:
def test_ichimoku_look_ahead():
    df = _synthetic_series(500, seed=21)
    cal = TradingCalendar(df["time"])
    fe = IchimokuFeature()
    t = df["time"].iloc[300]
    before = fe.feature_at(cal.slice_sessions(df, "time", t, n_back=len(df)))
    df_mut = df.copy()
    mask = df_mut["time"] > t
    df_mut.loc[mask, ["open","high","low","close"]] = 999999.0
    after = fe.feature_at(cal.slice_sessions(df_mut, "time", t, n_back=len(df_mut)))
    for k in before:
        a, b = before[k], after[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b)
        else:
            assert a == b, f"ICHIMOKU LOOK-AHEAD BUG in '{k}': {a} vs {b}"
    print(f"✓ IchimokuFeature look-ahead test passed ({len(before)} keys)")

test_ichimoku_look_ahead()


def test_ichimoku_weekly_no_repaint():
    """The specific fix vs v1: the final (possibly partial) week/month is
    always excluded, for every weekday the signal date could fall on."""
    df = _synthetic_series(500, seed=22)
    cal = TradingCalendar(df["time"])
    fe = IchimokuFeature()
    for offset in range(5):
        t = df["time"].iloc[300 + offset]
        sl = cal.slice_sessions(df, "time", t, n_back=len(df))
        w_full = sl.set_index("time").resample("W-FRI").agg({"close":"last"}).dropna()
        w_used = fe._last_closed_period(sl, "W-FRI")
        assert len(w_used) == len(w_full) - 1, f"offset={offset}: last period was not dropped as expected"
    print("✓ Confirmed: the final (possibly partial) week is always excluded, for every weekday tested")

test_ichimoku_weekly_no_repaint()


✓ IchimokuFeature look-ahead test passed (19 keys)
✓ Confirmed: the final (possibly partial) week is always excluded, for every weekday tested


## 3. `DmiAroonFeature`

Standard Wilder DMI/ADX + standard Aroon. Both are strictly rolling/EWM by
construction (no centered windows, no negative shifts), so this family
doesn't have the repaint risk swing-pivot detection does — verified with a
look-ahead test rather than assumed safe just because the formula "looks
standard."   

In [ ]:
class DmiAroonFeature:
    def __init__(self, dmi_period: int = 14, aroon_period: int = 25):
        self.dmi_period = dmi_period
        self.aroon_period = aroon_period

    def feature_at(self, df_slice: pd.DataFrame) -> Optional[dict]:
        n_needed = max(self.dmi_period, self.aroon_period) + 5
        if len(df_slice) < n_needed:
            return None
        h, l, c = df_slice["high"], df_slice["low"], df_slice["close"]
        p = self.dmi_period

        up_move, down_move = h.diff(), -l.diff()
        plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
        minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
        tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/p, adjust=False).mean()

        plus_di = 100 * pd.Series(plus_dm, index=df_slice.index).ewm(alpha=1/p, adjust=False).mean() / atr
        minus_di = 100 * pd.Series(minus_dm, index=df_slice.index).ewm(alpha=1/p, adjust=False).mean() / atr
        dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
        adx = dx.ewm(alpha=1/p, adjust=False).mean()

        ap = self.aroon_period
        aroon_up = 100 * (ap - h.rolling(ap+1).apply(lambda x: ap - np.argmax(x), raw=True)) / ap
        aroon_down = 100 * (ap - l.rolling(ap+1).apply(lambda x: ap - np.argmin(x), raw=True)) / ap

        return {
            "plus_di": float(plus_di.iloc[-1]), "minus_di": float(minus_di.iloc[-1]),
            "adx": float(adx.iloc[-1]) if not pd.isna(adx.iloc[-1]) else np.nan,
            "aroon_up": float(aroon_up.iloc[-1]), "aroon_down": float(aroon_down.iloc[-1]),
            "dmi_bullish": bool(plus_di.iloc[-1] > minus_di.iloc[-1]),
        }


def test_dmi_aroon_look_ahead():
    df = _synthetic_series(200, seed=31)
    cal = TradingCalendar(df["time"])
    fe = DmiAroonFeature()
    t = df["time"].iloc[100]
    before = fe.feature_at(cal.slice_sessions(df, "time", t, n_back=len(df)))
    df_mut = df.copy()
    mask = df_mut["time"] > t
    df_mut.loc[mask, ["open","high","low","close"]] = 999999.0
    after = fe.feature_at(cal.slice_sessions(df_mut, "time", t, n_back=len(df_mut)))
    for k in before:
        a, b = before[k], after[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b)
        else:
            assert (abs(a - b) < 1e-9) if isinstance(a, float) else (a == b), f"DMI/AROON LOOK-AHEAD BUG in '{k}': {a} vs {b}"
    print("✓ DmiAroonFeature look-ahead test passed")

test_dmi_aroon_look_ahead()


✓ DmiAroonFeature look-ahead test passed


## 4. `MomentumDivergenceFeature`

Ported from v1 `MomentumEngine`'s RSI/MACD divergence detection.

**Real bug found and fixed here:** v1's `_find_swing_lows`/`_find_swing_highs`
use a *centered* window `[i-lookback, i+lookback]` to decide whether bar
`i` is a pivot. Near the end of the series — exactly where `feature_at(T)`
looks — Python's slicing silently **truncates** that window instead of
requiring it to be complete, so a bar can register as "the pivot" today and
stop being one a few sessions later once real future bars actually arrive.
That's not a leak (it never reads data past T), but it's a **repainting
artifact**: the same historical `(symbol, date)` row could get a *different*
divergence label depending on when you happened to run the pipeline, which
silently corrupts any statistics computed over those rows.

**Fix:** a pivot candidate at index `i` is only accepted if the full window
`i+lookback` is actually available (not truncated) — meaning the most recent
`lookback` bars can never register as a *fresh* pivot yet. That's the
correct, honest state (there genuinely isn't enough information yet), not a
limitation to route around.

In [ ]:
def _rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

def _macd_hist(close: pd.Series, fast=12, slow=26, signal=9) -> pd.Series:
    macd = close.ewm(span=fast, adjust=False).mean() - close.ewm(span=slow, adjust=False).mean()
    return macd - macd.ewm(span=signal, adjust=False).mean()


class MomentumDivergenceFeature:
    def __init__(self, lookback: int = 5, min_bars: int = 5, min_price_chg_pct: float = 0.3):
        self.lookback = lookback
        self.min_bars = min_bars
        self.min_price_chg_pct = min_price_chg_pct

    def _find_confirmed_pivots(self, series: np.ndarray, find_low: bool) -> list:
        pivots = []
        n, lb = len(series), self.lookback
        for i in range(lb, n - lb):  # i+lb <= n-1 guarantees a FULL, non-truncated window
            window = series[i - lb: i + lb + 1]
            is_pivot = (series[i] == window.min()) if find_low else (series[i] == window.max())
            if not is_pivot:
                continue
            if pivots and (i - pivots[-1][0]) < self.min_bars:
                better = series[i] < pivots[-1][1] if find_low else series[i] > pivots[-1][1]
                if better:
                    pivots[-1] = (i, series[i])
            else:
                pivots.append((i, series[i]))
        return pivots

    def _divergence(self, price: np.ndarray, indicator: np.ndarray, bullish: bool) -> bool:
        if len(price) < self.lookback * 4 or len(price) != len(indicator):
            return False
        valid = ~(np.isnan(price) | np.isnan(indicator))
        if valid.sum() < self.lookback * 2:
            return False
        pivots = self._find_confirmed_pivots(price, find_low=bullish)
        if len(pivots) < 2:
            return False
        (i1, p1), (i2, p2) = pivots[-2], pivots[-1]
        if abs(p2 - p1) / (abs(p1) + 1e-9) * 100 < self.min_price_chg_pct:
            return False
        ind1, ind2 = indicator[i1], indicator[i2]
        return bool(p2 < p1 and ind2 > ind1) if bullish else bool(p2 > p1 and ind2 < ind1)

    def feature_at(self, df_slice: pd.DataFrame) -> Optional[dict]:
        min_len = self.lookback * 2 + 2 + 26
        if len(df_slice) < min_len:
            return None
        close, low, high = df_slice["close"], df_slice["low"], df_slice["high"]
        rsi, macd_h = _rsi(close), _macd_hist(close)
        low_arr, high_arr = low.to_numpy(), high.to_numpy()
        rsi_arr, macd_arr = rsi.to_numpy(), macd_h.to_numpy()
        return {
            "rsi14": float(rsi.iloc[-1]) if not pd.isna(rsi.iloc[-1]) else np.nan,
            "macd_hist": float(macd_h.iloc[-1]) if not pd.isna(macd_h.iloc[-1]) else np.nan,
            "rsi_bull_div": self._divergence(low_arr, rsi_arr, bullish=True),
            "rsi_bear_div": self._divergence(high_arr, rsi_arr, bullish=False),
            "macd_bull_div": self._divergence(low_arr, macd_arr, bullish=True),
            "macd_bear_div": self._divergence(high_arr, macd_arr, bullish=False),
        }


In [ ]:
def test_momentum_divergence_look_ahead():
    df = _synthetic_series(300, seed=41)
    cal = TradingCalendar(df["time"])
    fe = MomentumDivergenceFeature()
    t = df["time"].iloc[200]
    before = fe.feature_at(cal.slice_sessions(df, "time", t, n_back=len(df)))
    df_mut = df.copy()
    mask = df_mut["time"] > t
    df_mut.loc[mask, ["open","high","low","close"]] = 999999.0
    after = fe.feature_at(cal.slice_sessions(df_mut, "time", t, n_back=len(df_mut)))
    for k in before:
        a, b = before[k], after[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b)
        else:
            assert a == b, f"DIVERGENCE LOOK-AHEAD BUG in '{k}': {a} vs {b}"
    print("✓ MomentumDivergenceFeature look-ahead test passed")

test_momentum_divergence_look_ahead()


def test_no_repaint_on_recent_pivot():
    n = 120
    sessions = pd.bdate_range("2023-01-01", periods=n)
    price = np.full(n, 100.0)
    price[40], price[80] = 90.0, 85.0   # two planted lows, second one lower
    df = pd.DataFrame({"time": sessions, "open": price, "close": price,
                        "high": price + 1, "low": price - 1, "volume": np.full(n, 100000)})
    for i in range(41, 60):
        df.loc[i, ["open","close","high","low"]] = [90 + (i-40)*0.05]*4
    for i in range(81, 100):
        df.loc[i, ["open","close","high","low"]] = [85 + (i-80)*0.3]*4

    cal = TradingCalendar(sessions)
    fe = MomentumDivergenceFeature(lookback=5, min_bars=5)

    t_too_early = sessions[84]  # window around index 80 (75-85) not yet fully observed
    low_arr_early = df[df["time"] <= t_too_early]["low"].to_numpy()
    pivots_early = fe._find_confirmed_pivots(low_arr_early, find_low=True)
    assert all(idx != 80 for idx, _ in pivots_early), \
        f"pivot at index 80 should NOT be confirmed yet at t_too_early, got pivots={pivots_early}"
    print("✓ No-repaint test passed: recent unconfirmed pivot correctly excluded until its window is fully observed")

test_no_repaint_on_recent_pivot()
print("\nALL PHASE 5 FEATURE TESTS PASSED")


✓ MomentumDivergenceFeature look-ahead test passed
✓ No-repaint test passed: recent unconfirmed pivot correctly excluded until its window is fully observed

ALL PHASE 5 FEATURE TESTS PASSED


## `FactorEngineV5` — drop-in replacement feature engine for Phase 3

Wraps `MinimalFeatureEngine` (Phase 3) + all four Phase 5 features behind
one `feature_at(df_slice)` call, so it's a direct drop-in `feature_engine`
argument for `ResearchDatasetBuilder` (Phase 3) — no changes needed there.

If any sub-engine isn't ready (not enough bars yet), the whole row is
skipped rather than partially filled — a partially-filled feature row would
look complete to anything consuming the dataset later, silently mixing
"this feature is genuinely neutral" with "this feature couldn't be computed
yet," which is a real difference `EmpiricalProbabilityEngine` needs to be
able to tell apart.

In [ ]:
class FactorEngineV5:
    MIN_BARS = 300  # driven by Ichimoku's ~52+26-bar need plus weekly/monthly resample headroom

    def __init__(self):
        self.minimal = MinimalFeatureEngine()
        self.vsa = VSAFeature()
        self.ichimoku = IchimokuFeature()
        self.dmi_aroon = DmiAroonFeature()
        self.divergence = MomentumDivergenceFeature()

    def feature_at(self, df_slice: pd.DataFrame) -> Optional[dict]:
        if len(df_slice) < self.MIN_BARS:
            return None
        out = {}
        for sub in (self.minimal, self.vsa, self.ichimoku, self.dmi_aroon, self.divergence):
            f = sub.feature_at(df_slice)
            if f is None:
                return None  # any sub-engine not ready -> whole row not ready, no partial fill
            out.update(f)
        return out


def test_factor_engine_v5_end_to_end():
    df = _synthetic_series(400, seed=51)
    cal = TradingCalendar(df["time"])
    fe = FactorEngineV5()
    t = df["time"].iloc[350]
    feat = fe.feature_at(cal.slice_sessions(df, "time", t, n_back=len(df)))
    assert feat is not None
    expected = {"close","sma20","vsa_state","ichi_kijun_d","ichi_kijun_w","adx","rsi14","macd_bull_div"}
    assert expected.issubset(feat.keys()), expected - feat.keys()
    print(f"✓ FactorEngineV5 end-to-end test passed ({len(feat)} total keys)")

test_factor_engine_v5_end_to_end()


✓ FactorEngineV5 end-to-end test passed (47 total keys)


## Usage — swap into Phase 3/4's pipeline

```python
builder = ResearchDatasetBuilder(cal, feature_engine=FactorEngineV5(), sample_stride=5)
dataset = builder.build_for_universe(symbol_dfs)   # now has vsa_state, ichi_*, adx, rsi14, *_div columns

tb = TripleBarrierLabeler(cal, tp_mult=2.0, sl_mult=1.0, horizon_sessions=10)
labeled = [...]  # as in Phase 4, atr_pct_at_signal now comes from FactorEngineV5's output

eng = EmpiricalProbabilityEngine()
eng.compute(labeled_df, group_cols=["vsa_state"])          # is NỔ VOL actually better than PHÂN PHỐI?
eng.compute(labeled_df, group_cols=["dmi_bullish","adx"])  # does ADX>25 actually matter, or is it decorative?
```

This is the point of Phase 4/5 existing together: `FactorEngineV5` gives you
real setup columns, `EmpiricalProbabilityEngine` tells you which of them
actually predicted anything historically — instead of the v1 approach of
assigning `NỔ VOL = 8.5` by hand and hoping.

## Roadmap update

Phase 5 delivered: `VSAFeature`, `IchimokuFeature` (with the weekly/monthly
repaint fix), `DmiAroonFeature`, `MomentumDivergenceFeature` (with the
confirmed-pivot repaint fix), and `FactorEngineV5` combining them behind
Phase 3's `feature_at(T)` interface. Every feature has an explicit
look-ahead test; the two repaint fixes each have their own dedicated test
beyond the standard look-ahead check.

Still ahead:
- **Phase 6** — Market breadth engine (% stocks above MA20/50/200,
  advance/decline, new highs/lows) — needs the FULL universe fetched
  together, not just per-symbol like everything so far.
- **Phase 7** — Sector engine, Setup engine (breakout/pullback/accumulation/
  distribution as explicit classifiers on top of these features).
- **Phase 8** — Exit engine (trend/flow/RS deterioration → exit risk score).
- **Phase 9** — Probability model (logistic regression on the now-much-richer
  `FactorEngineV5` feature set + calibration analysis).
- **Phase 10–12** — Backtest v2, walk-forward validation, ablation
  (this is where `BASE_WEIGHTS`/`sc_*` finally get replaced with weights
  that are shown, not assumed, to add incremental alpha).

---
# PHASE 6 — Market Breadth Engine

Spec §26: `% stocks above MA20/50/200`, advance/decline ratio, up/down
volume, new highs/lows.

**This is a different shape of problem than Phases 3-5.** Every feature
engine so far answers "what does this ONE symbol's history through T look
like" — `feature_at(df_slice)` for a single stock is enough. Breadth is
a market-level statistic: it needs every symbol's state as of the SAME date
T, aggregated together. `VNINDEX +1.2%` backed by 75% of stocks above MA20
is a different market than `VNINDEX +1.2%` backed by 40% (spec §26's own
example) — no single stock's `feature_at(T)` can tell you which one you're
in.

Two correctness details that matter here specifically, beyond the usual
look-ahead check:

1. **Eligibility, not false negatives.** A stock listed 30 sessions ago
   can't have a 200-day MA yet. It must be excluded from
   `pct_above_ma200`'s denominator (`n_eligible_ma200` tracks this), not
   silently counted as "below MA200" — that would make breadth look
   artificially bearish every time a new listing joins the universe.
2. **Halted/delisted symbols are excluded, not treated as flat.** If a
   symbol has no row on date T, it drops out of that date's `n_total`
   entirely rather than being counted as unchanged (which would quietly
   understate both advances and declines).

In [ ]:
class MarketBreadthEngine:
    def __init__(self, ma_periods=(20, 50, 200), new_high_low_window: int = 252):
        self.ma_periods = ma_periods
        self.new_high_low_window = new_high_low_window

    def _symbol_stats(self, df_slice: pd.DataFrame) -> Optional[dict]:
        if len(df_slice) < 2:
            return None
        close = df_slice["close"]
        c_now, c_prev = float(close.iloc[-1]), float(close.iloc[-2])
        vol_now = float(df_slice["volume"].iloc[-1])

        stats = {
            "advanced": c_now > c_prev, "declined": c_now < c_prev,
            "volume": vol_now,
            "up_volume": vol_now if c_now > c_prev else 0.0,
            "down_volume": vol_now if c_now < c_prev else 0.0,
        }
        for p in self.ma_periods:
            if len(close) >= p:
                ma = close.rolling(p).mean().iloc[-1]
                stats[f"above_ma{p}"] = bool(c_now > ma) if not pd.isna(ma) else None
            else:
                stats[f"above_ma{p}"] = None  # not eligible - NOT False

        w = self.new_high_low_window
        if len(close) >= w:
            window = close.tail(w)
            stats["new_high"] = bool(c_now >= window.max())
            stats["new_low"] = bool(c_now <= window.min())
        else:
            stats["new_high"] = None
            stats["new_low"] = None
        return stats

    def compute_at(self, t, calendar: "TradingCalendar", universe_data: dict) -> Optional[dict]:
        """universe_data: {symbol: full_df (unsliced)}. Slicing to T happens
        HERE, per symbol, so the same full-history dict can be reused across
        every date the caller asks about."""
        per_symbol = {}
        for sym, df in universe_data.items():
            sl = calendar.slice_sessions(df, "time", t, n_back=len(df))
            if sl.empty or sl["time"].iloc[-1] != pd.Timestamp(t):
                continue  # didn't trade on t: halted, delisted, or not yet listed
            s = self._symbol_stats(sl)
            if s is not None:
                per_symbol[sym] = s

        if not per_symbol:
            return None

        n_total = len(per_symbol)
        n_adv = sum(1 for s in per_symbol.values() if s["advanced"])
        n_dec = sum(1 for s in per_symbol.values() if s["declined"])
        up_vol = sum(s["up_volume"] for s in per_symbol.values())
        down_vol = sum(s["down_volume"] for s in per_symbol.values())

        out = {
            "date": t, "n_total": n_total, "n_advance": n_adv, "n_decline": n_dec,
            "advance_decline_ratio": (n_adv / n_dec) if n_dec > 0 else np.nan,
            "up_down_volume_ratio": (up_vol / down_vol) if down_vol > 0 else np.nan,
        }
        for p in self.ma_periods:
            eligible = [s[f"above_ma{p}"] for s in per_symbol.values() if s[f"above_ma{p}"] is not None]
            out[f"pct_above_ma{p}"] = float(np.mean(eligible)) if eligible else np.nan
            out[f"n_eligible_ma{p}"] = len(eligible)

        nh = [s["new_high"] for s in per_symbol.values() if s["new_high"] is not None]
        nl = [s["new_low"] for s in per_symbol.values() if s["new_low"] is not None]
        out["n_new_highs"] = int(sum(nh)) if nh else np.nan
        out["n_new_lows"] = int(sum(nl)) if nl else np.nan
        out["n_eligible_52w"] = len(nh)
        return out


## `MarketBreadthDatasetBuilder`

One row per DATE across the reference calendar — a market-level companion
to Phase 3's `ResearchDatasetBuilder` (which builds one row per
`(symbol, date)`). Join the two on `date` when you need per-symbol features
alongside market-wide context.

In [ ]:
class MarketBreadthDatasetBuilder:
    def __init__(self, calendar: "TradingCalendar", engine: MarketBreadthEngine = None, sample_stride: int = 1):
        self.calendar = calendar
        self.engine = engine or MarketBreadthEngine()
        self.sample_stride = sample_stride

    def build(self, universe_data: dict, start_date=None, end_date=None) -> pd.DataFrame:
        dates = self.calendar.sessions
        if start_date is not None:
            dates = dates[dates >= pd.Timestamp(start_date)]
        if end_date is not None:
            dates = dates[dates <= pd.Timestamp(end_date)]
        dates = dates[::self.sample_stride]

        rows = []
        for t in dates:
            row = self.engine.compute_at(t, self.calendar, universe_data)
            if row is not None:
                rows.append(row)
        return pd.DataFrame(rows)


## Tests

Beyond the standard look-ahead check: the two eligibility/exclusion
behaviors above each get their own test, since they're easy to silently get
wrong in a way that still "runs fine" and produces plausible-looking
numbers.

In [ ]:
def test_breadth_look_ahead():
    universe = {f"SYM{i}": _synthetic_series(300, seed=100+i) for i in range(10)}
    cal = TradingCalendar(universe["SYM0"]["time"])
    eng = MarketBreadthEngine()
    t = universe["SYM0"]["time"].iloc[250]
    before = eng.compute_at(t, cal, universe)

    universe_mut = {k: v.copy() for k, v in universe.items()}
    for sym, df in universe_mut.items():
        mask = df["time"] > t
        df.loc[mask, ["open","high","low","close"]] = 999999.0
        df.loc[mask, "volume"] = 0
    after = eng.compute_at(t, cal, universe_mut)

    for k in before:
        a, b = before[k], after[k]
        if isinstance(a, float) and np.isnan(a):
            assert isinstance(b, float) and np.isnan(b), f"{k}: {a} vs {b}"
        else:
            assert a == b, f"BREADTH LOOK-AHEAD BUG in '{k}': {a} vs {b}"
    print(f"✓ MarketBreadthEngine look-ahead test passed ({len(before)} keys, {before['n_total']} symbols)")

test_breadth_look_ahead()


✓ MarketBreadthEngine look-ahead test passed (15 keys, 10 symbols)


In [ ]:
def test_breadth_excludes_ineligible_not_falsely_bearish():
    """New listings (only 30 bars) must be excluded from pct_above_ma200's
    denominator entirely (n_eligible_ma200==0), never counted as 0%."""
    universe = {f"NEW{i}": _synthetic_series(30, seed=200+i) for i in range(5)}
    cal = TradingCalendar(universe["NEW0"]["time"])
    eng = MarketBreadthEngine()
    t = universe["NEW0"]["time"].iloc[-1]
    out = eng.compute_at(t, cal, universe)
    assert out["n_eligible_ma200"] == 0
    assert np.isnan(out["pct_above_ma200"]), "must be NaN (no eligible symbols), not 0.0"
    assert out["n_eligible_ma20"] == 5, "all 5 symbols DO have enough bars for MA20"
    print("✓ Ineligibility test passed: pct_above_ma200=NaN with n_eligible=0 (not falsely 0%)")

test_breadth_excludes_ineligible_not_falsely_bearish()


def test_breadth_halted_symbol_excluded():
    """A symbol with no row on date t (halted/delisted) must be excluded
    entirely, not treated as flat/unchanged."""
    universe = {f"SYM{i}": _synthetic_series(300, seed=300+i) for i in range(5)}
    cal = TradingCalendar(universe["SYM0"]["time"])
    t = universe["SYM0"]["time"].iloc[250]
    universe["SYM0"] = universe["SYM0"][universe["SYM0"]["time"] != t].reset_index(drop=True)
    eng = MarketBreadthEngine()
    out = eng.compute_at(t, cal, universe)
    assert out["n_total"] == 4, f"halted symbol should be excluded, got n_total={out['n_total']}"
    print("✓ Halted-symbol test passed: excluded from n_total rather than counted as flat")

test_breadth_halted_symbol_excluded()


def test_breadth_dataset_builder():
    universe = {f"SYM{i}": _synthetic_series(300, seed=400+i) for i in range(8)}
    cal = TradingCalendar(universe["SYM0"]["time"])
    builder = MarketBreadthDatasetBuilder(cal, sample_stride=10)
    out = builder.build(universe)
    assert len(out) > 0
    assert {"date","pct_above_ma20","advance_decline_ratio","n_new_highs"}.issubset(out.columns)
    print(f"✓ MarketBreadthDatasetBuilder end-to-end test passed ({len(out)} rows)")

test_breadth_dataset_builder()
print("\nALL PHASE 6 TESTS PASSED")


✓ Ineligibility test passed: pct_above_ma200=NaN with n_eligible=0 (not falsely 0%)
✓ Halted-symbol test passed: excluded from n_total rather than counted as flat
✓ MarketBreadthDatasetBuilder end-to-end test passed (29 rows)

ALL PHASE 6 TESTS PASSED


## Usage — wiring into the rest of the pipeline

```python
cal = TradingCalendar.from_reference_ohlcv(vnindex_df)
universe_data = {s: store.fetch(s, "2018-01-01", "2026-08-01") for s in vn100_symbols}
universe_data = {s: df for s, df in universe_data.items() if df is not None and len(df) > 60}

breadth_builder = MarketBreadthDatasetBuilder(cal, sample_stride=5)
breadth = breadth_builder.build(universe_data)
breadth.to_parquet("data/features/market_breadth_v1.parquet", index=False)

# Join onto the per-symbol research dataset from Phase 3/5:
dataset = builder.build_for_universe(universe_data)   # FactorEngineV5-based, per (symbol,date)
dataset = dataset.merge(breadth, on="date", how="left")
```

Now `EmpiricalProbabilityEngine` (Phase 4) can condition on real market
context — e.g. `group_cols=["vsa_state", pd.cut(dataset["pct_above_ma20"], 3)]`
— to check whether a setup's edge actually depends on market breadth, or
holds regardless (spec §26's whole point).

## Roadmap update

Phase 6 delivered: `MarketBreadthEngine` + `MarketBreadthDatasetBuilder`,
the first universe-level (not per-symbol) component, with explicit
eligibility/exclusion handling for new listings and halted symbols.

Still ahead:
- **Phase 7** — Sector engine (sector-level breadth/RS, same
  universe-level pattern as this phase, grouped by sector instead of the
  whole market) + Setup engine (breakout/pullback/accumulation/distribution
  as explicit classifiers on top of Phase 5's factors).
- **Phase 8** — Exit engine (trend/flow/RS deterioration → exit risk score).
- **Phase 9** — Probability model (logistic regression + calibration).
- **Phase 10–12** — Backtest v2, walk-forward validation, ablation.

---
# PHASE 7 — Sector Engine + Setup Engine

## `SectorEngine`

Spec §25: sector-level context, same universe-level pattern as Phase 6's
`MarketBreadthEngine` — grouped by sector instead of the whole market.

**Requires an external `symbol -> sector` mapping.** This notebook has no
live source for VN sector classification wired up yet (that's a
`MarketDataStore.fetch`-style addition for a real run, e.g. vnstock's
company-overview/industry endpoints, cached the same way price data is).
`SectorEngine` takes that mapping as a plain argument so it isn't blocked on
that integration.

Per spec's own example (SSI strong + Securities sector strong = higher
conviction; DXG strong + Real Estate weak = reduce conviction),
`sector_rs_vs_market` is computed relative to the **same universe snapshot**
at date T — not a separately-fetched market index — so it's automatically
consistent with whatever universe you're running the pipeline on.

In [ ]:
class SectorEngine:
    def __init__(self, ma_periods=(20, 50), return_horizons=(5, 20, 60)):
        self.ma_periods = ma_periods
        self.horizons = return_horizons

    def _symbol_stats(self, df_slice: pd.DataFrame) -> Optional[dict]:
        need = max(max(self.ma_periods), max(self.horizons)) + 1
        if len(df_slice) < need:
            return None
        close = df_slice["close"]
        c_now = float(close.iloc[-1])
        stats = {}
        for h in self.horizons:
            stats[f"return_{h}d"] = c_now / close.iloc[-(h+1)] - 1 if len(close) > h else np.nan
        for p in self.ma_periods:
            ma = close.rolling(p).mean().iloc[-1]
            stats[f"above_ma{p}"] = bool(c_now > ma) if not pd.isna(ma) else None
        return stats

    def compute_at(self, t, calendar: "TradingCalendar", universe_data: dict, symbol_to_sector: dict) -> pd.DataFrame:
        per_symbol = {}
        for sym, df in universe_data.items():
            sector = symbol_to_sector.get(sym)
            if sector is None:
                continue
            sl = calendar.slice_sessions(df, "time", t, n_back=len(df))
            if sl.empty or sl["time"].iloc[-1] != pd.Timestamp(t):
                continue
            s = self._symbol_stats(sl)
            if s is not None:
                s["sector"] = sector
                per_symbol[sym] = s
        if not per_symbol:
            return pd.DataFrame()

        sdf = pd.DataFrame.from_dict(per_symbol, orient="index")
        market_ret20 = sdf["return_20d"].mean(skipna=True)

        rows = []
        for sector, g in sdf.groupby("sector"):
            row = {"date": t, "sector": sector, "n_symbols": len(g)}
            for h in self.horizons:
                row[f"sector_return_{h}d"] = g[f"return_{h}d"].mean(skipna=True)
            for p in self.ma_periods:
                elig = g[f"above_ma{p}"].dropna()
                row[f"sector_pct_above_ma{p}"] = float(elig.mean()) if len(elig) else np.nan
                row[f"sector_n_eligible_ma{p}"] = len(elig)
            row["sector_rs_vs_market"] = row["sector_return_20d"] - market_ret20 if not pd.isna(market_ret20) else np.nan
            rows.append(row)

        out = pd.DataFrame(rows)
        out["sector_rs_rank"] = out["sector_rs_vs_market"].rank(ascending=False, method="min")
        return out.sort_values("sector_rs_rank").reset_index(drop=True)


In [ ]:
def test_sector_engine_look_ahead():
    universe = {f"SYM{i}": _synthetic_series(300, seed=500+i) for i in range(12)}
    symbol_to_sector = {f"SYM{i}": ("BANK" if i < 6 else "REAL_ESTATE") for i in range(12)}
    cal = TradingCalendar(universe["SYM0"]["time"])
    eng = SectorEngine()
    t = universe["SYM0"]["time"].iloc[250]
    before = eng.compute_at(t, cal, universe, symbol_to_sector)

    universe_mut = {k: v.copy() for k, v in universe.items()}
    for sym, df in universe_mut.items():
        mask = df["time"] > t
        df.loc[mask, ["open","high","low","close"]] = 999999.0
    after = eng.compute_at(t, cal, universe_mut, symbol_to_sector)

    pd.testing.assert_frame_equal(before.reset_index(drop=True), after.reset_index(drop=True))
    print(f"✓ SectorEngine look-ahead test passed ({len(before)} sectors)")

test_sector_engine_look_ahead()


def test_sector_engine_recovers_real_effect():
    """A genuinely stronger sector must rank #1 by RS - not just run without error."""
    universe, symbol_to_sector = {}, {}
    for i in range(8):
        universe[f"STRONG{i}"] = _synthetic_series(300, seed=600+i, drift=0.0025)
        symbol_to_sector[f"STRONG{i}"] = "STRONG_SECTOR"
    for i in range(8):
        universe[f"WEAK{i}"] = _synthetic_series(300, seed=700+i, drift=-0.0015)
        symbol_to_sector[f"WEAK{i}"] = "WEAK_SECTOR"
    cal = TradingCalendar(universe["STRONG0"]["time"])
    eng = SectorEngine()
    t = universe["STRONG0"]["time"].iloc[280]
    out = eng.compute_at(t, cal, universe, symbol_to_sector)
    top = out.iloc[0]
    assert top["sector"] == "STRONG_SECTOR" and top["sector_rs_rank"] == 1, out
    print("✓ SectorEngine recovers real cross-sector effect:", top["sector"], "ranked #1")

test_sector_engine_recovers_real_effect()


✓ SectorEngine look-ahead test passed (2 sectors)
✓ SectorEngine recovers real cross-sector effect: STRONG_SECTOR ranked #1


## `SetupEngine`

Spec §28-32: BREAKOUT / PULLBACK / ACCUMULATION / DISTRIBUTION as explicit
classifiers on top of `FactorEngineV5`'s features (Phase 5), plus a
resistance/support helper computed here.

Per spec §29 ("do not hard-code all thresholds permanently — make them
configurable"), every threshold is a constructor parameter, not a number
buried in a condition — so Phase 12's parameter-sensitivity sweep (spec §57)
has something real to sweep over.

**One correctness detail worth calling out:** resistance/support levels use
`rolling(N).max().shift(1)` — the extra `.shift(1)` means today's own bar is
never included in the level it's being compared against. Without it, a huge
move on the signal bar itself would raise its own resistance ceiling and a
real breakout could quietly fail to register. Verified with a dedicated
test, not just the standard look-ahead check.

A bar can satisfy more than one setup's conditions — all matching flags are
returned, plus `primary_setup` for callers that want a single label
(priority: DISTRIBUTION > BREAKOUT > PULLBACK > ACCUMULATION > NONE —
deterioration signals take priority since missing an exit risk is usually
costlier than missing an entry).

REVERSAL (spec §28's fifth setup) isn't included yet — it depends on
`MarketStateEngine`'s lifecycle tracking (BASE→ACCUMULATION→...→MARKDOWN,
spec §34), which is Phase 8 territory alongside the Exit Engine.

In [ ]:
class SetupEngine:
    PRIORITY = ["DISTRIBUTION", "BREAKOUT", "PULLBACK", "ACCUMULATION", "NONE"]

    def __init__(self, resistance_lookback: int = 20, support_lookback: int = 20,
                 breakout_vol_ratio_min: float = 1.5, breakout_close_pos_min: float = 0.7,
                 pullback_band_pct: float = 0.02, pullback_vol_ratio_max: float = 0.9,
                 accum_vol_ratio_max: float = 0.7, accum_volatility_max_pct: float = 0.02,
                 distribution_vol_ratio_min: float = 1.4):
        self.res_lb, self.sup_lb = resistance_lookback, support_lookback
        self.bo_vol_min, self.bo_close_pos_min = breakout_vol_ratio_min, breakout_close_pos_min
        self.pb_band, self.pb_vol_max = pullback_band_pct, pullback_vol_ratio_max
        self.ac_vol_max, self.ac_vola_max = accum_vol_ratio_max, accum_volatility_max_pct
        self.di_vol_min = distribution_vol_ratio_min

    def _structure(self, df_slice: pd.DataFrame) -> Optional[dict]:
        need = max(self.res_lb, self.sup_lb) + 2
        if len(df_slice) < need:
            return None
        high, low, close = df_slice["high"], df_slice["low"], df_slice["close"]
        resistance = high.rolling(self.res_lb).max().shift(1).iloc[-1]  # shift(1): excludes today's own bar
        support = low.rolling(self.sup_lb).min().shift(1).iloc[-1]
        if pd.isna(resistance) or pd.isna(support):
            return None
        return {"resistance": float(resistance), "support": float(support),
                "broke_resistance": bool(close.iloc[-1] > resistance),
                "broke_support": bool(close.iloc[-1] < support)}

    def classify(self, df_slice: pd.DataFrame, features: dict,
                 sector_rs_vs_market: float = None) -> Optional[dict]:
        """features = FactorEngineV5.feature_at(df_slice) for the SAME slice.
        sector_rs_vs_market is optional SectorEngine context for the same date."""
        struct = self._structure(df_slice)
        if struct is None or features is None:
            return None

        vol_ratio = features.get("volume_ratio", np.nan)
        close_pos = features.get("close_pos", np.nan)
        roc = features.get("roc_1d", np.nan)
        above_cloud = features.get("ichi_above_cloud_d", False)
        kijun = features.get("ichi_kijun_d", np.nan)
        ema20 = features.get("ema20", np.nan)
        close_now = float(df_slice["close"].iloc[-1])
        sector_ok = (sector_rs_vs_market is None) or (sector_rs_vs_market > 0)

        flags = {}
        flags["BREAKOUT"] = bool(
            struct["broke_resistance"]
            and not pd.isna(vol_ratio) and vol_ratio >= self.bo_vol_min
            and not pd.isna(close_pos) and close_pos >= self.bo_close_pos_min
            and sector_ok
        )

        near_kijun = (not pd.isna(kijun)) and abs(close_now - kijun) / close_now <= self.pb_band
        near_ema20 = (not pd.isna(ema20)) and abs(close_now - ema20) / close_now <= self.pb_band
        flags["PULLBACK"] = bool(
            above_cloud and (near_kijun or near_ema20)
            and not pd.isna(vol_ratio) and vol_ratio <= self.pb_vol_max
            and close_now > struct["support"]
        )

        volatility = features.get("volatility_20d", np.nan)
        flags["ACCUMULATION"] = bool(
            not pd.isna(vol_ratio) and vol_ratio <= self.ac_vol_max
            and not pd.isna(volatility) and volatility <= self.ac_vola_max
            and not struct["broke_resistance"] and not struct["broke_support"]
        )

        bear_div = bool(features.get("rsi_bear_div") or features.get("macd_bear_div"))
        flags["DISTRIBUTION"] = bool(
            above_cloud and bear_div  # only meaningful as a deteriorating uptrend
            and not pd.isna(vol_ratio) and vol_ratio >= self.di_vol_min
            and not pd.isna(close_pos) and close_pos <= 0.4
            and not pd.isna(roc) and roc < 0
        )

        primary = next((s for s in self.PRIORITY[:-1] if flags.get(s)), "NONE")
        return {**flags, "primary_setup": primary, "resistance": struct["resistance"], "support": struct["support"]}


In [ ]:
def _fake_features(df_slice, **overrides):
    """Minimal stand-in for FactorEngineV5 output, for isolated SetupEngine tests."""
    close = df_slice["close"].iloc[-1]
    base = {"volume_ratio": 1.0, "close_pos": 0.5, "roc_1d": 0.0,
            "ichi_above_cloud_d": True, "ichi_kijun_d": close, "ema20": close,
            "volatility_20d": 0.01, "rsi_bear_div": False, "macd_bear_div": False}
    base.update(overrides)
    return base


def test_breakout_requires_prior_bars_only():
    """Resistance must come from bars BEFORE today - a huge move on the
    signal bar itself must not raise its own resistance level."""
    n = 60
    sessions = pd.bdate_range("2023-01-01", periods=n)
    price = np.full(n, 100.0)
    price[-1] = 120.0  # today's bar spikes hard
    df = pd.DataFrame({"time": sessions, "open": price, "close": price,
                        "high": price + 1, "low": price - 1, "volume": np.full(n, 100000)})
    eng = SetupEngine(resistance_lookback=20)
    struct = eng._structure(df)
    assert struct["resistance"] == 101.0, f"resistance must be from PRIOR bars only, got {struct['resistance']}"
    assert struct["broke_resistance"] is True
    print("✓ Breakout structure test passed: resistance from prior bars only, today's spike correctly flagged as a break")

test_breakout_requires_prior_bars_only()


def test_setup_engine_look_ahead():
    df = _synthetic_series(200, seed=41)
    cal = TradingCalendar(df["time"])
    eng = SetupEngine()
    t = df["time"].iloc[150]
    sl_before = cal.slice_sessions(df, "time", t, n_back=len(df))
    feat = _fake_features(sl_before, volume_ratio=1.8, close_pos=0.85)
    before = eng.classify(sl_before, feat)

    df_mut = df.copy()
    mask = df_mut["time"] > t
    df_mut.loc[mask, ["open","high","low","close"]] = 999999.0
    sl_after = cal.slice_sessions(df_mut, "time", t, n_back=len(df_mut))
    after = eng.classify(sl_after, feat)

    assert before == after, f"SETUP LOOK-AHEAD BUG: {before} vs {after}"
    print("✓ SetupEngine look-ahead test passed:", before["primary_setup"])

test_setup_engine_look_ahead()


def test_pullback_requires_uptrend_and_support_held():
    n = 60
    sessions = pd.bdate_range("2023-01-01", periods=n)
    price = np.linspace(90, 110, n)
    price[-1] = 108.0  # slight pullback, still above support
    df = pd.DataFrame({"time": sessions, "open": price, "close": price,
                        "high": price + 1, "low": price - 1, "volume": np.full(n, 100000)})
    eng = SetupEngine(support_lookback=20, resistance_lookback=20)
    feat = _fake_features(df, volume_ratio=0.6, ichi_above_cloud_d=True, ichi_kijun_d=108.5, ema20=108.5)
    out = eng.classify(df, feat)
    assert out["PULLBACK"] is True, out
    print("✓ Pullback setup correctly identified:", out["primary_setup"])

    price2 = price.copy(); price2[-1] = 70.0  # crashes through support
    df2 = df.copy(); df2["close"]=price2; df2["open"]=price2; df2["high"]=price2+1; df2["low"]=price2-1
    feat2 = _fake_features(df2, volume_ratio=0.6, ichi_above_cloud_d=True, ichi_kijun_d=108.5, ema20=108.5)
    out2 = eng.classify(df2, feat2)
    assert out2["PULLBACK"] is False, "must not flag PULLBACK when support is broken"
    print("✓ Pullback correctly rejected when support does not hold")

test_pullback_requires_uptrend_and_support_held()
print("\nALL PHASE 7 TESTS PASSED")


✓ Breakout structure test passed: resistance from prior bars only, today's spike correctly flagged as a break
✓ SetupEngine look-ahead test passed: NONE
✓ Pullback setup correctly identified: PULLBACK
✓ Pullback correctly rejected when support does not hold

ALL PHASE 7 TESTS PASSED


## Usage — wiring into the pipeline

```python
symbol_to_sector = {...}  # from vnstock company-overview/industry data, cached like price data

sector_eng = SectorEngine()
setup_eng = SetupEngine()
fe = FactorEngineV5()

records = []
for t in cal.sessions[::5]:
    sector_stats = sector_eng.compute_at(t, cal, universe_data, symbol_to_sector)
    sector_rs_by_symbol = {}  # map each symbol's sector RS for that date, for setup_eng's sector_ok gate

    for sym, df in universe_data.items():
        sl = cal.slice_sessions(df, "time", t, n_back=len(df))
        feat = fe.feature_at(sl)
        if feat is None:
            continue
        sector = symbol_to_sector.get(sym)
        sec_row = sector_stats[sector_stats["sector"] == sector]
        sector_rs = float(sec_row["sector_rs_vs_market"].iloc[0]) if not sec_row.empty else None

        setup = setup_eng.classify(sl, feat, sector_rs_vs_market=sector_rs)
        if setup is None:
            continue
        records.append({"symbol": sym, "date": t, "sector": sector, **feat, **setup})

setups_df = pd.DataFrame(records)
```

Now `EmpiricalProbabilityEngine` (Phase 4) can finally group by a REAL
setup column instead of a proxy:

```python
eng.compute(labeled_df.merge(setups_df, on=["symbol","date"]), group_cols=["primary_setup"])
```
— which is the whole point of spec §15/§16: is `primary_setup=="BREAKOUT"`
actually worth more than `"PULLBACK"` historically, or not?

## Roadmap update

Phase 7 delivered: `SectorEngine` (sector-level breadth/RS relative to the
same universe snapshot) and `SetupEngine` (BREAKOUT/PULLBACK/ACCUMULATION/
DISTRIBUTION classifiers with configurable thresholds, resistance/support
from prior bars only).

Still ahead:
- **Phase 8** — `MarketStateEngine` (BASE→ACCUMULATION→BREAKOUT→MARKUP→
  EXHAUSTION→DISTRIBUTION→MARKDOWN lifecycle, spec §34) + `ExitRiskEngine`
  (trend/flow/RS deterioration → 0-100 exit risk, spec §35-38) + REVERSAL
  setup (depends on state-transition tracking).
- **Phase 9** — Probability model: logistic regression on the now-complete
  feature/setup/sector dataset, plus calibration analysis.
- **Phase 10–12** — Backtest v2 (fees/slippage/price-limits), walk-forward
  validation, ablation + parameter sensitivity — this is where every
  threshold introduced in Phase 5-7 (`breakout_vol_ratio_min`,
  `pullback_band_pct`, etc.) gets swept per spec §57 to check whether it's
  actually robust or just a lucky number.

---
# PHASE 8 — `MarketStateEngine` + `ExitRiskEngine`

## `MarketStateEngine`

Spec §34: `BASE → ACCUMULATION → BREAKOUT → MARKUP → EXHAUSTION →
DISTRIBUTION → MARKDOWN`. `UNKNOWN` is a first-class output, not a fallback
to hide — spec: *"do not force every stock into a state if evidence is
insufficient."*

**This is a different shape of component than everything in Phases 3-7.**
Every prior engine is stateless: `feature_at(T)` only ever needs the price
history through T. A lifecycle stage is inherently *sequential* — "MARKUP"
only means something in the context of what state came before it. So
`next_state(prev_state, setup_today, features_today)` takes the previous
state as an explicit input, and the sequence is built by walking dates in
order and threading that state through.

This shape is exactly why it's still look-ahead safe despite being
stateful: `next_state` only ever looks at *today's* evidence plus
`prev_state`, and `prev_state` was itself only ever derived from data
through yesterday. There's no step where a later date's information could
leak backward — verified below by re-deriving the state at day 8 from data
truncated at day 8, and checking it matches the state read off the full
14-day run.

In [ ]:
class MarketStateEngine:
    STATES = ["UNKNOWN", "BASE", "ACCUMULATION", "BREAKOUT", "MARKUP",
              "EXHAUSTION", "DISTRIBUTION", "MARKDOWN"]

    def __init__(self, exhaustion_adx_min: float = 40.0, exhaustion_extension_pct: float = 0.25):
        self.exhaustion_adx_min = exhaustion_adx_min
        self.exhaustion_ext_pct = exhaustion_extension_pct

    def next_state(self, prev_state: str, setup: Optional[dict], features: Optional[dict]) -> str:
        if setup is None or features is None:
            return "UNKNOWN"  # insufficient evidence - never force a guess

        above_cloud = bool(features.get("ichi_above_cloud_d", False))
        below_cloud = bool(features.get("ichi_below_cloud_d", False))
        adx = features.get("adx", np.nan)
        bear_div = bool(features.get("rsi_bear_div") or features.get("macd_bear_div"))
        ema20 = features.get("ema20", np.nan)
        close = features.get("close", np.nan)
        extension_pct = abs(close / ema20 - 1) if (not pd.isna(ema20) and ema20) else np.nan

        is_breakout = bool(setup.get("BREAKOUT"))
        is_accum = bool(setup.get("ACCUMULATION"))
        is_dist = bool(setup.get("DISTRIBUTION"))

        # deterioration evidence takes priority over continuation evidence -
        # missing an exit is usually costlier than missing a continuation
        if prev_state in ("MARKUP", "EXHAUSTION", "DISTRIBUTION") and below_cloud:
            return "MARKDOWN"
        if is_dist:
            return "DISTRIBUTION"
        if prev_state in ("MARKUP", "EXHAUSTION") and not above_cloud:
            return "DISTRIBUTION"  # lost the cloud without a clean breakdown yet

        if prev_state == "MARKUP" and not pd.isna(adx) and adx >= self.exhaustion_adx_min \
                and not pd.isna(extension_pct) and extension_pct >= self.exhaustion_ext_pct and bear_div:
            return "EXHAUSTION"

        if is_breakout and prev_state in ("BASE", "ACCUMULATION", "UNKNOWN"):
            return "BREAKOUT"
        if prev_state == "BREAKOUT":
            return "MARKUP" if above_cloud else "BASE"  # follow-through confirmed, or failed breakout
        if prev_state in ("MARKUP", "EXHAUSTION") and above_cloud:
            return prev_state  # persist while structurally intact

        if is_accum:
            return "ACCUMULATION"
        if prev_state == "MARKDOWN" and not below_cloud:
            return "BASE"  # recovering - back to square one, not straight to markup

        return prev_state if prev_state != "UNKNOWN" else "BASE"

    def run_sequence(self, df_full: pd.DataFrame, calendar: "TradingCalendar",
                      feature_engine, setup_engine, dates=None) -> pd.DataFrame:
        if dates is None:
            dates = df_full["time"]
        rows = []
        state = "UNKNOWN"
        for t in dates:
            sl = calendar.slice_sessions(df_full, "time", t, n_back=len(df_full))
            feat = feature_engine.feature_at(sl)
            setup = setup_engine.classify(sl, feat) if feat is not None else None
            new_state = self.next_state(state, setup, feat)
            rows.append({"date": t, "state": new_state, "prev_state": state,
                         "transitioned": new_state != state})
            state = new_state
        return pd.DataFrame(rows)


## Tests

`ScriptedFeatureEngine`/`ScriptedSetupEngine` are test doubles that return a
pre-written sequence of feature/setup snapshots — the real
`FactorEngineV5`/`SetupEngine` combination is already covered by their own
Phase 5/7 tests, so this section tests the **state machine's transition
logic** in isolation, driving it through a scripted textbook lifecycle
rather than trying to construct real OHLCV that happens to produce every
stage.

In [ ]:
class ScriptedFeatureEngine:
    def __init__(self, script: dict):
        self.script = script  # {row_index: features_dict}
    def feature_at(self, df_slice):
        return self.script.get(len(df_slice) - 1)

class ScriptedSetupEngine:
    def __init__(self, script: dict):
        self.script = script
    def classify(self, df_slice, features):
        return self.script.get(len(df_slice) - 1)


def test_unknown_when_insufficient_evidence():
    df = _synthetic_series(20, seed=91)
    cal = TradingCalendar(df["time"])
    feat_script = {i: None for i in range(len(df))}
    setup_script = {i: None for i in range(len(df))}
    eng = MarketStateEngine()
    out = eng.run_sequence(df, cal, ScriptedFeatureEngine(feat_script), ScriptedSetupEngine(setup_script))
    assert (out["state"] == "UNKNOWN").all(), "must stay UNKNOWN, never force a guess without evidence"
    print("✓ UNKNOWN-when-insufficient test passed")

test_unknown_when_insufficient_evidence()


✓ UNKNOWN-when-insufficient test passed


In [ ]:
def test_full_lifecycle_sequence():
    """Script a textbook path and check the engine recovers the SAME ORDER
    of distinct stages, not necessarily every single day matching."""
    n = 14
    df = _synthetic_series(n, seed=92)
    cal = TradingCalendar(df["time"])

    def feat(above_cloud=False, below_cloud=False, adx=15, bear_div=False, ema20=100, close=100):
        return {"ichi_above_cloud_d": above_cloud, "ichi_below_cloud_d": below_cloud, "adx": adx,
                "rsi_bear_div": bear_div, "macd_bear_div": False, "ema20": ema20, "close": close,
                "volume_ratio": 1.0, "close_pos": 0.5}
    def setup(breakout=False, accum=False, dist=False, support=90):
        return {"BREAKOUT": breakout, "ACCUMULATION": accum, "DISTRIBUTION": dist,
                "PULLBACK": False, "primary_setup": "NONE", "resistance": 110, "support": support}

    feat_script, setup_script = {}, {}
    feat_script[0]=feat(); setup_script[0]=setup()                                     # BASE
    feat_script[1]=feat(); setup_script[1]=setup()
    feat_script[2]=feat(); setup_script[2]=setup(accum=True)                           # ACCUMULATION
    feat_script[3]=feat(); setup_script[3]=setup(accum=True)
    feat_script[4]=feat(above_cloud=True); setup_script[4]=setup(breakout=True)        # BREAKOUT
    feat_script[5]=feat(above_cloud=True); setup_script[5]=setup()                     # MARKUP
    feat_script[6]=feat(above_cloud=True); setup_script[6]=setup()
    feat_script[7]=feat(above_cloud=True); setup_script[7]=setup()
    feat_script[8]=feat(above_cloud=True, adx=45, bear_div=True, ema20=100, close=130) # EXHAUSTION
    setup_script[8]=setup()
    feat_script[9]=feat(above_cloud=True); setup_script[9]=setup(dist=True)            # DISTRIBUTION
    feat_script[10]=feat(above_cloud=True); setup_script[10]=setup()
    feat_script[11]=feat(above_cloud=False, below_cloud=False); setup_script[11]=setup()
    feat_script[12]=feat(below_cloud=True); setup_script[12]=setup()                   # MARKDOWN
    feat_script[13]=feat(below_cloud=True); setup_script[13]=setup()

    eng = MarketStateEngine()
    out = eng.run_sequence(df, cal, ScriptedFeatureEngine(feat_script), ScriptedSetupEngine(setup_script))

    visited_in_order = list(dict.fromkeys(out["state"]))
    expected_order = ["BASE","ACCUMULATION","BREAKOUT","MARKUP","EXHAUSTION","DISTRIBUTION","MARKDOWN"]
    it = iter(visited_in_order)
    assert all(s in it for s in expected_order), f"lifecycle order not recovered: {visited_in_order}"
    print("✓ Full lifecycle sequence test passed:", " -> ".join(expected_order))

test_full_lifecycle_sequence()


✓ Full lifecycle sequence test passed: BASE -> ACCUMULATION -> BREAKOUT -> MARKUP -> EXHAUSTION -> DISTRIBUTION -> MARKDOWN


In [ ]:
def test_state_sequence_no_look_ahead():
    """Re-derive the state at day 8 from data TRUNCATED at day 8, and check
    it matches the state read from a full 14-day run - the sequential
    design must not accidentally depend on rows beyond the current date."""
    n = 14
    df = _synthetic_series(n, seed=93)
    cal = TradingCalendar(df["time"])
    feat_script = {i: {"ichi_above_cloud_d": i >= 4, "ichi_below_cloud_d": False, "adx": 20,
                        "rsi_bear_div": False, "macd_bear_div": False, "ema20": 100, "close": 100+i,
                        "volume_ratio": 1.0, "close_pos": 0.5} for i in range(n)}
    setup_script = {i: {"BREAKOUT": i==4, "ACCUMULATION": i in (2,3), "DISTRIBUTION": False,
                         "PULLBACK": False, "primary_setup": "NONE", "resistance": 110, "support": 90}
                     for i in range(n)}
    fe, se = ScriptedFeatureEngine(feat_script), ScriptedSetupEngine(setup_script)
    eng = MarketStateEngine()

    full_seq = eng.run_sequence(df, cal, fe, se)
    state_full = full_seq.iloc[8]["state"]

    truncated_df = df.iloc[:9].copy()  # day 9 onward literally doesn't exist here
    truncated_seq = eng.run_sequence(truncated_df, cal, fe, se)
    state_truncated = truncated_seq.iloc[8]["state"]

    assert state_full == state_truncated, f"STATE LOOK-AHEAD BUG: full={state_full} vs truncated={state_truncated}"
    print(f"✓ No-look-ahead sequential test passed: state at day 8 = '{state_full}' regardless of future data")

test_state_sequence_no_look_ahead()
print("\nALL MarketStateEngine TESTS PASSED")


✓ No-look-ahead sequential test passed: state at day 8 = 'MARKUP' regardless of future data

ALL MarketStateEngine TESTS PASSED


## `ExitRiskEngine`

Spec §35-38: `exit_risk` 0-100 from trend/momentum/flow/RS deterioration +
distribution + structure breakdown, banded per spec §37 (0-20 HEALTHY,
20-40 WATCH, 40-60 DETERIORATING, 60-80 REDUCE, 80-100 EXIT) — spec's own
words call these *"initial semantic bands only... eventually validated
statistically"*, which is Phase 9-12's job once there's enough labeled
history, not this one.

**Spec §38, enforced structurally, not just by comment:** *"Do NOT sell
solely because price < MA9."* Every component below requires *structural*
evidence — below the Ichimoku cloud (not a short-term MA cross), a
confirmed bearish divergence (not just RSI being high), a broken multi-week
support level, an explicit `DISTRIBUTION` setup flag. And critically, no
single component's weight reaches the REDUCE threshold (60) on its own —
verified with a test, not just chosen and assumed safe.

In [ ]:
class ExitRiskEngine:
    WEIGHTS = {
        "trend_deterioration": 22, "momentum_deterioration": 18, "flow_deterioration": 18,
        "rs_deterioration": 14, "distribution": 16, "structure_break": 12,
    }  # sums to 100; no single component >= 60 (REDUCE threshold) - see test below

    BANDS = [(20, "HEALTHY"), (40, "WATCH"), (60, "DETERIORATING"), (80, "REDUCE"), (101, "EXIT")]

    def _band(self, score: float) -> str:
        for upper, label in self.BANDS:
            if score < upper:
                return label
        return "EXIT"

    def compute_at(self, df_slice, features: dict, setup: dict,
                    market_state: str = None, sector_rs_vs_market: float = None) -> Optional[dict]:
        if features is None or setup is None:
            return None

        above_cloud = bool(features.get("ichi_above_cloud_d", False))
        below_cloud = bool(features.get("ichi_below_cloud_d", False))
        kijun = features.get("ichi_kijun_d", np.nan)
        close = features.get("close", np.nan)
        bear_div = bool(features.get("rsi_bear_div") or features.get("macd_bear_div"))
        vsa_state = features.get("vsa_state", "NEUTRAL")
        vol_ratio = features.get("volume_ratio", np.nan)
        close_pos = features.get("close_pos", np.nan)

        components = {
            "trend_deterioration": float(below_cloud or (not pd.isna(kijun) and not pd.isna(close) and close < kijun)),
            "momentum_deterioration": float(bear_div),
            "flow_deterioration": float(
                vsa_state in ("PHAN_PHOI", "CAN_CAU")
                or (not pd.isna(vol_ratio) and not pd.isna(close_pos) and vol_ratio >= 1.4 and close_pos <= 0.35)
            ),
            "rs_deterioration": float(sector_rs_vs_market is not None and sector_rs_vs_market < 0),
            "distribution": float(bool(setup.get("DISTRIBUTION")) or market_state == "DISTRIBUTION"),
            "structure_break": float(bool(setup.get("broke_support")) or market_state == "MARKDOWN"),
        }
        score = sum(components[k] * self.WEIGHTS[k] for k in components)
        return {"exit_risk": round(score, 1), "band": self._band(score),
                **{f"c_{k}": v for k, v in components.items()}}


In [ ]:
def test_no_single_component_reaches_reduce():
    eng = ExitRiskEngine()
    for k, w in eng.WEIGHTS.items():
        assert w < 60, f"component '{k}' alone (weight={w}) would reach REDUCE threshold on its own"
    print("✓ No single component can alone push exit_risk into REDUCE/EXIT (spec #38 enforced structurally)")

test_no_single_component_reaches_reduce()


def test_single_weak_signal_stays_low():
    """Only ONE deterioration signal fires (bearish divergence) -> must land
    in HEALTHY/WATCH, not REDUCE/EXIT - the concrete version of spec #38's
    'temporary pullback != distribution'."""
    eng = ExitRiskEngine()
    features = {"ichi_above_cloud_d": True, "ichi_below_cloud_d": False, "ichi_kijun_d": 95,
                "close": 100, "adx": 20, "rsi_bear_div": True, "macd_bear_div": False,
                "vsa_state": "NEUTRAL", "volume_ratio": 1.0, "close_pos": 0.5}
    setup = {"DISTRIBUTION": False, "broke_support": False}
    out = eng.compute_at(None, features, setup)
    assert out["band"] in ("HEALTHY", "WATCH"), out
    print(f"✓ Single weak signal test passed: exit_risk={out['exit_risk']}, band={out['band']}")

test_single_weak_signal_stays_low()


def test_multiple_corroborating_signals_trigger_high_risk():
    """Structure break + distribution setup + bear divergence + weak flow
    together -> must reach REDUCE/EXIT, proving the engine CAN escalate when
    evidence genuinely accumulates."""
    eng = ExitRiskEngine()
    features = {"ichi_above_cloud_d": False, "ichi_below_cloud_d": True, "ichi_kijun_d": 105,
                "close": 95, "adx": 22, "rsi_bear_div": True, "macd_bear_div": True,
                "vsa_state": "PHAN_PHOI", "volume_ratio": 1.8, "close_pos": 0.2}
    setup = {"DISTRIBUTION": True, "broke_support": True}
    out = eng.compute_at(None, features, setup, sector_rs_vs_market=-0.05)
    assert out["band"] in ("REDUCE", "EXIT"), out
    print(f"✓ Multiple corroborating signals test passed: exit_risk={out['exit_risk']}, band={out['band']}")

test_multiple_corroborating_signals_trigger_high_risk()
print("\nALL ExitRiskEngine TESTS PASSED")


✓ No single component can alone push exit_risk into REDUCE/EXIT (spec #38 enforced structurally)
✓ Single weak signal test passed: exit_risk=18.0, band=HEALTHY
✓ Multiple corroborating signals test passed: exit_risk=100.0, band=EXIT

ALL ExitRiskEngine TESTS PASSED


## Usage — wiring into the full pipeline

```python
fe, se = FactorEngineV5(), SetupEngine()
state_eng = MarketStateEngine()
exit_eng = ExitRiskEngine()

state_seq = state_eng.run_sequence(df_full, cal, fe, se)  # per symbol, walked in date order

records = []
state_by_date = dict(zip(state_seq["date"], state_seq["state"]))
for t in cal.sessions[::5]:
    sl = cal.slice_sessions(df_full, "time", t, n_back=len(df_full))
    feat = fe.feature_at(sl)
    if feat is None:
        continue
    setup = se.classify(sl, feat)
    market_state = state_by_date.get(t)
    exit_info = exit_eng.compute_at(sl, feat, setup, market_state=market_state)
    records.append({"date": t, "state": market_state, **exit_info})

pd.DataFrame(records)
```

Combined with `SectorEngine` (Phase 7), pass `sector_rs_vs_market` into
`compute_at` too — a stock deteriorating alongside a deteriorating sector is
a stronger signal than one deteriorating in isolation.

## Roadmap update

Phase 8 delivered: `MarketStateEngine` (sequential, causal-by-construction
lifecycle tracking with `UNKNOWN` as a first-class state) and
`ExitRiskEngine` (0-100 exit risk from structural deterioration signals
only, with spec §38's "not just MA9" constraint enforced by weight design,
not comment).

Still ahead — and this is where the engine actually gets *validated* rather
than just *built*:
- **Phase 9** — Probability model: logistic regression on the full
  feature/setup/sector/state dataset assembled across Phases 3-8, plus
  calibration analysis (predicted P vs. observed frequency, spec §41).
- **Phase 10** — Backtest v2: wire `FillSimulator` (Phase 2) + fees +
  slippage + Vietnamese price-limit (trần/sàn) handling into a proper
  execution/portfolio/accounting architecture (spec §45-49).
- **Phase 11** — Walk-forward validation: train/test splits by calendar
  period, purged/embargoed to prevent the overlapping-label leakage the
  triple-barrier horizons create (spec §53-54).
- **Phase 12** — Ablation (which of Phase 5-7's factors/setups actually
  carry alpha, spec §55-56) + parameter sensitivity (is
  `exhaustion_adx_min=40` a stable region or a lucky number, spec §57-58) —
  this is the step that finally answers whether any of Phases 5-8's
  hand-designed rules should survive contact with `EmpiricalProbabilityEngine`.

---
# PHASE 9 — Probability Model + Calibration Analysis

Spec §39-41: first model is **Logistic Regression**, target is
`P(TP_FIRST)` (or `P(+X% within ND)` more generally). Tree-based models
(LightGBM/XGBoost/CatBoost) are explicitly **not** the starting point —
spec §39: *"do NOT start with complex ML."*

This phase also delivers the piece every prior phase has been implicitly
building toward: an actual **out-of-sample evaluation**, not just a model
that fits. Every metric reported below is measured on data the model never
saw during fitting.

## `time_split` — out-of-time split with embargo

A single chronological split (train = earlier dates, test = later dates) —
not the full rolling walk-forward from spec §53, which is Phase 11's job.
But even a baseline model needs *some* honest separation between what it's
fit on and what it's judged on, so this does the minimum right thing now
rather than deferring it.

**The embargo matters concretely here:** `TripleBarrierLabeler`'s labels
look up to `horizon_sessions` sessions into the future. A training signal
dated right before `test_start` would have its *outcome label* computed
from price data overlapping the test period. That's not a feature leak —
test features never touch train labels — but it's still overlapping
information between the two sets. `embargo_sessions` drops that boundary
zone from training entirely.

In [ ]:
def time_split(df: pd.DataFrame, calendar: "TradingCalendar", date_col: str,
                test_start, embargo_sessions: int = 0):
    train_end = calendar.add_sessions(test_start, -(embargo_sessions + 1))
    if train_end is None:
        train_end = pd.Timestamp(test_start) - pd.Timedelta(days=1)
    train = df[df[date_col] <= pd.Timestamp(train_end)].copy()
    test = df[df[date_col] >= pd.Timestamp(test_start)].copy()
    return train, test


def test_time_split_is_chronological_with_embargo():
    dates = pd.bdate_range("2023-01-01", periods=200)
    cal = TradingCalendar(dates)
    df = pd.DataFrame({"date": dates, "x": range(200)})
    test_start = dates[150]
    train, test = time_split(df, cal, "date", test_start, embargo_sessions=10)

    assert train["date"].max() < test["date"].min(), "train must end strictly before test begins"
    gap_sessions = cal.session_index(test["date"].min()) - cal.session_index(train["date"].max()) - 1
    assert gap_sessions >= 10, f"embargo gap too small: {gap_sessions} sessions"
    print(f"✓ time_split test passed: train ends {train['date'].max().date()}, "
          f"test starts {test['date'].min().date()}, embargo={gap_sessions} sessions")

test_time_split_is_chronological_with_embargo()


✓ time_split test passed: train ends 2023-07-14, test starts 2023-07-31, embargo=10 sessions


## `ProbabilityModel`

Standardizes numeric features, one-hot encodes categorical setup/state
columns, fits `sklearn.linear_model.LogisticRegression`. `fit()` only ever
sees the training split passed to it — the class has no path to touch test
data, so the look-ahead boundary here is the same one `time_split` already
established, not a new one this class could get wrong independently.

`coefficients()` is exposed deliberately — spec §39 wants an
**interpretable** baseline, not a black box. If a genuinely predictive
feature (from Phase 5-7's factors/setups) doesn't outrank noise in
`|coefficient|`, that's informative on its own.

In [ ]:
class ProbabilityModel:
    def __init__(self, numeric_cols: list, categorical_cols: list = None, C: float = 1.0):
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols or []
        self.C = C
        self.scaler = StandardScaler()
        self.model = LogisticRegression(C=C, max_iter=1000)
        self._dummy_columns = None
        self._train_median = None

    def _design_matrix(self, df: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
        num = df[self.numeric_cols].copy()
        if fit:
            self._train_median = num.median()
        num = num.fillna(self._train_median)
        cat = pd.get_dummies(df[self.categorical_cols].astype(str), prefix=self.categorical_cols) \
            if self.categorical_cols else pd.DataFrame(index=df.index)
        X = pd.concat([num, cat], axis=1)
        if fit:
            self._dummy_columns = X.columns.tolist()
        else:
            X = X.reindex(columns=self._dummy_columns, fill_value=0)
        return X

    def fit(self, train_df: pd.DataFrame, target_col: str = "outcome", positive_label: str = "TP_FIRST"):
        train_df = train_df.dropna(subset=self.numeric_cols + [target_col])
        y = (train_df[target_col] == positive_label).astype(int)
        X = self._design_matrix(train_df, fit=True)
        Xs = self.scaler.fit_transform(X)
        self.model.fit(Xs, y)
        return self

    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:
        X = self._design_matrix(df, fit=False)
        Xs = self.scaler.transform(X)
        return self.model.predict_proba(Xs)[:, 1]

    def coefficients(self) -> pd.Series:
        return pd.Series(self.model.coef_[0], index=self._dummy_columns).sort_values(key=abs, ascending=False)


## Tests

`_make_labeled_dataset` builds a synthetic dataset with a **genuine**
relationship (`rs_score` and `vol_ratio` actually drive `P(TP_FIRST)`)
alongside pure-noise columns — so these tests check the model *recovers
real signal*, not just that it runs.

In [ ]:
def _make_labeled_dataset(n=1000, seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2022-01-01", periods=300)
    rows = []
    for i in range(n):
        d = dates[rng.integers(0, len(dates))]
        rs_score, vol_ratio = rng.normal(0, 1), rng.normal(1, 0.3)
        noise1, noise2 = rng.normal(0, 1), rng.normal(0, 1)
        logit = 0.9 * rs_score + 0.6 * (vol_ratio - 1) - 0.3
        p_tp = 1 / (1 + np.exp(-logit))
        outcome = rng.choice(["TP_FIRST", "SL_FIRST", "TIMEOUT"],
                              p=[p_tp*0.7, (1-p_tp)*0.6, 1 - p_tp*0.7 - (1-p_tp)*0.6])
        setup = rng.choice(["BREAKOUT", "PULLBACK"])
        rows.append({"date": d, "rs_score": rs_score, "vol_ratio": vol_ratio,
                      "noise1": noise1, "noise2": noise2, "primary_setup": setup, "outcome": outcome})
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def test_model_recovers_real_signal_out_of_sample():
    df = _make_labeled_dataset(1500, seed=10)
    cal = TradingCalendar(pd.bdate_range("2022-01-01", periods=300))
    test_start = df["date"].quantile(0.7, interpolation="nearest")
    train, test = time_split(df, cal, "date", test_start, embargo_sessions=10)
    assert len(train) > 200 and len(test) > 100

    model = ProbabilityModel(numeric_cols=["rs_score","vol_ratio","noise1","noise2"],
                              categorical_cols=["primary_setup"])
    model.fit(train)

    y_test = (test["outcome"] == "TP_FIRST").astype(int)
    p_test = model.predict_proba(test)
    auc = roc_auc_score(y_test, p_test)
    assert auc > 0.60, f"model should recover the genuine signal OUT OF SAMPLE, got AUC={auc:.3f}"
    print(f"✓ Out-of-sample signal-recovery test passed: test AUC={auc:.3f}")

    coefs = model.coefficients()
    top2 = coefs.index[:2].tolist()
    assert any("rs_score" in c or "vol_ratio" in c for c in top2), \
        f"real signal features should dominate noise, got top coefs: {coefs.head(4)}"
    print("✓ Interpretability check passed: real signal features rank above noise in |coefficient|")

test_model_recovers_real_signal_out_of_sample()


✓ Out-of-sample signal-recovery test passed: test AUC=0.648
✓ Interpretability check passed: real signal features rank above noise in |coefficient|


In [ ]:
def test_train_metrics_are_optimistic_vs_test():
    """Illustrates spec #69 directly: AUC on the TRAINING set is a biased
    estimate of real performance - this is exactly why the model must
    always be judged on held-out data, never on what it was fit to."""
    df = _make_labeled_dataset(1500, seed=30)
    cal = TradingCalendar(pd.bdate_range("2022-01-01", periods=300))
    test_start = df["date"].quantile(0.7, interpolation="nearest")
    train, test = time_split(df, cal, "date", test_start, embargo_sessions=10)

    model = ProbabilityModel(numeric_cols=["rs_score","vol_ratio","noise1","noise2"],
                              categorical_cols=["primary_setup"])
    model.fit(train)

    auc_train = roc_auc_score((train["outcome"]=="TP_FIRST").astype(int), model.predict_proba(train))
    auc_test = roc_auc_score((test["outcome"]=="TP_FIRST").astype(int), model.predict_proba(test))
    print(f"  train AUC={auc_train:.3f}  test AUC={auc_test:.3f}")
    print("✓ Demonstrated: train-set metrics read optimistic vs. real (test) performance - never report the former as if it were the latter")

test_train_metrics_are_optimistic_vs_test()
print("\nALL ProbabilityModel TESTS PASSED")


  train AUC=0.687  test AUC=0.674
✓ Demonstrated: train-set metrics read optimistic vs. real (test) performance - never report the former as if it were the latter

ALL ProbabilityModel TESTS PASSED


## `CalibrationAnalyzer`

Spec §41: a model predicting `P=70%` should historically win **~70% of the
time within that probability bucket** — a reliability curve, not just a
single accuracy number. Always computed on the held-out test set — the
entire point of calibration is checking whether the probabilities can be
trusted on data the model didn't fit to.

In [ ]:
class CalibrationAnalyzer:
    def __init__(self, n_bins: int = 10):
        self.n_bins = n_bins

    def reliability_curve(self, y_true, y_pred_proba) -> pd.DataFrame:
        df = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(y_pred_proba)})
        df["bucket"] = pd.qcut(df["p"], q=min(self.n_bins, df["p"].nunique()), duplicates="drop")
        out = df.groupby("bucket", observed=True).agg(
            n=("y", "size"), predicted_mean=("p", "mean"), observed_freq=("y", "mean")
        ).reset_index()
        out["calibration_error"] = (out["predicted_mean"] - out["observed_freq"]).abs()
        return out

    def brier_score(self, y_true, y_pred_proba) -> float:
        y_true, y_pred_proba = np.asarray(y_true), np.asarray(y_pred_proba)
        return float(np.mean((y_pred_proba - y_true) ** 2))


In [ ]:
def test_calibration_perfect_vs_random():
    rng = np.random.default_rng(20)
    n = 2000
    y_true = rng.integers(0, 2, n)
    p_perfect = np.full(n, y_true.mean())          # predicted == true underlying rate
    p_bad = np.full(n, 0.9)                          # always overconfident

    eng = CalibrationAnalyzer(n_bins=5)
    brier_perfect, brier_bad = eng.brier_score(y_true, p_perfect), eng.brier_score(y_true, p_bad)
    assert brier_perfect < brier_bad, f"perfect={brier_perfect:.3f} should beat bad={brier_bad:.3f}"
    print(f"✓ Brier score sanity test passed: perfect={brier_perfect:.3f} < miscalibrated={brier_bad:.3f}")

test_calibration_perfect_vs_random()


def test_reliability_curve_flags_miscalibration():
    """A systematically overconfident model (always predicts +0.25 over the
    true rate) must show predicted_mean > observed_freq in EVERY bucket -
    not hidden by averaging across buckets."""
    rng = np.random.default_rng(21)
    n = 3000
    true_p = rng.uniform(0.1, 0.6, n)
    y = rng.binomial(1, true_p)
    p_overconfident = np.clip(true_p + 0.25, 0, 1)

    eng = CalibrationAnalyzer(n_bins=5)
    curve = eng.reliability_curve(y, p_overconfident)
    assert (curve["predicted_mean"] > curve["observed_freq"]).all(), \
        "overconfident model should show predicted > observed in every bucket"
    assert curve["calibration_error"].mean() > 0.15
    print("✓ Reliability curve correctly flags systematic overconfidence in every bucket")
    print(curve.to_string())

test_reliability_curve_flags_miscalibration()
print("\nALL PHASE 9 TESTS PASSED")


✓ Brier score sanity test passed: perfect=0.250 < miscalibrated=0.418
✓ Reliability curve correctly flags systematic overconfidence in every bucket
           bucket    n  predicted_mean  observed_freq  calibration_error
0  (0.349, 0.452]  600        0.400004       0.166667           0.233337
1  (0.452, 0.552]  600        0.501926       0.233333           0.268592
2  (0.552, 0.653]  600        0.602364       0.338333           0.264031
3  (0.653, 0.753]  600        0.703809       0.473333           0.230475
4   (0.753, 0.85]  600        0.801754       0.518333           0.283421

ALL PHASE 9 TESTS PASSED


## Usage — wiring the full pipeline into a probability model

```python
# assemble the dataset from everything built so far
records = []
for sym, df in universe_data.items():
    for t in cal.sessions[::5]:
        sl = cal.slice_sessions(df, "time", t, n_back=len(df))
        feat = fe.feature_at(sl)
        if feat is None: continue
        setup = se.classify(sl, feat)
        outcome = tb.label(df, t, atr_pct_at_signal=feat["atr_pct"])
        if outcome is None: continue
        records.append({"symbol": sym, "date": t, **feat, **setup, **outcome})
dataset = pd.DataFrame(records)

test_start = dataset["date"].quantile(0.75, interpolation="nearest")
train, test = time_split(dataset, cal, "date", test_start, embargo_sessions=10)

model = ProbabilityModel(
    numeric_cols=["rsi14", "adx", "volatility_20d", "atr_pct", "volume_ratio"],
    categorical_cols=["primary_setup", "vsa_state"],
)
model.fit(train)

y_test = (test["outcome"] == "TP_FIRST").astype(int)
p_test = model.predict_proba(test)
print("Test AUC:", roc_auc_score(y_test, p_test))
print(model.coefficients())

cal_eng = CalibrationAnalyzer(n_bins=10)
print(cal_eng.reliability_curve(y_test, p_test))
print("Brier score:", cal_eng.brier_score(y_test, p_test))
```

If `test AUC` is close to `train AUC` from the illustrative test above and
the reliability curve tracks the diagonal reasonably well, the baseline is
worth building on. If test AUC collapses toward 0.5 or the reliability
curve is systematically off, that's Phase 5-7's factors/setups not carrying
the alpha they were designed to test for — which is exactly the kind of
finding this whole phased approach exists to surface honestly, rather than
hiding it behind a backtest run only once on the full history.

## Roadmap update

Phase 9 delivered: `time_split` (embargoed out-of-time split),
`ProbabilityModel` (interpretable logistic regression baseline), and
`CalibrationAnalyzer` (reliability curve + Brier score) — all evaluated
out-of-sample by construction, with tests that check *recovery of real
signal* and *honest calibration*, not just that the code runs.

Still ahead:
- **Phase 10** — Backtest v2: wire `FillSimulator` (Phase 2) into a full
  execution/portfolio/accounting engine with fees, slippage, and
  Vietnamese price-limit (trần/sàn) handling (spec §45-49).
- **Phase 11** — Walk-forward validation: generalize this phase's single
  `time_split` into the full rolling train/test windows from spec §53,
  with proper purging (spec §54) across every window, not just one.
- **Phase 12** — Ablation (spec §55-56: which factors/setups actually
  survive contact with out-of-sample AUC) + parameter sensitivity
  (spec §57-58) — the final validation step before any of this is trusted
  for live signals.

# PHASE 10 — Execution / Portfolio Backtest v2

This phase converts the signal/model layer into a realistic portfolio-level execution simulation.

**Information boundary:** the model may use information available at signal time T. The execution engine must never use `outcome`, `tb_ret`, `forward_return`, MFE/MAE, or any other future-derived label to determine an order, fill, exit, or P&L.

Architecture:

`signal @ T → order → T+1 execution → settlement constraint → future OHLCV → TP/SL/TIMEOUT → cash/equity`

The engine below is deliberately independent from the Phase 9 probability model so that the backtest cannot accidentally turn labels into P&L.

In [ ]:
# ============================================================
# PHASE 10.0 — Strict execution contract
# ============================================================

FUTURE_LABEL_COLUMNS = {
    "outcome", "tb_ret", "forward_return", "future_return",
    "future_return_3d", "future_return_5d", "future_return_10d",
    "future_return_20d", "mfe", "mae", "exit_idx", "sessions_held_label"
}

EXECUTION_REQUIRED = {"symbol", "signal_date", "entry_price", "sl_pct", "tp_pct"}

def validate_execution_signals(signals: pd.DataFrame, reject_future_labels=True):
    if not isinstance(signals, pd.DataFrame):
        raise TypeError("signals must be a pandas DataFrame")

    missing = EXECUTION_REQUIRED - set(signals.columns)
    if missing:
        raise ValueError(f"Missing execution columns: {sorted(missing)}")

    x = signals.copy()
    x["signal_date"] = pd.to_datetime(x["signal_date"], errors="coerce")

    for c in ["entry_price", "sl_pct", "tp_pct"]:
        x[c] = pd.to_numeric(x[c], errors="coerce")

    if x[list(EXECUTION_REQUIRED)].isna().any().any():
        raise ValueError("Execution signal contains NaN values in required fields")

    if (x["entry_price"] <= 0).any():
        raise ValueError("entry_price must be > 0")
    if (x["sl_pct"] <= 0).any() or (x["tp_pct"] <= 0).any():
        raise ValueError("sl_pct/tp_pct must be > 0")

    if reject_future_labels:
        leaked = sorted(FUTURE_LABEL_COLUMNS.intersection(x.columns))
        if leaked:
            raise ValueError(
                "Future-derived columns detected in execution signals: "
                f"{leaked}. Keep labels separate from execution inputs."
            )

    return x.sort_values(["signal_date", "symbol"]).reset_index(drop=True)


def make_execution_signals(dataset: pd.DataFrame) -> pd.DataFrame:
    """Build the execution-only view from a research dataset."""
    x = dataset.copy()

    if "signal_date" not in x.columns and "date" in x.columns:
        x["signal_date"] = pd.to_datetime(x["date"])

    if "sl_pct" not in x.columns:
        # Conservative fallback: use 1.5 ATR if ATR% exists.
        if "atr_pct" in x.columns:
            x["sl_pct"] = (1.5 * pd.to_numeric(x["atr_pct"], errors="coerce")).clip(0.01, 0.15)
        else:
            raise ValueError("No sl_pct supplied and atr_pct is unavailable")

    if "tp_pct" not in x.columns:
        x["tp_pct"] = x["sl_pct"] * 2.0

    keep = [
        c for c in [
            "symbol", "signal_date", "entry_price", "sl_pct", "tp_pct",
            "primary_setup", "sector", "market_state", "state",
            "market_regime", "avg_daily_value", "avg_val_20d",
            "prob_tp", "p_tp", "score"
        ] if c in x.columns
    ]

    return validate_execution_signals(x[keep], reject_future_labels=True)


print("✓ Phase 10 execution contract loaded.")

✓ Phase 10 execution contract loaded.


In [ ]:
# ============================================================
# PHASE 10.1 — Portfolio execution engine
# ============================================================

@dataclass
class BacktestConfig:
    initial_capital: float = 100_000_000.0

    # Risk / exposure
    risk_per_trade: float = 0.0075
    max_position_pct: float = 0.15
    max_total_exposure: float = 0.75
    max_open_positions: int = 5
    lot_size: int = 100

    # Execution / holding
    settlement_sessions: int = 2
    max_hold_sessions: int = 20

    # VN market costs — configurable research assumptions
    buy_fee: float = 0.0015
    sell_fee: float = 0.0015
    sell_tax: float = 0.001
    slippage_bps: float = 5.0

    # Liquidity impact
    liquidity_slippage_bps: float = 50.0

    # Conservative same-bar convention
    same_bar_policy: str = "SL_FIRST"

    # Optional price-limit approximation
    use_price_limit: bool = True
    limit_pct: float = 0.07


class PortfolioBacktestV2:
    """
    Event-driven long-only portfolio backtest.

    Signal is observed at T.
    Entry is attempted on a future bar.
    P&L is determined only from OHLCV after the actual fill.
    """

    def __init__(self, config=None):
        self.cfg = config or BacktestConfig()

    def _prepare(self, df):
        x = df.copy()
        x["time"] = pd.to_datetime(x["time"]).dt.normalize()
        x = x.sort_values("time").drop_duplicates("time").reset_index(drop=True)

        required = {"time", "open", "high", "low", "close", "volume"}
        missing = required - set(x.columns)
        if missing:
            raise ValueError(f"OHLCV missing: {sorted(missing)}")

        for c in ["open", "high", "low", "close", "volume"]:
            x[c] = pd.to_numeric(x[c], errors="coerce")

        x["prev_close"] = x["close"].shift(1)
        x["traded_value"] = x["close"] * x["volume"]
        return x

    def _round_lot(self, shares):
        return int(max(0, shares) // self.cfg.lot_size) * self.cfg.lot_size

    def _slippage_rate(self, notional, adv):
        base = self.cfg.slippage_bps / 10000.0
        if pd.notna(adv) and adv > 0:
            participation = min(1.0, max(0.0, notional / adv))
            base += (
                self.cfg.liquidity_slippage_bps / 10000.0
                * participation
            )
        return base

    def _buy_price(self, price, notional, adv):
        return price * (1.0 + self._slippage_rate(notional, adv))

    def _sell_price(self, price, notional, adv):
        return price * (1.0 - self._slippage_rate(notional, adv))

    def _buy_fee(self, gross):
        return gross * self.cfg.buy_fee

    def _sell_fee(self, gross):
        return gross * (self.cfg.sell_fee + self.cfg.sell_tax)

    def _limit_prices(self, prev_close):
        if not self.cfg.use_price_limit or pd.isna(prev_close):
            return None, None
        return (
            float(prev_close) * (1 - self.cfg.limit_pct),
            float(prev_close) * (1 + self.cfg.limit_pct)
        )

    def _is_pinned_up(self, row):
        lo, hi = self._limit_prices(row.get("prev_close", np.nan))
        if hi is None:
            return False
        return (
            row["open"] >= hi * 0.999999 and
            row["high"] >= hi * 0.999999 and
            row["low"] >= hi * 0.999999 and
            row["close"] >= hi * 0.999999
        )

    def _is_pinned_down(self, row):
        lo, hi = self._limit_prices(row.get("prev_close", np.nan))
        if lo is None:
            return False
        return (
            row["open"] <= lo * 1.000001 and
            row["high"] <= lo * 1.000001 and
            row["low"] <= lo * 1.000001 and
            row["close"] <= lo * 1.000001
        )

    def _size_position(self, available_cash, entry, sl_pct, current_exposure):
        risk_budget = available_cash * self.cfg.risk_per_trade
        stop_distance = entry * sl_pct

        if stop_distance <= 0:
            return 0

        by_risk = int(risk_budget / stop_distance)
        by_name = int((available_cash * self.cfg.max_position_pct) / entry)

        remaining_exposure = max(
            0.0,
            available_cash * self.cfg.max_total_exposure - current_exposure
        )
        by_portfolio = int(remaining_exposure / entry)

        return self._round_lot(
            min(by_risk, by_name, by_portfolio)
        )

    def _find_entry_bar(self, bars, signal_date):
        future = bars[bars["time"] > pd.Timestamp(signal_date)]
        return None if future.empty else future.iloc[0]

    def _open(self, signal, bars, cash, exposure):
        entry_bar = self._find_entry_bar(bars, signal["signal_date"])
        if entry_bar is None:
            return None

        # Avoid assuming a buy can be filled when the stock is locked at ceiling.
        if self._is_pinned_up(entry_bar):
            return None

        raw = float(entry_bar["open"])
        adv = signal.get("avg_daily_value", signal.get("avg_val_20d", np.nan))
        qty = self._size_position(
            cash, raw, float(signal["sl_pct"]), exposure
        )

        if qty <= 0:
            return None

        fill = self._buy_price(raw, raw * qty, adv)
        gross = fill * qty
        fee = self._buy_fee(gross)
        cash_used = gross + fee

        if cash_used > cash:
            qty = self._round_lot(
                cash / (fill * (1 + self.cfg.buy_fee))
            )
            if qty <= 0:
                return None
            gross = fill * qty
            fee = self._buy_fee(gross)
            cash_used = gross + fee

        return {
            "symbol": str(signal["symbol"]),
            "signal_date": pd.Timestamp(signal["signal_date"]),
            "entry_date": pd.Timestamp(entry_bar["time"]),
            "entry_price": float(fill),
            "shares": int(qty),
            "gross_entry": float(gross),
            "entry_fee": float(fee),
            "cash_used": float(cash_used),
            "sl_pct": float(signal["sl_pct"]),
            "tp_pct": float(signal["tp_pct"]),
            "sl_price": float(fill * (1 - signal["sl_pct"])),
            "tp_price": float(fill * (1 + signal["tp_pct"])),
            "setup": signal.get("primary_setup", ""),
            "market_state": signal.get("market_state", signal.get("state", "")),
            "avg_daily_value": adv
        }

    def run(self, signals, universe_data, benchmark=None):
        signals = validate_execution_signals(signals)

        bars = {
            str(sym): self._prepare(df)
            for sym, df in universe_data.items()
        }

        all_dates = set(pd.to_datetime(signals["signal_date"]))
        for df in bars.values():
            all_dates.update(df["time"].tolist())

        dates = sorted(all_dates)

        cash = float(self.cfg.initial_capital)
        positions = []
        trades = []
        equity_rows = []

        for date in dates:
            date = pd.Timestamp(date)

            # -----------------------------
            # New entries
            # -----------------------------
            todays = signals[signals["signal_date"] == date]
            occupied = {p["symbol"] for p in positions}
            exposure = sum(p["gross_entry"] for p in positions)

            for _, signal in todays.iterrows():
                symbol = str(signal["symbol"])

                if symbol not in bars:
                    continue
                if symbol in occupied:
                    continue
                if len(positions) >= self.cfg.max_open_positions:
                    break

                pos = self._open(signal, bars[symbol], cash, exposure)
                if pos is None:
                    continue

                cash -= pos["cash_used"]
                exposure += pos["gross_entry"]
                positions.append(pos)
                occupied.add(symbol)

            # -----------------------------
            # Manage existing positions
            # -----------------------------
            survivors = []

            for pos in positions:
                d = bars[pos["symbol"]]
                today = d[d["time"] == date]

                if today.empty or date < pos["entry_date"]:
                    survivors.append(pos)
                    continue

                row = today.iloc[0]

                held_sessions = (
                    int((d["time"] >= pos["entry_date"]) &
                        (d["time"] <= date)).sum()
                )

                # The investor cannot sell before settlement.
                can_exit = held_sessions > self.cfg.settlement_sessions

                hit_sl = float(row["low"]) <= pos["sl_price"]
                hit_tp = float(row["high"]) >= pos["tp_price"]

                reason = None
                raw_exit = None

                if can_exit and hit_sl and hit_tp:
                    if self.cfg.same_bar_policy == "TP_FIRST":
                        reason, raw_exit = "TP_FIRST", pos["tp_price"]
                    else:
                        reason, raw_exit = "SL_FIRST", pos["sl_price"]

                elif can_exit and hit_sl:
                    reason, raw_exit = "SL_FIRST", pos["sl_price"]

                elif can_exit and hit_tp:
                    reason, raw_exit = "TP_FIRST", pos["tp_price"]

                elif held_sessions >= self.cfg.max_hold_sessions:
                    reason, raw_exit = "TIMEOUT", float(row["close"])

                if reason is None:
                    survivors.append(pos)
                    continue

                # Cannot assume liquidity if price is locked at floor.
                if self._is_pinned_down(row):
                    survivors.append(pos)
                    continue

                adv = pos.get("avg_daily_value", np.nan)
                exit_price = self._sell_price(
                    float(raw_exit), pos["shares"], adv
                )

                gross_exit = exit_price * pos["shares"]
                exit_fee = self._sell_fee(gross_exit)

                pnl_gross = gross_exit - pos["gross_entry"]
                pnl_net = pnl_gross - pos["entry_fee"] - exit_fee

                cash += pos["cash_used"] + pnl_net

                trades.append({
                    **pos,
                    "exit_date": date,
                    "exit_price": exit_price,
                    "gross_exit": gross_exit,
                    "exit_fee": exit_fee,
                    "pnl_gross": pnl_gross,
                    "pnl_net": pnl_net,
                    "return_net": pnl_net / pos["gross_entry"],
                    "exit_reason": reason,
                    "sessions_held": held_sessions
                })

            positions = survivors

            # -----------------------------
            # Mark-to-market
            # -----------------------------
            market_value = 0.0
            for pos in positions:
                q = bars[pos["symbol"]]
                q = q[q["time"] <= date]
                if not q.empty:
                    market_value += float(q.iloc[-1]["close"]) * pos["shares"]

            equity = cash + market_value

            equity_rows.append({
                "date": date,
                "cash": cash,
                "market_value": market_value,
                "equity": equity,
                "n_open": len(positions),
                "exposure": market_value / equity if equity > 0 else np.nan
            })

        # -----------------------------
        # Force close remaining positions
        # -----------------------------
        for pos in positions:
            d = bars[pos["symbol"]]
            row = d.iloc[-1]

            adv = pos.get("avg_daily_value", np.nan)
            exit_price = self._sell_price(
                float(row["close"]), pos["shares"], adv
            )

            gross_exit = exit_price * pos["shares"]
            exit_fee = self._sell_fee(gross_exit)

            pnl_gross = gross_exit - pos["gross_entry"]
            pnl_net = pnl_gross - pos["entry_fee"] - exit_fee

            trades.append({
                **pos,
                "exit_date": pd.Timestamp(row["time"]),
                "exit_price": exit_price,
                "gross_exit": gross_exit,
                "exit_fee": exit_fee,
                "pnl_gross": pnl_gross,
                "pnl_net": pnl_net,
                "return_net": pnl_net / pos["gross_entry"],
                "exit_reason": "FORCED_END",
                "sessions_held": int(
                    (d["time"] >= pos["entry_date"]).sum()
                )
            })

        trades_df = pd.DataFrame(trades)
        equity_df = pd.DataFrame(equity_rows)
        metrics = portfolio_metrics(trades_df, equity_df, benchmark)

        return {
            "trades": trades_df,
            "equity": equity_df,
            "metrics": metrics
        }


def portfolio_metrics(trades, equity, benchmark=None):
    metrics = {}

    if equity is not None and not equity.empty:
        e = equity.sort_values("date").drop_duplicates("date")
        initial = float(e["equity"].iloc[0])
        final = float(e["equity"].iloc[-1])

        days = max(
            1,
            (e["date"].iloc[-1] - e["date"].iloc[0]).days
        )
        years = days / 365.25

        metrics["total_return"] = final / initial - 1 if initial else np.nan
        metrics["CAGR"] = (
            (final / initial) ** (1 / years) - 1
            if initial > 0 and final > 0 and years > 0
            else np.nan
        )

        dd = e["equity"] / e["equity"].cummax() - 1
        metrics["max_drawdown"] = float(dd.min())

        daily = (
            e.set_index("date")["equity"]
             .resample("1D").last()
             .ffill()
             .pct_change()
             .dropna()
        )

        metrics["Sharpe"] = (
            float(np.sqrt(252) * daily.mean() / daily.std())
            if len(daily) > 1 and daily.std() > 0
            else np.nan
        )

        downside = daily[daily < 0]
        metrics["Sortino"] = (
            float(np.sqrt(252) * daily.mean() / downside.std())
            if len(downside) > 1 and downside.std() > 0
            else np.nan
        )

        metrics["Calmar"] = (
            metrics["CAGR"] / abs(metrics["max_drawdown"])
            if metrics["max_drawdown"] < 0
            else np.nan
        )

    if trades is None or trades.empty:
        metrics.update({
            "n_trades": 0,
            "win_rate": np.nan,
            "profit_factor": np.nan,
            "expectancy": np.nan,
            "avg_win": np.nan,
            "avg_loss": np.nan,
            "payoff_ratio": np.nan,
            "total_cost": 0.0
        })
        return metrics

    r = pd.to_numeric(trades["return_net"], errors="coerce").dropna()
    wins = r[r > 0]
    losses = r[r < 0]

    gross_profit = float(
        trades.loc[trades["pnl_net"] > 0, "pnl_net"].sum()
    )
    gross_loss = float(
        -trades.loc[trades["pnl_net"] < 0, "pnl_net"].sum()
    )

    metrics.update({
        "n_trades": int(len(r)),
        "win_rate": float((r > 0).mean()),
        "avg_win": float(wins.mean()) if len(wins) else np.nan,
        "avg_loss": float(losses.mean()) if len(losses) else np.nan,
        "payoff_ratio": (
            abs(float(wins.mean() / losses.mean()))
            if len(wins) and len(losses) and losses.mean() != 0
            else np.nan
        ),
        "profit_factor": gross_profit / gross_loss if gross_loss > 0 else np.inf,
        "expectancy": float(r.mean()),
        "total_cost": float(
            trades["entry_fee"].sum() + trades["exit_fee"].sum()
        )
    })

    if benchmark is not None and not equity.empty:
        b = benchmark.copy()
        b["time"] = pd.to_datetime(b["time"]).dt.normalize()
        b = b.sort_values("time")
        b = b[
            (b["time"] >= equity["date"].min()) &
            (b["time"] <= equity["date"].max())
        ]

        if len(b) >= 2:
            benchmark_return = (
                b["close"].iloc[-1] / b["close"].iloc[0] - 1
            )
            metrics["benchmark_return"] = float(benchmark_return)
            metrics["excess_return"] = (
                metrics.get("total_return", np.nan) - benchmark_return
            )

    return metrics


print("✓ Phase 10 portfolio engine loaded.")

✓ Phase 10 portfolio engine loaded.


## Phase 10.2 — Why this is separate from `simulate_trade_v2`

`simulate_trade_v2` is useful for single-trade validation and was intentionally introduced in Phase 2.

Phase 10 adds the missing **portfolio layer**:

- multiple symbols,
- limited capital,
- position sizing,
- concurrent positions,
- fees/tax,
- slippage,
- exposure constraints,
- mark-to-market equity,
- drawdown,
- portfolio-level risk.

A signal can therefore have positive single-trade expectancy but still be unusable once turnover, correlation, capital constraints, and drawdown are included.

# PHASE 11 — Purged Rolling Walk-Forward

The Phase 9 `time_split()` is retained as the simple chronological baseline.

Phase 11 adds rolling folds:

`TRAIN → PURGE/EMBARGO → VALIDATION → PURGE/EMBARGO → TEST → ROLL`

The final test period is never used for parameter selection.

In [ ]:
# ============================================================
# PHASE 11.0 — Rolling calendar walk-forward
# ============================================================

@dataclass
class WalkForwardConfig:
    train_months: int = 24
    validation_months: int = 3
    test_months: int = 6
    step_months: int = 6
    embargo_sessions: int = 10

    min_train_rows: int = 200
    min_validation_rows: int = 30
    min_test_rows: int = 30


class PurgedWalkForward:
    def __init__(self, config=None):
        self.cfg = config or WalkForwardConfig()

    @staticmethod
    def _shift_months(ts, months):
        return pd.Timestamp(ts) + pd.DateOffset(months=months)

    def make_folds(self, df, date_col="date"):
        x = df.copy()
        x[date_col] = pd.to_datetime(x[date_col]).dt.normalize()
        x = x.sort_values(date_col)

        if x.empty:
            return pd.DataFrame()

        start = x[date_col].min()
        end = x[date_col].max()
        cursor = start
        rows = []

        while True:
            train_end = self._shift_months(
                cursor, self.cfg.train_months
            )
            val_end = self._shift_months(
                train_end, self.cfg.validation_months
            )
            test_end = self._shift_months(
                val_end, self.cfg.test_months
            )

            if test_end > end:
                break

            train = x[
                (x[date_col] >= cursor) &
                (x[date_col] < train_end)
            ]
            validation = x[
                (x[date_col] >= train_end) &
                (x[date_col] < val_end)
            ]
            test = x[
                (x[date_col] >= val_end) &
                (x[date_col] < test_end)
            ]

            if (
                len(train) >= self.cfg.min_train_rows and
                len(validation) >= self.cfg.min_validation_rows and
                len(test) >= self.cfg.min_test_rows
            ):
                rows.append({
                    "fold": len(rows) + 1,
                    "train_start": train[date_col].min(),
                    "train_end": train[date_col].max(),
                    "validation_start": validation[date_col].min(),
                    "validation_end": validation[date_col].max(),
                    "test_start": test[date_col].min(),
                    "test_end": test[date_col].max(),
                    "n_train_raw": len(train),
                    "n_validation_raw": len(validation),
                    "n_test": len(test)
                })

            cursor = self._shift_months(
                cursor, self.cfg.step_months
            )

        return pd.DataFrame(rows)

    def split(self, df, fold, date_col="date", calendar=None):
        x = df.copy()
        x[date_col] = pd.to_datetime(x[date_col]).dt.normalize()

        train = x[
            (x[date_col] >= fold["train_start"]) &
            (x[date_col] <= fold["train_end"])
        ].copy()

        validation = x[
            (x[date_col] >= fold["validation_start"]) &
            (x[date_col] <= fold["validation_end"])
        ].copy()

        test = x[
            (x[date_col] >= fold["test_start"]) &
            (x[date_col] <= fold["test_end"])
        ].copy()

        # Purge/embargo the end of each preceding window.
        if calendar is not None:
            train_cut = calendar.add_sessions(
                fold["validation_start"],
                -(self.cfg.embargo_sessions + 1)
            )
            val_cut = calendar.add_sessions(
                fold["test_start"],
                -(self.cfg.embargo_sessions + 1)
            )

            if train_cut is not None:
                train = train[train[date_col] <= train_cut]

            if val_cut is not None:
                validation = validation[
                    validation[date_col] <= val_cut
                ]
        else:
            # Fallback only when a real trading calendar is unavailable.
            if len(train) > self.cfg.embargo_sessions:
                train = train.iloc[:-self.cfg.embargo_sessions]
            if len(validation) > self.cfg.embargo_sessions:
                validation = validation.iloc[:-self.cfg.embargo_sessions]

        return (
            train.reset_index(drop=True),
            validation.reset_index(drop=True),
            test.reset_index(drop=True)
        )

    def audit(self, df, date_col="date", calendar=None):
        folds = self.make_folds(df, date_col)
        rows = []

        for _, fold in folds.iterrows():
            train, validation, test = self.split(
                df, fold, date_col, calendar
            )

            rows.append({
                "fold": int(fold["fold"]),
                "train_rows": len(train),
                "validation_rows": len(validation),
                "test_rows": len(test),
                "train_max": train[date_col].max() if len(train) else pd.NaT,
                "validation_min": validation[date_col].min() if len(validation) else pd.NaT,
                "validation_max": validation[date_col].max() if len(validation) else pd.NaT,
                "test_min": test[date_col].min() if len(test) else pd.NaT,
                "test_max": test[date_col].max() if len(test) else pd.NaT,
            })

        return pd.DataFrame(rows)

    @staticmethod
    def assert_no_overlap(audit_df):
        if audit_df.empty:
            raise AssertionError("No walk-forward folds were generated")

        for _, r in audit_df.iterrows():
            assert r["train_max"] < r["validation_min"]
            assert r["validation_max"] < r["test_min"]

        return True


print("✓ Phase 11 walk-forward engine loaded.")

✓ Phase 11 walk-forward engine loaded.


In [ ]:
# ============================================================
# PHASE 11.1 — Walk-forward model runner
# ============================================================

def run_probability_walk_forward(
    dataset: pd.DataFrame,
    numeric_cols,
    categorical_cols=None,
    target_col="outcome",
    positive_label="TP_FIRST",
    calendar=None,
    wf_config=None,
    probability_threshold=0.50
):
    """
    Fits a fresh ProbabilityModel on each TRAIN window.

    Validation is used to choose a probability threshold.
    TEST is evaluated only after the threshold is frozen.
    """
    categorical_cols = categorical_cols or []
    wf = PurgedWalkForward(wf_config)
    folds = wf.make_folds(dataset, "date")

    fold_rows = []
    oos_predictions = []

    for _, fold in folds.iterrows():
        train, validation, test = wf.split(
            dataset, fold, "date", calendar
        )

        model = ProbabilityModel(
            numeric_cols=[c for c in numeric_cols if c in dataset.columns],
            categorical_cols=[
                c for c in categorical_cols if c in dataset.columns
            ]
        )

        model.fit(
            train,
            target_col=target_col,
            positive_label=positive_label
        )

        p_val = model.predict_proba(validation)
        y_val = (
            validation[target_col].astype(str) == positive_label
        ).astype(int)

        # Threshold selection belongs to validation, not test.
        candidate_thresholds = np.arange(0.40, 0.81, 0.05)
        best_threshold = float(probability_threshold)
        best_val_score = -np.inf

        for threshold in candidate_thresholds:
            pred = (p_val >= threshold).astype(int)
            if pred.sum() == 0:
                continue

            # Simple precision proxy for model selection.
            precision = float(y_val[pred == 1].mean())
            if precision > best_val_score:
                best_val_score = precision
                best_threshold = float(threshold)

        p_test = model.predict_proba(test)
        y_test = (
            test[target_col].astype(str) == positive_label
        ).astype(int)

        test_auc = (
            float(roc_auc_score(y_test, p_test))
            if y_test.nunique() == 2 else np.nan
        )

        selected = p_test >= best_threshold
        selected_rate = (
            float(y_test[selected].mean())
            if selected.sum() else np.nan
        )

        fold_rows.append({
            "fold": int(fold["fold"]),
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "validation_start": fold["validation_start"],
            "validation_end": fold["validation_end"],
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            "n_train": len(train),
            "n_validation": len(validation),
            "n_test": len(test),
            "selected_threshold": best_threshold,
            "validation_precision": best_val_score,
            "test_auc": test_auc,
            "test_selected_hit_rate": selected_rate,
            "test_selected_n": int(selected.sum())
        })

        pred_frame = test[["date"]].copy()
        if "symbol" in test.columns:
            pred_frame["symbol"] = test["symbol"].values

        pred_frame["fold"] = int(fold["fold"])
        pred_frame["y_true"] = y_test.to_numpy()
        pred_frame["p_tp"] = p_test
        pred_frame["selected"] = selected

        oos_predictions.append(pred_frame)

    fold_df = pd.DataFrame(fold_rows)
    pred_df = (
        pd.concat(oos_predictions, ignore_index=True)
        if oos_predictions else pd.DataFrame()
    )

    return {
        "folds": fold_df,
        "oos_predictions": pred_df,
        "stability": walk_forward_stability(fold_df)
    }


def walk_forward_stability(fold_df):
    if fold_df is None or fold_df.empty:
        return pd.DataFrame()

    metrics = [
        c for c in [
            "test_auc",
            "test_selected_hit_rate",
            "validation_precision"
        ] if c in fold_df.columns
    ]

    rows = []

    for c in metrics:
        s = pd.to_numeric(
            fold_df[c], errors="coerce"
        ).dropna()

        if len(s) == 0:
            continue

        rows.append({
            "metric": c,
            "mean": float(s.mean()),
            "median": float(s.median()),
            "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "min": float(s.min()),
            "max": float(s.max()),
            "positive_fold_pct": float((s > 0).mean()),
            "n_folds": int(len(s))
        })

    return pd.DataFrame(rows)


print("✓ Phase 11 model runner loaded.")

✓ Phase 11 model runner loaded.


# PHASE 12 — Ablation + Parameter Sensitivity + Stability

The purpose of Phase 12 is not to find the single best backtest number.

It is to answer:

1. Which factor groups add incremental OOS information?
2. Are the thresholds stable across a region?
3. Does the model survive when one factor family is removed?
4. Does the signal remain useful after execution costs?
5. Does the result persist across different market periods?

In [ ]:
# ============================================================
# PHASE 12.0 — Ablation study
# ============================================================

def evaluate_probability_predictions(y_true, p):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)

    out = {
        "n": int(len(y)),
        "base_rate": float(y.mean()) if len(y) else np.nan,
        "brier": float(np.mean((p - y) ** 2)) if len(y) else np.nan
    }

    if len(np.unique(y)) == 2:
        out["auc"] = float(roc_auc_score(y, p))
    else:
        out["auc"] = np.nan

    if len(y) >= 20:
        cutoff = np.nanquantile(p, 0.90)
        top = y[p >= cutoff]

        out["top_decile_n"] = int(len(top))
        out["top_decile_rate"] = (
            float(top.mean()) if len(top) else np.nan
        )
        out["top_decile_lift"] = (
            out["top_decile_rate"] / out["base_rate"]
            if out["base_rate"] > 0 else np.nan
        )
    else:
        out["top_decile_n"] = 0
        out["top_decile_rate"] = np.nan
        out["top_decile_lift"] = np.nan

    return out


def run_ablation(
    dataset,
    train,
    test,
    feature_groups,
    base_features,
    categorical_cols=None,
    target_col="outcome",
    positive_label="TP_FIRST"
):
    categorical_cols = categorical_cols or []
    rows = []

    for name, extra_features in feature_groups.items():
        numeric = [
            c for c in list(base_features) + list(extra_features)
            if c in dataset.columns
        ]

        if not numeric:
            continue

        model = ProbabilityModel(
            numeric_cols=numeric,
            categorical_cols=[
                c for c in categorical_cols
                if c in dataset.columns
            ]
        )

        model.fit(
            train,
            target_col=target_col,
            positive_label=positive_label
        )

        p = model.predict_proba(test)
        y = (
            test[target_col].astype(str) == positive_label
        ).astype(int)

        result = evaluate_probability_predictions(y, p)

        rows.append({
            "ablation": name,
            "n_numeric_features": len(numeric),
            **result
        })

    return pd.DataFrame(rows).sort_values(
        ["auc", "brier"],
        ascending=[False, True]
    ).reset_index(drop=True)


def ablation_delta(ablation_df, baseline_name="BASE"):
    if ablation_df.empty:
        return ablation_df

    x = ablation_df.copy()
    base = x[x["ablation"] == baseline_name]

    if base.empty:
        x["auc_delta_vs_base"] = np.nan
        x["brier_delta_vs_base"] = np.nan
        return x

    base_auc = float(base["auc"].iloc[0])
    base_brier = float(base["brier"].iloc[0])

    x["auc_delta_vs_base"] = x["auc"] - base_auc
    x["brier_delta_vs_base"] = x["brier"] - base_brier

    return x


print("✓ Phase 12 ablation engine loaded.")

✓ Phase 12 ablation engine loaded.


In [ ]:
# ============================================================
# PHASE 12.1 — Parameter sensitivity
# ============================================================

def parameter_grid(grid):
    """Yield dictionaries without requiring sklearn/grid-search machinery."""
    import itertools

    keys = list(grid.keys())

    for values in itertools.product(
        *(grid[key] for key in keys)
    ):
        yield dict(zip(keys, values))


def run_parameter_sensitivity(evaluator, grid):
    """
    evaluator(params) -> float or dict.

    The evaluator should itself perform the appropriate train/validation/test
    protocol. This function is only the parameter combinator.
    """
    rows = []

    for params in parameter_grid(grid):
        try:
            result = evaluator(params)

            if isinstance(result, dict):
                rows.append({**params, **result})
            else:
                rows.append({
                    **params,
                    "score": float(result)
                })

        except Exception as exc:
            rows.append({
                **params,
                "score": np.nan,
                "error": repr(exc)
            })

    return pd.DataFrame(rows)


def sensitivity_stability(
    result_df,
    score_col="score",
    tolerance=0.05
):
    """
    Finds the stable region within `tolerance` of the best score.

    This is more useful than reporting only argmax(parameter).
    """
    if result_df is None or result_df.empty:
        return pd.DataFrame()

    x = result_df.copy()
    x[score_col] = pd.to_numeric(
        x[score_col], errors="coerce"
    )

    best = x[score_col].max()

    x["within_best_region"] = (
        x[score_col] >= best * (1 - tolerance)
    )

    return x.sort_values(
        score_col,
        ascending=False
    ).reset_index(drop=True)


def summarize_parameter_region(result_df, parameter_cols, score_col="score"):
    if result_df is None or result_df.empty:
        return pd.DataFrame()

    x = result_df.copy()
    x[score_col] = pd.to_numeric(
        x[score_col], errors="coerce"
    )

    rows = []

    for col in parameter_cols:
        if col not in x.columns:
            continue

        g = x.groupby(col, dropna=False)[score_col].agg(
            ["mean", "median", "std", "min", "max", "count"]
        ).reset_index()

        g.insert(0, "parameter", col)
        g = g.rename(columns={col: "value"})
        rows.append(g)

    return (
        pd.concat(rows, ignore_index=True)
        if rows else pd.DataFrame()
    )


print("✓ Phase 12 parameter sensitivity engine loaded.")

✓ Phase 12 parameter sensitivity engine loaded.


In [ ]:
# ============================================================
# PHASE 12.2 — Fold stability / research gates
# ============================================================

def fold_stability_table(fold_results, metric_cols=None):
    if fold_results is None or fold_results.empty:
        return pd.DataFrame()

    metric_cols = metric_cols or [
        c for c in [
            "test_auc",
            "test_selected_hit_rate",
            "validation_precision",
            "test_return",
            "test_profit_factor",
            "test_expectancy"
        ] if c in fold_results.columns
    ]

    rows = []

    for col in metric_cols:
        s = pd.to_numeric(
            fold_results[col],
            errors="coerce"
        ).dropna()

        if s.empty:
            continue

        rows.append({
            "metric": col,
            "mean": float(s.mean()),
            "median": float(s.median()),
            "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "min": float(s.min()),
            "max": float(s.max()),
            "n_folds": int(len(s)),
            "positive_fold_pct": float((s > 0).mean())
        })

    return pd.DataFrame(rows)


def research_gate(
    portfolio_metrics_dict=None,
    probability_metrics_dict=None,
    min_auc=0.55,
    max_drawdown=-0.35
):
    """
    Research diagnostic, NOT an investment recommendation.
    """
    p = probability_metrics_dict or {}
    b = portfolio_metrics_dict or {}

    checks = {}

    if "auc" in p:
        checks["auc_above_threshold"] = bool(
            pd.notna(p["auc"]) and p["auc"] >= min_auc
        )

    if "test_auc" in p:
        checks["oos_auc_above_threshold"] = bool(
            pd.notna(p["test_auc"]) and p["test_auc"] >= min_auc
        )

    if "expectancy" in b:
        checks["positive_expectancy"] = bool(
            pd.notna(b["expectancy"]) and b["expectancy"] > 0
        )

    if "profit_factor" in b:
        checks["profit_factor_gt_1"] = bool(
            pd.notna(b["profit_factor"]) and b["profit_factor"] > 1
        )

    if "max_drawdown" in b:
        checks["drawdown_within_limit"] = bool(
            pd.notna(b["max_drawdown"]) and
            b["max_drawdown"] > max_drawdown
        )

    checks["PASS"] = (
        all(checks.values())
        if checks else False
    )

    return checks


print("✓ Phase 12 research gates loaded.")

✓ Phase 12 research gates loaded.


## Phase 12.3 — Recommended ablation definition

When the real research dataset is available, use groups aligned with the notebook's existing architecture rather than arbitrary columns.

Example:

```python
feature_groups = {
    "BASE": [],
    "FLOW": [
        "volume_ratio", "vsa_score", "vsa_state"
    ],
    "MOMENTUM": [
        "rsi14", "macd", "adx", "aroon_up", "aroon_down"
    ],
    "STRUCTURE": [
        "support", "resistance", "distance_to_resistance",
        "distance_to_support"
    ],
    "SETUP": [
        "primary_setup"
    ],
    "STATE": [
        "market_state"
    ],
    "EXIT_RISK": [
        "exit_risk"
    ],
}
```

Do **not** interpret this example as proof that every named column exists in every generated dataset. Intersect the groups with the actual dataframe columns before fitting.

The key question is:

> Does adding this factor family improve held-out performance consistently across folds?

If the answer is only "yes" on one historical period, it should not be treated as robust alpha.

# PHASE 10–12 — Integration / Smoke Tests

These tests are intentionally synthetic. They verify architecture and information boundaries without requiring a VNStock API call.

The tests do **not** prove profitability.

# FINAL PIPELINE — How to use the completed notebook

The intended research flow is:

`Historical OHLCV`
→ `causal features`
→ `setups / market state / exit risk`
→ `triple-barrier labels`
→ `train-only ProbabilityModel`
→ `validation threshold`
→ `Phase 10 execution backtest`
→ `Phase 11 rolling OOS`
→ `Phase 12 ablation`
→ `parameter stability`
→ `portfolio robustness`

## Important separation

Keep these objects conceptually separate:

- **Research dataset:** may contain labels.
- **Model training dataset:** labels are used only as the target.
- **Execution signals:** must not contain future labels.
- **OHLCV execution data:** determines actual future fills/exits.
- **OOS test:** is never used to tune the final strategy.

The final daily scanner should therefore consume something like:

```python
execution_signals = pd.DataFrame([{
    "symbol": "FPT",
    "signal_date": pd.Timestamp("2026-08-10"),
    "entry_price": 123.0,
    "sl_pct": 0.05,
    "tp_pct": 0.10,
    "primary_setup": "BREAKOUT",
    "market_state": "MARKUP",
    "p_tp": 0.67,
}])
```

and then pass that signal into the execution/risk layer.

**Do not use the triple-barrier `outcome` to simulate whether the trade won.**
That outcome exists for research/evaluation; the Phase 10 engine determines the result from future OHLCV.

# PHASE 13 — DAILY LIVE / EOD STOCK SCANNER

Mục tiêu của Phase 13:

> Mỗi ngày mở Google Colab → Run All → scanner tự quét toàn bộ watchlist → trả lời:
>
> 1. Hôm nay mã nào đáng chú ý để nghiên cứu mua?
> 2. Mã nào đang hình thành setup tốt nhưng chưa đủ điều kiện?
> 3. Mã nào đang có dấu hiệu kết thúc uptrend / phân phối / tăng exit-risk?
> 4. Mã nào bị loại vì thanh khoản hoặc chất lượng dữ liệu?

## Important

This is a **research / decision-support scanner**, not an automatic trading system.

The scanner deliberately separates:
- market regime,
- stock features,
- setup,
- probability,
- exit risk,
- liquidity,
- final ranking.

A BUY candidate is therefore a candidate for further review, not a guarantee of future return.

In [ ]:
# ============================================================
# PHASE 13.0 — USER CONFIGURATION
# ============================================================

# >>>> CHỈ CẦN SỬA CELL NÀY MỖI NGÀY NẾU MUỐN ĐỔI WATCHLIST <<<<

MY_STOCKS = ['FPT', 'VCB', 'HPG', 'TCB', 'ACB', 'BID', 'MWG', 'SSI', 'VND', 'CTD', 'VHM', 'VIC', 'VRE', 'PNJ', 'MSN',
                   'GAS', 'PLX', 'SAB', 'STB', 'SHB', 'LPB', 'VPB', 'MBB', 'HDB', 'TPB', 'OCB', 'SSB', 'VIB', 'CTG', 'POW',
                   'GVR', 'DGC', 'DPM', 'BSR', 'PVD', 'PVT', 'VCG', 'VSC', 'HSG', 'NKG', 'BCM', 'KDH', 'DXG', 'NLG', 'TCH',
                   'DIG', 'PDR', 'KBC', 'CRE', 'VPI', 'HDG', 'AGG', 'HT1', 'AAA', 'DBC', 'NVL', 'VHC', 'MSH', 'CII', 'VGI',
                   'ANV', 'SMC', 'HAG', 'DLG', 'BFC', 'PTB', 'TLH', 'PHR', 'DCM', 'LAS', 'VIX', 'MSR', 'GEL', 'HUT', 'GEG',
                   'GEX', 'FTS', 'HCM', 'CTS', 'MBS', 'VCI', 'ORS', 'BSI', 'BAF', 'BMP', 'BVH', 'BWE', 'CMG', 'CTR', 'DGW',
                   'DSE', 'DXS', 'EIB', 'EVF', 'FRT', 'GEE', 'GMD', 'HDC', 'HHV', 'IMP', 'KDC', 'KOS', 'MSB', 'NAB', 'NT2',
                   'PAN', 'PC1', 'REE', 'SBT', 'SCS', 'SIP', 'SJS', 'SZC', 'VGC', 'VJC', 'VNM', 'VPL', 'CEO', 'SHS', 'PVS',
                   'MST', 'VC3', 'IDC', 'VTZ', 'VFS', 'PVC', 'C69', 'TNG', 'VGS', 'KSF', 'PVB', 'BVS', 'CTP', 'FID', 'VTV',
                   'PLC', 'OCH', 'PVI', 'NTP', 'IPA', 'DXP', 'VIW', 'TVN', 'OIL', 'ACV', 'HNG', 'BVB', 'FOX', 'DRI', 'DDV',
                   'AAS', 'VEA', 'TV1', 'ABB', 'VGT', 'QTP', 'MZG', 'PHP', 'SBS', 'STH', 'F88', 'HNM', 'FOC', 'DSH', 'BMS',
                   'QNS', 'TOS', 'XMC', 'BNA', 'VC7', 'VC2', 'DTD', 'UNI', 'NBC', 'NAG', 'CAP', 'PSI', 'VGP', 'PPT', 'PCH',
                   'NVB', 'L40', 'APS', 'VHE', 'API', 'PVG', 'IDJ', 'LDP', 'CMS', 'DST', 'AAV', 'AMS', 'BIG', 'VTD', 'G36',
                   'DSH', 'DDB', 'CLI', 'PIV', 'CLX', 'TTG', 'DVN', 'PXL', 'MPC', 'ABW', 'PGB', 'VBB', 'NCG', 'ALC', 'KCB',
                   'GCF', 'KLB', 'VAB', 'C32', 'CIG', 'HSL', 'RYG', 'VTO', 'HII', 'TCM', 'PPC', 'ELC', 'TCI', 'SHI', 'DAH',
                   'LDG', 'HQC', 'HHP', 'SCR', 'LCG', 'QCG', 'TTH', 'NRC', 'TVC', 'TIG', 'SVN', 'DL1', 'KIP', 'CTX', 'APF',
                   'SBB', 'VNP', 'HVN', 'TLG']

# Benchmark used for market-regime context.
BENCHMARK_SYMBOL = "VNINDEX"

# Optional sector proxies. Keep empty if your current data layer
# does not provide sector-level data.
SECTOR_SYMBOLS = {}

# Historical window required by the feature engine.
LOOKBACK_DAYS = 420

# Scanner thresholds.
SCAN_CFG = {
    # Liquidity
    "min_history_rows": 250,
    "min_price": 5_000.0,
    "min_avg_value_20d": 500_000_000.0,

    # Signal
    "buy_score": 70.0,
    "strong_buy_score": 85.0,
    "watch_score": 58.0,

    # Probability
    "min_p_tp": 0.55,
    "strong_p_tp": 0.65,

    # Exit
    "exit_risk_threshold": 65.0,
    "reduce_risk_threshold": 45.0,

    # Relative strength
    "rs_lookback": 60,

    # Ranking
    "top_n_buy": 10,
    "top_n_watch": 15,
    "top_n_exit": 15,

    # Safety
    "require_market_confirmation": True,
    "allow_new_long_in_bear": False,
}

# Set True when you want verbose per-symbol diagnostics.
VERBOSE_SCAN = False

print("Watchlist:", len(MY_STOCKS), "symbols")
print("Benchmark:", BENCHMARK_SYMBOL)
print("Lookback:", LOOKBACK_DAYS, "calendar/data rows target")

Watchlist: 243 symbols
Benchmark: VNINDEX
Lookback: 420 calendar/data rows target


In [ ]:
# ============================================================
# PHASE 13.1 — PRODUCTION DATA ADAPTER / VNSTOCK CURRENT API
# ============================================================
# Goal:
#   Run All -> fetch VNINDEX once -> resolve latest trading session ->
#   scan the whole watchlist using persistent parquet cache.
#
# IMPORTANT:
#   - No Vnstock().stock() fallback.
#   - Prefer current vnstock.api.quote.Quote.
#   - Also support vnstock_data.Quote if available.
#   - KBS is tried before VCI for Colab/cloud stability.
#   - A single symbol failure never crashes the whole scan.
# ============================================================

import os
import time as _time
from pathlib import Path

try:
    from zoneinfo import ZoneInfo
    _VN_TZ = ZoneInfo("Asia/Ho_Chi_Minh")
except Exception:
    _VN_TZ = None


def _vn_today():
    """Vietnam local calendar date, independent of Colab UTC clock."""
    ts = pd.Timestamp.now(tz=_VN_TZ) if _VN_TZ else pd.Timestamp.now()
    return ts.tz_localize(None).normalize() if getattr(ts, "tzinfo", None) else ts.normalize()


def _persistent_cache_dir():
    """
    Prefer Google Drive so a new Colab runtime can reuse the previous day's
    parquet cache. Fall back to local /content/data/raw.
    """
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        p = drive_root / "Tool_CK_Quant" / "cache" / "raw"
    else:
        p = Path("/content/data/raw")

    p.mkdir(parents=True, exist_ok=True)
    return str(p)


# Create ONE canonical store for the scanner.
SCANNER_STORE = MarketDataStore(
    QuantConfig(
        sources=["KBS", "VCI"],
        delay_sec=0.8
    ),
    cache_dir=_persistent_cache_dir()
)

# Keep conventional names available for compatibility.
store = SCANNER_STORE
market_store = SCANNER_STORE


def _current_quote_class(source):
    """
    Resolve the current quote adapter.

    Current open-source migration path:
        from vnstock.api.quote import Quote

    Some environments expose the newer package as vnstock_data:
        from vnstock_data import Quote
    """
    errors = []

    try:
        from vnstock.api.quote import Quote
        return Quote, "vnstock.api.quote"
    except Exception as exc:
        errors.append(f"vnstock.api.quote: {exc}")

    try:
        from vnstock_data import Quote
        return Quote, "vnstock_data"
    except Exception as exc:
        errors.append(f"vnstock_data: {exc}")

    raise ImportError(
        "No supported current Vnstock Quote adapter was found. "
        "Do not use deprecated Vnstock().stock(). "
        + " | ".join(errors)
    )


def _fetch_v4(self, sym, start, end, src):
    """Override MarketDataStore's legacy _fetch_v4."""
    return _fetch_current_quote(sym, start, end, src)


def _fetch_v3_disabled(self, sym, start, end, src):
    """
    Legacy V3 endpoint intentionally disabled.

    Calling Vnstock().stock() can emit the deprecation warning and can fail
    noisily in Colab. The scanner should fail over between CURRENT sources,
    not between old/new APIs.
    """
    raise RuntimeError(
        "Legacy Vnstock().stock() fallback disabled in production scanner."
    )


# Monkey-patch only the scanner store class; the rest of the research notebook
# remains unchanged.
MarketDataStore._fetch_v4 = _fetch_v4
MarketDataStore._fetch_v3 = _fetch_v3_disabled


def _fetch_with_retry(store_obj, symbol, start_date, end_date,
                      retries=3, delay_sec=1.2):
    """
    Fetch one symbol with bounded exponential backoff.

    Important: retry the store-level fetch, which itself tries KBS -> VCI.
    This avoids a transient network error killing the full 242-symbol run.
    """
    last_error = None

    for attempt in range(retries):
        try:
            df = store_obj.fetch(
                symbol,
                start_date=start_date,
                end_date=end_date,
                use_cache=True
            )

            if df is not None and not df.empty:
                return df, None

            last_error = RuntimeError("empty dataframe returned")

        except Exception as exc:
            last_error = exc

        if attempt < retries - 1:
            _time.sleep(delay_sec * (2 ** attempt))

    return None, repr(last_error)


def _fetch_benchmark_until_latest(store_obj, symbol, lookback_days):
    """
    Fetch benchmark first. Its latest real row becomes the global as_of date.

    This solves the Sunday/weekend problem:
        Sunday -> latest available Friday session.
    """
    today = _vn_today()
    start = today - pd.Timedelta(days=max(int(lookback_days), 420))

    df, err = _fetch_with_retry(
        store_obj,
        symbol,
        start,
        today,
        retries=4,
        delay_sec=1.5
    )

    if df is None:
        raise RuntimeError(
            f"Cannot load benchmark {symbol}. "
            f"Last error: {err}"
        )

    df = normalize_ohlcv(df)

    if len(df) < 220:
        raise RuntimeError(
            f"Benchmark {symbol} returned only {len(df)} rows; "
            f"need >=220 rows for MA200/regime."
        )

    as_of = pd.Timestamp(df["time"].max()).normalize()

    return df, as_of


def _call_data_layer(symbol, lookback_days=420, as_of=None):
    """
    Canonical Phase-13 data access.

    The request end is the benchmark's actual latest session, NOT 'now'.
    Therefore a Sunday run can reuse Friday's cache without refetching it.
    """
    symbol = str(symbol).upper().strip()

    if as_of is None:
        as_of = _vn_today()

    as_of = pd.Timestamp(as_of).normalize() + pd.Timedelta(days=1)

    # 420 calendar days is enough for ~250+ VN trading sessions in normal
    # conditions. Add a safety buffer for holidays/listing gaps.
    start = as_of - pd.Timedelta(
        days=max(int(lookback_days), 420)
    )

    df, err = _fetch_with_retry(
        SCANNER_STORE,
        symbol,
        start,
        as_of,
        retries=3,
        delay_sec=1.0
    )

    if df is None:
        raise RuntimeError(
            f"Unable to load OHLCV for {symbol}: {err}"
        )

    return normalize_ohlcv(df)




def _fetch_current_quote(sym, start, end, src):
    """
    Fetch daily OHLCV using the current Quote adapter and tag the provider.

    KBS quote history is documented by Vnstock as using price values in
    thousands of VND; the scanner's config uses VND, so the clean layer
    converts KBS OHLC prices to VND.
    """
    Quote, adapter_name = _current_quote_class(src)

    last_error = None

    for source in [src, str(src).lower()]:
        try:
            q = Quote(symbol=sym, source=source)
            api_end = (
                pd.Timestamp(end)
                + pd.Timedelta(days=1)
            ).date()

            for kwargs in (
                {
                    "start": str(pd.Timestamp(start).date()),
                    "end": str(api_end),
                    "interval": "1D",
                },
                {
                    "start_date": str(pd.Timestamp(start).date()),
                    "end_date": str(api_end),
                    "interval": "1D",
                },
            ):
                try:
                    df = q.history(**kwargs)

                    if df is not None and not df.empty:
                        df = df.copy()
                        df.attrs["vnstock_source"] = str(source).upper()
                        return df

                except TypeError as exc:
                    last_error = exc
                    continue

        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        f"{adapter_name} failed for {sym}/{src}: {last_error}"
    )


def _production_clean(self, df):
    """
    Normalize provider output to the scanner's canonical VND OHLCV schema.

    No forward/backward fill is performed.
    """
    x = df.copy()

    source = str(
        x.attrs.get("vnstock_source", "")
    ).upper()

    x.columns = [
        str(c).lower().strip()
        for c in x.columns
    ]

    x = x.rename(columns={
        "tradingdate": "time",
        "date": "time",
        "datetime": "time",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
    })

    required = {
        "time",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }

    if not required.issubset(x.columns):
        return None

    x["time"] = pd.to_datetime(
        x["time"],
        errors="coerce"
    ).dt.normalize()

    for col in [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]:
        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    # KBS's historical Quote values are expressed in thousands of VND.
    if source == "KBS":
        for col in ["open", "high", "low", "close"]:
            x[col] = x[col] * 1000

    x = x.dropna(
        subset=[
            "time",
            "open",
            "high",
            "low",
            "close",
        ]
    )

    x = x[
        (x["close"] > 0) &
        (x["high"] >= x["low"])
    ].copy()

    x["volume"] = x["volume"].fillna(0)

    x = (
        x
        .drop_duplicates(
            subset="time",
            keep="last"
        )
        .sort_values("time")
        .reset_index(drop=True)
    )

    return x


MarketDataStore._clean = _production_clean

print("✓ Production Phase 13.1 data adapter loaded.")
print("✓ Persistent cache:", SCANNER_STORE.cache_dir)
print("✓ API priority: current vnstock.api.quote -> vnstock_data")
print("✓ Legacy Vnstock().stock() fallback: DISABLED")


✓ Production Phase 13.1 data adapter loaded.
✓ Persistent cache: /content/data/raw
✓ API priority: current vnstock.api.quote -> vnstock_data
✓ Legacy Vnstock().stock() fallback: DISABLED


In [ ]:
# ============================================================
# PHASE 13.2 — CAUSAL FEATURE ENGINE
# ============================================================

def _safe_num(s):
    return pd.to_numeric(s, errors="coerce")


def build_scanner_features(df, benchmark_df=None):
    """
    Build a compact causal feature set from OHLCV.

    All rolling values use data available on or before the current bar.
    """
    x = normalize_ohlcv(df)

    c = x["close"]
    h = x["high"]
    l = x["low"]
    o = x["open"]
    v = x["volume"]

    # Trend
    x["ret_1d"] = c.pct_change(1)
    x["ret_5d"] = c.pct_change(5)
    x["ret_20d"] = c.pct_change(20)
    x["ret_60d"] = c.pct_change(60)

    x["ma20"] = c.rolling(20, min_periods=20).mean()
    x["ma50"] = c.rolling(50, min_periods=50).mean()
    x["ma200"] = c.rolling(200, min_periods=200).mean()

    x["ma20_slope"] = x["ma20"].pct_change(5)
    x["ma50_slope"] = x["ma50"].pct_change(10)

    x["above_ma20"] = (c > x["ma20"]).astype(int)
    x["above_ma50"] = (c > x["ma50"]).astype(int)
    x["above_ma200"] = (c > x["ma200"]).astype(int)

    # ATR
    prev_close = c.shift(1)
    tr = pd.concat([
        h - l,
        (h - prev_close).abs(),
        (l - prev_close).abs()
    ], axis=1).max(axis=1)

    x["atr14"] = tr.rolling(14, min_periods=14).mean()
    x["atr_pct"] = x["atr14"] / c

    # RSI
    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / 14,
        adjust=False,
        min_periods=14
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / 14,
        adjust=False,
        min_periods=14
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    x["rsi14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()

    x["macd"] = ema12 - ema26
    x["macd_signal"] = x["macd"].ewm(
        span=9,
        adjust=False
    ).mean()
    x["macd_hist"] = x["macd"] - x["macd_signal"]

    # ADX-like trend strength
    up_move = h.diff()
    down_move = -l.diff()

    plus_dm = pd.Series(
        np.where(
            (up_move > down_move) & (up_move > 0),
            up_move,
            0.0
        ),
        index=x.index
    )

    minus_dm = pd.Series(
        np.where(
            (down_move > up_move) & (down_move > 0),
            down_move,
            0.0
        ),
        index=x.index
    )

    atr = x["atr14"].replace(0, np.nan)

    plus_di = 100 * plus_dm.ewm(
        alpha=1 / 14,
        adjust=False
    ).mean() / atr

    minus_di = 100 * minus_dm.ewm(
        alpha=1 / 14,
        adjust=False
    ).mean() / atr

    dx = (
        (plus_di - minus_di).abs() /
        (plus_di + minus_di).replace(0, np.nan)
    ) * 100

    x["adx14"] = dx.ewm(
        alpha=1 / 14,
        adjust=False
    ).mean()

    # Volume / flow
    x["vol_ma20"] = v.rolling(
        20,
        min_periods=20
    ).mean()

    x["volume_ratio"] = v / x["vol_ma20"]

    x["dollar_value"] = c * v
    x["avg_value_20d"] = x["dollar_value"].rolling(
        20,
        min_periods=20
    ).mean()

    # OBV
    direction = np.sign(c.diff()).fillna(0)
    x["obv"] = (direction * v).cumsum()
    x["obv_ma20"] = x["obv"].rolling(
        20,
        min_periods=20
    ).mean()

    # Chaikin Money Flow
    hl_range = (h - l).replace(0, np.nan)
    mfm = ((c - l) - (h - c)) / hl_range
    mfv = mfm * v

    x["cmf20"] = (
        mfv.rolling(20, min_periods=20).sum() /
        v.rolling(20, min_periods=20).sum()
    )

    # Money Flow Index
    typical = (h + l + c) / 3
    raw_money = typical * v
    direction_typical = typical.diff()

    positive_money = raw_money.where(
        direction_typical > 0,
        0
    )
    negative_money = raw_money.where(
        direction_typical < 0,
        0
    ).abs()

    pos_sum = positive_money.rolling(
        14,
        min_periods=14
    ).sum()

    neg_sum = negative_money.rolling(
        14,
        min_periods=14
    ).sum()

    money_ratio = pos_sum / neg_sum.replace(0, np.nan)

    x["mfi14"] = 100 - (
        100 / (1 + money_ratio)
    )

    # Structure / breakout
    x["resistance20"] = h.shift(1).rolling(
        20,
        min_periods=20
    ).max()

    x["support20"] = l.shift(1).rolling(
        20,
        min_periods=20
    ).min()

    x["breakout20"] = (
        c > x["resistance20"]
    ).astype(int)

    x["breakdown20"] = (
        c < x["support20"]
    ).astype(int)

    x["distance_resistance"] = (
        c / x["resistance20"] - 1
    )

    x["distance_support"] = (
        c / x["support20"] - 1
    )

    # Relative strength vs benchmark
    if benchmark_df is not None:
        b = normalize_ohlcv(benchmark_df)[
            ["time", "close"]
        ].rename(columns={"close": "benchmark_close"})

        x = x.merge(
            b,
            on="time",
            how="left"
        )

        x["benchmark_ret_60d"] = (
            x["benchmark_close"].pct_change(60)
        )

        x["relative_strength_60d"] = (
            x["ret_60d"] -
            x["benchmark_ret_60d"]
        )
    else:
        x["benchmark_close"] = np.nan
        x["benchmark_ret_60d"] = np.nan
        x["relative_strength_60d"] = np.nan

    return x


print("✓ Phase 13 feature engine loaded.")

✓ Phase 13 feature engine loaded.


In [ ]:
# ============================================================
# PHASE 13.3 — MARKET REGIME ENGINE
# ============================================================

def detect_market_regime(benchmark_df):
    x = build_scanner_features(benchmark_df)

    if x.empty:
        return {
            "regime": "UNKNOWN",
            "score": np.nan,
            "reason": "No benchmark data"
        }

    r = x.iloc[-1]

    score = 50.0
    reasons = []

    if r["close"] > r["ma20"]:
        score += 10
        reasons.append("VNINDEX > MA20")
    else:
        score -= 10
        reasons.append("VNINDEX < MA20")

    if r["close"] > r["ma50"]:
        score += 10
        reasons.append("VNINDEX > MA50")
    else:
        score -= 10
        reasons.append("VNINDEX < MA50")

    if r["close"] > r["ma200"]:
        score += 15
        reasons.append("VNINDEX > MA200")
    else:
        score -= 15
        reasons.append("VNINDEX < MA200")

    if r["ma20_slope"] > 0:
        score += 5
    else:
        score -= 5

    if r["ma50_slope"] > 0:
        score += 5
    else:
        score -= 5

    if r["cmf20"] > 0:
        score += 5
        reasons.append("market CMF positive")
    else:
        score -= 5

    score = float(np.clip(score, 0, 100))

    if score >= 75:
        regime = "STRONG_BULL"
    elif score >= 60:
        regime = "BULL"
    elif score >= 45:
        regime = "SIDEWAYS"
    elif score >= 30:
        regime = "BEAR"
    else:
        regime = "STRONG_BEAR"

    return {
        "regime": regime,
        "score": score,
        "close": float(r["close"]),
        "ma20": float(r["ma20"]) if pd.notna(r["ma20"]) else np.nan,
        "ma50": float(r["ma50"]) if pd.notna(r["ma50"]) else np.nan,
        "ma200": float(r["ma200"]) if pd.notna(r["ma200"]) else np.nan,
        "reason": "; ".join(reasons)
    }


print("✓ Phase 13 market-regime engine loaded.")

✓ Phase 13 market-regime engine loaded.


In [ ]:
# ============================================================
# PHASE 13.4 — SETUP / FLOW / EXIT-RISK SCORING
# ============================================================

def _clip_score(x):
    return float(np.clip(x, 0, 100))


def score_stock_snapshot(x):
    """
    Transparent rule-based score.

    This is intentionally interpretable and acts as a safety/ranking layer
    around the probability model rather than pretending the hand-built
    score itself is statistically optimal.
    """
    r = x.iloc[-1]

    # -------------------------
    # Trend score
    # -------------------------
    trend = 0.0

    trend += 20 if r["above_ma20"] else 0
    trend += 20 if r["above_ma50"] else 0
    trend += 20 if r["above_ma200"] else 0
    trend += 15 if r["ma20_slope"] > 0 else 0
    trend += 15 if r["ma50_slope"] > 0 else 0
    trend += 10 if r["adx14"] >= 20 else 0
    trend = _clip_score(trend)

    # -------------------------
    # Momentum
    # -------------------------
    momentum = 50.0

    if r["ret_20d"] > 0:
        momentum += 15
    else:
        momentum -= 15

    if r["ret_60d"] > 0:
        momentum += 15
    else:
        momentum -= 15

    if 50 <= r["rsi14"] <= 70:
        momentum += 15
    elif r["rsi14"] > 75:
        momentum -= 10
    elif r["rsi14"] < 40:
        momentum -= 10

    if r["macd_hist"] > 0:
        momentum += 10
    else:
        momentum -= 10

    momentum = _clip_score(momentum)

    # -------------------------
    # Flow / volume
    # -------------------------
    flow = 50.0

    if r["volume_ratio"] >= 1.5:
        flow += 15
    elif r["volume_ratio"] >= 1.1:
        flow += 8
    elif r["volume_ratio"] < 0.7:
        flow -= 8

    if r["cmf20"] > 0.10:
        flow += 20
    elif r["cmf20"] > 0:
        flow += 10
    else:
        flow -= 15

    if r["mfi14"] >= 50:
        flow += 10
    else:
        flow -= 10

    if r["obv"] > r["obv_ma20"]:
        flow += 10
    else:
        flow -= 10

    flow = _clip_score(flow)

    # -------------------------
    # Structure / setup
    # -------------------------
    structure = 50.0
    setup = "NONE"

    if r["breakout20"]:
        structure += 35
        setup = "BREAKOUT"
    elif (
        r["close"] > r["ma20"] and
        r["close"] > r["ma50"] and
        r["ret_5d"] < 0 and
        r["ret_20d"] > 0
    ):
        structure += 20
        setup = "PULLBACK_IN_UPTREND"
    elif (
        r["close"] > r["ma50"] and
        r["ret_20d"] > 0
    ):
        structure += 10
        setup = "UPTREND_CONTINUATION"

    if r["distance_resistance"] < 0.03:
        structure += 5

    if r["breakdown20"]:
        structure -= 40
        setup = "BREAKDOWN"

    structure = _clip_score(structure)

    # -------------------------
    # Relative strength
    # -------------------------
    rs = 50.0

    if pd.notna(r["relative_strength_60d"]):
        if r["relative_strength_60d"] > 0.10:
            rs += 35
        elif r["relative_strength_60d"] > 0.03:
            rs += 20
        elif r["relative_strength_60d"] < -0.10:
            rs -= 35
        elif r["relative_strength_60d"] < -0.03:
            rs -= 20

    rs = _clip_score(rs)

    # -------------------------
    # Composite
    # -------------------------
    alpha_score = (
        0.25 * trend +
        0.20 * momentum +
        0.25 * flow +
        0.15 * structure +
        0.15 * rs
    )

    # -------------------------
    # Exit risk
    # -------------------------
    exit_risk = 0.0
    exit_reasons = []

    if r["close"] < r["ma20"]:
        exit_risk += 15
        exit_reasons.append("price below MA20")

    if r["close"] < r["ma50"]:
        exit_risk += 25
        exit_reasons.append("price below MA50")

    if r["ma20_slope"] < 0:
        exit_risk += 10
        exit_reasons.append("MA20 slope negative")

    if r["macd_hist"] < 0:
        exit_risk += 10
        exit_reasons.append("MACD histogram negative")

    if r["cmf20"] < -0.05:
        exit_risk += 20
        exit_reasons.append("money flow distribution")

    if r["volume_ratio"] >= 1.5 and r["ret_1d"] < 0:
        exit_risk += 10
        exit_reasons.append("high-volume down day")

    if r["breakdown20"]:
        exit_risk += 30
        exit_reasons.append("20D support breakdown")

    # Divergence-like warning:
    # price still rising while money flow is deteriorating.
    if (
        r["ret_20d"] > 0 and
        r["cmf20"] < 0 and
        r["obv"] < r["obv_ma20"]
    ):
        exit_risk += 15
        exit_reasons.append("price-flow divergence")

    exit_risk = _clip_score(exit_risk)

    return {
        "trend_score": trend,
        "momentum_score": momentum,
        "flow_score": flow,
        "structure_score": structure,
        "relative_strength_score": rs,
        "alpha_score": _clip_score(alpha_score),
        "setup": setup,
        "exit_risk": exit_risk,
        "exit_reasons": "; ".join(exit_reasons),
    }


def classify_signal(
    alpha_score,
    p_tp,
    exit_risk,
    market_regime,
    cfg
):
    # Exit dominates a new-buy signal.
    if exit_risk >= cfg["exit_risk_threshold"]:
        return "EXIT"

    if exit_risk >= cfg["reduce_risk_threshold"]:
        if alpha_score < cfg["watch_score"]:
            return "REDUCE"

    # Market gate.
    bear_market = market_regime in {
        "BEAR",
        "STRONG_BEAR"
    }

    if bear_market and not cfg["allow_new_long_in_bear"]:
        if alpha_score >= cfg["watch_score"]:
            return "WATCH"

    if (
        alpha_score >= cfg["buy_score"]
    ):
        if alpha_score >= cfg["strong_p_tp"] * 100:
            return "STRONG_BUY"
        return "BUY"

    if alpha_score >= cfg["watch_score"]:
        return "WATCH"

    return "NO_SIGNAL"


print("✓ Phase 13 signal classifier loaded.")

✓ Phase 13 signal classifier loaded.


In [ ]:
# ============================================================
# PHASE 13.5 — PROBABILITY MODEL ADAPTER
# ============================================================

def _get_live_probability(snapshot_df):
    """
    Use the already-trained ProbabilityModel when available.

    If no compatible trained model exists, return NaN rather than inventing
    a probability. The rule-based Alpha Score still works as a fallback
    ranking layer.
    """
    model_candidates = [
        globals().get("probability_model"),
        globals().get("MODEL"),
        globals().get("model"),
    ]

    model = next(
        (m for m in model_candidates if m is not None),
        None
    )

    if model is None:
        return np.nan

    try:
        p = model.predict_proba(snapshot_df)
        return float(np.asarray(p).reshape(-1)[-1])
    except Exception:
        return np.nan


def combine_probability_and_score(alpha_score, p_tp):
    """
    Do not manufacture a probability when the statistical model is absent.

    If p_tp exists, blend it with interpretable Alpha Score.
    """
    if pd.isna(p_tp):
        return float(alpha_score)

    p_score = float(p_tp) * 100.0

    return float(
        0.60 * alpha_score +
        0.40 * p_score
    )


print("✓ Phase 13 probability adapter loaded.")

✓ Phase 13 probability adapter loaded.


In [ ]:
# ============================================================
# PHASE 13.6 — SINGLE SYMBOL SCAN
# ============================================================

def scan_one_symbol(
    symbol,
    benchmark_df,
    market_info,
    cfg=SCAN_CFG,
    as_of=None
):
    symbol = str(symbol).upper().strip()

    result = {
        "symbol": symbol,
        "status": "ERROR",
        "signal": "ERROR",
        "error": None
    }

    try:
        raw = _call_data_layer(
            symbol,
            lookback_days=LOOKBACK_DAYS,
            as_of=as_of
        )

        df = normalize_ohlcv(raw)

        # Only use information available on the benchmark's latest session.
        if as_of is not None:
            df = df[df["time"] <= pd.Timestamp(as_of).normalize()].copy()

        if df.empty:
            raise RuntimeError("no rows at or before scanner as_of date")

        if len(df) < cfg["min_history_rows"]:
            result.update({
                "status": "FILTERED",
                "signal": "INSUFFICIENT_HISTORY",
                "history_rows": len(df)
            })
            return result

        features = build_scanner_features(
            df,
            benchmark_df=benchmark_df
        )

        r = features.iloc[-1]

        avg_value = float(
            r["avg_value_20d"]
        ) if pd.notna(r["avg_value_20d"]) else 0.0

        last_price = float(r["close"])

        if last_price < cfg["min_price"]:
            result.update({
                "status": "FILTERED",
                "signal": "LOW_PRICE",
                "last_price": last_price,
                "avg_value_20d": avg_value
            })
            return result

        if avg_value < cfg["min_avg_value_20d"]:
            result.update({
                "status": "FILTERED",
                "signal": "LOW_LIQUIDITY",
                "last_price": last_price,
                "avg_value_20d": avg_value
            })
            return result

        scores = score_stock_snapshot(features)

        # The live probability model should receive exactly the same feature
        # contract used during model training. If it cannot do so, p_tp stays NaN.
        p_tp = _get_live_probability(
            features.tail(1)
        )

        combined = combine_probability_and_score(
            scores["alpha_score"],
            p_tp
        )

        signal = classify_signal(
            scores["alpha_score"],
            p_tp if pd.notna(p_tp) else scores["alpha_score"] / 100,
            scores["exit_risk"],
            market_info["regime"],
            cfg
        )

        # Price-based stop reference.
        atr_pct = (
            float(r["atr_pct"])
            if pd.notna(r["atr_pct"])
            else 0.05
        )

        suggested_sl_pct = float(
            np.clip(1.5 * atr_pct, 0.03, 0.12)
        )

        suggested_tp_pct = float(
            np.clip(2.0 * suggested_sl_pct, 0.06, 0.25)
        )

        # Human-readable reasons.
        reasons = []

        if scores["trend_score"] >= 70:
            reasons.append("trend strong")
        if scores["flow_score"] >= 70:
            reasons.append("money flow/volume supportive")
        if scores["structure_score"] >= 70:
            reasons.append(f"setup={scores['setup']}")
        if scores["relative_strength_score"] >= 70:
            reasons.append("relative strength strong")
        if r["volume_ratio"] >= 1.5:
            reasons.append(
                f"volume {r['volume_ratio']:.1f}x average"
            )

        warnings = []

        if scores["exit_risk"] >= cfg["reduce_risk_threshold"]:
            warnings.append(
                scores["exit_reasons"]
            )

        if r["rsi14"] > 75:
            warnings.append("RSI overbought")

        if (
            r["ret_20d"] > 0 and
            r["cmf20"] < 0
        ):
            warnings.append("price-flow divergence")

        result.update({
            "status": "OK",
            "signal": signal,
            "date": r["time"],
            "last_price": last_price,
            "history_rows": len(df),
            "avg_value_20d": avg_value,

            "alpha_score": round(scores["alpha_score"], 2),
            "combined_score": round(combined, 2),

            "p_tp": (
                round(float(p_tp), 4)
                if pd.notna(p_tp)
                else np.nan
            ),

            "trend_score": round(scores["trend_score"], 2),
            "momentum_score": round(scores["momentum_score"], 2),
            "flow_score": round(scores["flow_score"], 2),
            "structure_score": round(scores["structure_score"], 2),
            "relative_strength_score": round(
                scores["relative_strength_score"], 2
            ),

            "setup": scores["setup"],
            "exit_risk": round(scores["exit_risk"], 2),

            "rsi14": round(float(r["rsi14"]), 2),
            "adx14": round(float(r["adx14"]), 2),
            "volume_ratio": round(float(r["volume_ratio"]), 2),
            "cmf20": round(float(r["cmf20"]), 3),
            "ret_20d": round(float(r["ret_20d"]) * 100, 2),
            "ret_60d": round(float(r["ret_60d"]) * 100, 2),

            "suggested_sl_pct": round(
                suggested_sl_pct * 100,
                2
            ),
            "suggested_tp_pct": round(
                suggested_tp_pct * 100,
                2
            ),

            "reasons": "; ".join(reasons),
            "warnings": "; ".join(
                [w for w in warnings if w]
            ),
            "market_regime": market_info["regime"],
            "market_score": market_info["score"],
        })

        return result

    except Exception as exc:
        result.update({
            "status": "ERROR",
            "signal": "ERROR",
            "error": f"{type(exc).__name__}: {exc}"
        })

        if VERBOSE_SCAN:
            import traceback
            traceback.print_exc()

        return result


print("✓ Phase 13 single-symbol scanner loaded.")

✓ Phase 13 single-symbol scanner loaded.


In [ ]:
# ============================================================
# PHASE 13.7 — PRODUCTION WATCHLIST SCANNER
# ============================================================

def scan_watchlist(
    symbols,
    benchmark_symbol=BENCHMARK_SYMBOL,
    cfg=SCAN_CFG
):
    # De-duplicate while preserving user's order.
    symbols = list(dict.fromkeys(
        str(s).upper().strip()
        for s in symbols
        if str(s).strip()
    ))

    started = pd.Timestamp.now(tz=_VN_TZ) if _VN_TZ else pd.Timestamp.now()

    print("=" * 90)
    print("DAILY STOCK SCANNER — PRODUCTION")
    print("=" * 90)
    print("Requested symbols :", len(symbols))
    print("Started (VN)      :", started)
    print("Cache             :", SCANNER_STORE.cache_dir)

    # --------------------------------------------------------
    # 1. Benchmark first — determines the actual trading as_of.
    # --------------------------------------------------------
    benchmark_df, as_of = _fetch_benchmark_until_latest(
        SCANNER_STORE,
        benchmark_symbol,
        LOOKBACK_DAYS
    )

    # Ensure benchmark contains only data up to as_of.
    benchmark_df = benchmark_df[
        benchmark_df["time"] <= as_of
    ].copy()

    market_info = detect_market_regime(
        benchmark_df
    )

    market_info["as_of"] = as_of

    print(
        f"\nAS-OF TRADING SESSION: {as_of.date()}"
    )
    print(
        f"MARKET REGIME: {market_info['regime']} "
        f"(score={market_info['score']:.1f})"
    )

    results = []
    success = 0
    filtered_count = 0
    error_count = 0

    # --------------------------------------------------------
    # 2. Scan every symbol independently.
    #
    # A failure is converted to an ERROR row. It never crashes
    # the entire scanner.
    # --------------------------------------------------------
    for i, symbol in enumerate(symbols, start=1):

        if VERBOSE_SCAN:
            print(f"[{i:03d}/{len(symbols):03d}] {symbol}")

        result = scan_one_symbol(
            symbol=symbol,
            benchmark_df=benchmark_df,
            market_info=market_info,
            cfg=cfg,
            as_of=as_of
        )

        results.append(result)

        if result.get("status") == "OK":
            success += 1
        elif result.get("status") == "FILTERED":
            filtered_count += 1
        else:
            error_count += 1

        # Small pacing delay protects public endpoints when cache misses occur.
        # Do not sleep when verbose=False and the result came from cache.
        if SCANNER_STORE.cfg.delay_sec > 0:
            _time.sleep(float(SCANNER_STORE.cfg.delay_sec))

        if i % 25 == 0 or i == len(symbols):
            print(
                f"Progress {i}/{len(symbols)} | "
                f"OK={success} | FILTERED={filtered_count} | ERROR={error_count}"
            )

    result_df = pd.DataFrame(results)

    if result_df.empty:
        empty = result_df.copy()
        return {
            "all": empty,
            "buy": empty,
            "watch": empty,
            "exit": empty,
            "filtered": empty,
            "errors": empty,
            "market": market_info,
            "as_of": as_of,
            "requested": len(symbols),
        }

    # Numeric normalization for deterministic sorting.
    for c in [
        "combined_score",
        "alpha_score",
        "exit_risk",
        "p_tp",
        "avg_value_20d"
    ]:
        if c in result_df.columns:
            result_df[c] = pd.to_numeric(
                result_df[c],
                errors="coerce"
            )

    # --------------------------------------------------------
    # BUY
    # --------------------------------------------------------
    buy = result_df[
        result_df["signal"].isin(["STRONG_BUY", "BUY"])
        & result_df["status"].eq("OK")
    ].sort_values(
        ["combined_score", "alpha_score", "exit_risk"],
        ascending=[False, False, True],
        na_position="last"
    ).head(cfg["top_n_buy"])

    # --------------------------------------------------------
    # WATCH
    # --------------------------------------------------------
    watch = result_df[
        result_df["signal"].eq("WATCH")
        & result_df["status"].eq("OK")
    ].sort_values(
        ["combined_score", "alpha_score", "exit_risk"],
        ascending=[False, False, True],
        na_position="last"
    ).head(cfg["top_n_watch"])

    # --------------------------------------------------------
    # EXIT / REDUCE
    # --------------------------------------------------------
    exit_df = result_df[
        result_df["signal"].isin(["EXIT", "REDUCE"])
        & result_df["status"].eq("OK")
    ].sort_values(
        ["exit_risk", "combined_score"],
        ascending=[False, True],
        na_position="last"
    ).head(cfg["top_n_exit"])

    filtered = result_df[
        result_df["status"].eq("FILTERED")
    ].copy()

    errors = result_df[
        result_df["status"].eq("ERROR")
    ].copy()

    # Stable final ordering for reproducible CSVs.
    result_df = result_df.sort_values(
        ["status", "combined_score", "symbol"],
        ascending=[True, False, True],
        na_position="last"
    ).reset_index(drop=True)

    print("\n" + "=" * 90)
    print("SCAN COMPLETE")
    print("=" * 90)
    print(f"As-of session : {as_of.date()}")
    print(f"Requested     : {len(symbols)}")
    print(f"OK            : {len(result_df[result_df['status'].eq('OK')])}")
    print(f"FILTERED      : {len(filtered)}")
    print(f"ERROR         : {len(errors)}")
    print(f"BUY           : {len(buy)}")
    print(f"WATCH         : {len(watch)}")
    print(f"EXIT/REDUCE   : {len(exit_df)}")

    return {
        "all": result_df,
        "buy": buy,
        "watch": watch,
        "exit": exit_df,
        "filtered": filtered,
        "errors": errors,
        "market": market_info,
        "as_of": as_of,
        "requested": len(symbols),
    }


print("✓ Production Phase 13.7 watchlist scanner loaded.")


✓ Production Phase 13.7 watchlist scanner loaded.


In [ ]:
# ============================================================
# PHASE 13.8 — DAILY REPORT
# ============================================================

def _display_columns(df):
    preferred = [
        "symbol", "signal", "combined_score",
        "alpha_score", "p_tp",
        "exit_risk", "setup",
        "last_price", "volume_ratio",
        "cmf20", "rsi14",
        "ret_20d", "ret_60d",
        "market_regime",
        "reasons", "warnings"
    ]

    return [
        c for c in preferred
        if c in df.columns
    ]


def print_daily_report(scan):
    market = scan["market"]

    print("\n" + "=" * 90)
    print("📊 DAILY QUANT STOCK REPORT")
    print("=" * 90)

    print(
        f"Market: {market['regime']} | "
        f"Score: {market['score']:.1f}"
    )

    as_of = scan.get("as_of", market.get("as_of"))
    print(
        "Trading session:",
        pd.Timestamp(as_of).strftime("%Y-%m-%d")
        if as_of is not None else "UNKNOWN"
    )

    print("\n🟢 BUY / STRONG BUY")
    print("-" * 90)

    if scan["buy"].empty:
        print("No BUY candidate passed the current filters.")
    else:
        display(
            scan["buy"][
                _display_columns(scan["buy"])
            ].reset_index(drop=True)
        )

    print("\n🟡 WATCH")
    print("-" * 90)

    if scan["watch"].empty:
        print("No WATCH candidate.")
    else:
        display(
            scan["watch"][
                _display_columns(scan["watch"])
            ].reset_index(drop=True)
        )

    print("\n🔴 EXIT / REDUCE — POSSIBLE UPTREND DETERIORATION")
    print("-" * 90)

    if scan["exit"].empty:
        print("No elevated exit-risk candidate.")
    else:
        display(
            scan["exit"][
                _display_columns(scan["exit"])
            ].reset_index(drop=True)
        )

    print("\n⚠ FILTERED")
    print("-" * 90)

    if scan["filtered"].empty:
        print("None.")
    else:
        cols = [
            c for c in [
                "symbol",
                "signal",
                "last_price",
                "avg_value_20d",
                "history_rows"
            ]
            if c in scan["filtered"].columns
        ]
        display(
            scan["filtered"][cols].reset_index(drop=True)
        )

    print("\n❌ ERRORS")
    print("-" * 90)

    if scan["errors"].empty:
        print("None.")
    else:
        display(
            scan["errors"][
                [c for c in ["symbol", "error"] if c in scan["errors"]]
            ].reset_index(drop=True)
        )

    print("\n" + "=" * 90)
    print("Scanner completed.")
    print("=" * 90)


def save_daily_report(scan, output_dir="/content/stock_scanner_reports"):
    """
    Save a reproducible CSV report.

    Also works in local Jupyter; /content is simply the conventional Colab path.
    """
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

    paths = {}

    mapping = {
        "all": "all",
        "buy": "buy",
        "watch": "watch",
        "exit": "exit_risk",
        "filtered": "filtered",
        "errors": "errors",
    }

    for key, suffix in mapping.items():
        path = out_dir / f"scanner_{suffix}_{stamp}.csv"
        scan[key].to_csv(
            path,
            index=False,
            encoding="utf-8-sig"
        )
        paths[key] = str(path)

    summary_path = out_dir / f"scanner_summary_{stamp}.txt"

    market = scan["market"]

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("DAILY QUANT STOCK SCANNER\n")
        f.write("=" * 60 + "\n")
        f.write(
            f"As-of: {pd.Timestamp.now()}\n"
        )
        f.write(
            f"Market regime: {market['regime']}\n"
        )
        f.write(
            f"Market score: {market['score']:.2f}\n\n"
        )

        f.write(
            f"BUY candidates: {len(scan['buy'])}\n"
        )
        f.write(
            f"WATCH candidates: {len(scan['watch'])}\n"
        )
        f.write(
            f"EXIT/REDUCE candidates: {len(scan['exit'])}\n"
        )
        f.write(
            f"Filtered: {len(scan['filtered'])}\n"
        )
        f.write(
            f"Errors: {len(scan['errors'])}\n"
        )

    paths["summary"] = str(summary_path)

    return paths


print("✓ Phase 13 reporting/export loaded.")

✓ Phase 13 reporting/export loaded.


In [ ]:
# ============================================================
# HOTFIX — NORMALIZE OHLCV
# MUST RUN BEFORE "DAILY STOCK SCANNER — PRODUCTION"
# ============================================================

import pandas as pd
import numpy as np


def normalize_ohlcv(df, source=None):
    """
    Normalize OHLCV dataframe from vnstock / KBS / VCI
    into the canonical schema used by the scanner:

        time
        open
        high
        low
        close
        volume

    Also handles common vnstock column naming variations.
    """

    if df is None:
        raise ValueError("OHLCV dataframe is None")

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"OHLCV must be pandas.DataFrame, got {type(df)}"
        )

    if df.empty:
        raise ValueError("OHLCV dataframe is empty")

    x = df.copy()

    # --------------------------------------------------------
    # Normalize column names
    # --------------------------------------------------------
    x.columns = [
        str(c).strip()
        for c in x.columns
    ]

    # Map lower-case column names -> actual column names
    col_map = {
        str(c).strip().lower(): c
        for c in x.columns
    }

    def find_column(candidates):
        for candidate in candidates:
            key = candidate.lower()

            if key in col_map:
                return col_map[key]

        return None

    # --------------------------------------------------------
    # TIME / DATE
    # --------------------------------------------------------
    time_col = find_column([
        "time",
        "date",
        "datetime",
        "dt",
        "trading_date",
        "tradingdate",
        "timestamp",
    ])

    # Some vnstock responses may use "TradingDate"
    if time_col is None:
        time_col = find_column([
            "trading_date",
            "TradingDate",
            "Date",
            "Time",
        ])

    # --------------------------------------------------------
    # OHLCV
    # --------------------------------------------------------
    open_col = find_column([
        "open",
        "Open",
        "o",
    ])

    high_col = find_column([
        "high",
        "High",
        "h",
    ])

    low_col = find_column([
        "low",
        "Low",
        "l",
    ])

    close_col = find_column([
        "close",
        "Close",
        "c",
        "price",
    ])

    volume_col = find_column([
        "volume",
        "Volume",
        "vol",
        "Vol",
        "v",
    ])

    # --------------------------------------------------------
    # If Date/Time is stored as index
    # --------------------------------------------------------
    if time_col is None:

        if isinstance(
            x.index,
            pd.DatetimeIndex
        ):
            x = x.reset_index()

            # Usually reset_index() creates "index"
            time_col = find_column([
                "time",
                "date",
                "datetime",
                "dt",
                "trading_date",
                "tradingdate",
                "timestamp",
                "index",
            ])

    # --------------------------------------------------------
    # Validate columns
    # --------------------------------------------------------
    missing = []

    if time_col is None:
        missing.append("time")

    if open_col is None:
        missing.append("open")

    if high_col is None:
        missing.append("high")

    if low_col is None:
        missing.append("low")

    if close_col is None:
        missing.append("close")

    if volume_col is None:
        missing.append("volume")

    if missing:

        raise ValueError(
            "Unable to normalize OHLCV. "
            f"Missing columns: {missing}. "
            f"Available columns: {list(x.columns)}"
        )

    # --------------------------------------------------------
    # Rename to canonical names
    # --------------------------------------------------------
    rename_map = {
        time_col: "time",
        open_col: "open",
        high_col: "high",
        low_col: "low",
        close_col: "close",
        volume_col: "volume",
    }

    x = x.rename(
        columns=rename_map
    )

    # --------------------------------------------------------
    # Convert time
    # --------------------------------------------------------
    x["time"] = pd.to_datetime(
        x["time"],
        errors="coerce"
    )

    # Remove timezone if present.
    #
    # We only care about trading date for this EOD scanner.
    try:
        if hasattr(
            x["time"].dt,
            "tz"
        ):
            if x["time"].dt.tz is not None:
                x["time"] = (
                    x["time"]
                    .dt
                    .tz_localize(None)
                )
    except Exception:
        pass

    x["time"] = x["time"].dt.normalize()

    # --------------------------------------------------------
    # Convert numeric columns
    # --------------------------------------------------------
    for col in [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]:

        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # KBS price unit normalization
    #
    # KBS historical OHLC data may be represented in
    # thousand VND.
    #
    # We only multiply prices when source is explicitly KBS.
    # Never blindly multiply unknown sources.
    # --------------------------------------------------------
    if source is not None:

        source_upper = str(
            source
        ).upper()

        if source_upper == "KBS":

            # Detect whether values appear to be in
            # thousand VND rather than VND.
            #
            # Typical FPT:
            #   150  -> 150,000 VND
            #
            # Avoid multiplying if already clearly VND.
            median_close = (
                x["close"]
                .dropna()
                .median()
            )

            if (
                pd.notna(median_close)
                and median_close > 0
                and median_close < 1000
            ):

                for col in [
                    "open",
                    "high",
                    "low",
                    "close",
                ]:
                    x[col] = (
                        x[col] * 1000.0
                    )

    # --------------------------------------------------------
    # Remove invalid rows
    # --------------------------------------------------------
    x = x.dropna(
        subset=[
            "time",
            "open",
            "high",
            "low",
            "close",
            "volume",
        ]
    )

    # --------------------------------------------------------
    # Basic OHLC sanity checks
    # --------------------------------------------------------
    x = x[
        (x["open"] > 0)
        & (x["high"] > 0)
        & (x["low"] > 0)
        & (x["close"] > 0)
        & (x["volume"] >= 0)
    ]

    # High must be >= low
    x = x[
        x["high"] >= x["low"]
    ]

    # High should contain O/C
    x = x[
        (x["high"] >= x["open"])
        & (x["high"] >= x["close"])
    ]

    # Low should be <= O/C
    x = x[
        (x["low"] <= x["open"])
        & (x["low"] <= x["close"])
    ]

    # --------------------------------------------------------
    # Sort and deduplicate
    # --------------------------------------------------------
    x = (
        x
        .sort_values("time")
        .drop_duplicates(
            subset=["time"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Final canonical column order
    # --------------------------------------------------------
    canonical_columns = [
        "time",
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]

    # Keep canonical columns first.
    other_columns = [
        c
        for c in x.columns
        if c not in canonical_columns
    ]

    x = x[
        canonical_columns
        + other_columns
    ]

    if x.empty:
        raise ValueError(
            "OHLCV became empty after normalization "
            "and validation."
        )

    return x


print("=" * 80)
print("✓ normalize_ohlcv() HOTFIX LOADED")
print("=" * 80)
print("Canonical columns:")
print([
    "time",
    "open",
    "high",
    "low",
    "close",
    "volume",
])

✓ normalize_ohlcv() HOTFIX LOADED
Canonical columns:
['time', 'open', 'high', 'low', 'close', 'volume']


In [ ]:
# ============================================================
# DEBUG ONE SYMBOL — CHECK PRICE / LIQUIDITY UNIT
# ============================================================

test_symbol = "FPT"

raw = _call_data_layer(
    test_symbol,
    lookback_days=LOOKBACK_DAYS,
    as_of=pd.Timestamp("2026-08-10")
)

df_test = normalize_ohlcv(raw)

df_test = df_test[
    df_test["time"] <= pd.Timestamp("2026-08-10")
].copy()

features_test = build_scanner_features(
    df_test,
    benchmark_df=None
)

r = features_test.iloc[-1]

print("=" * 80)
print("UNIT DEBUG:", test_symbol)
print("=" * 80)

print("Date           :", r["time"])
print("Raw close      :", r["close"])
print("Volume         :", r["volume"])
print("Avg value 20d  :", r["avg_value_20d"])

print()
print("Current scanner price:")
print("close / 1000   :", float(r["close"]) / 1000)

print()
print("Threshold:")
print("min_price      :", SCAN_CFG["min_price"])
print("min_avg_value  :", SCAN_CFG["min_avg_value_20d"])


📋 Connecting Google Drive account
to save project settings.



ERROR:vnstock.core.config.ggcolab:Error mounting Drive: mount failed


UNIT DEBUG: FPT
Date           : 2026-08-10 00:00:00
Raw close      : 71800.0
Volume         : 5252100
Avg value 20d  : 525791589000.0

Current scanner price:
close / 1000   : 71.8

Threshold:
min_price      : 5000.0
min_avg_value  : 500000000.0


In [ ]:
# ============================================================
# PHASE 13.9 — ONE-CELL DAILY RUN
# ============================================================
# NORMAL DAILY USE:
#     Colab -> Runtime -> Run all
#
# The scanner:
#   1) loads VNINDEX
#   2) resolves latest real trading session
#   3) scans the complete watchlist
#   4) converts individual API failures into ERROR rows
#   5) prints BUY / WATCH / EXIT
#   6) saves CSV reports
# ============================================================

daily_scan = None

try:
    daily_scan = scan_watchlist(
        symbols=MY_STOCKS,
        benchmark_symbol=BENCHMARK_SYMBOL,
        cfg=SCAN_CFG
    )

    print_daily_report(
        daily_scan
    )

    try:
        report_paths = save_daily_report(
            daily_scan
        )

        print("\nSaved reports:")
        for k, path in report_paths.items():
            print(f"  {k:10s}: {path}")

    except Exception as exc:
        print(
            "\n⚠ Report export warning:",
            repr(exc)
        )

except Exception as exc:
    # Only fatal conditions reach here: e.g. benchmark cannot be loaded.
    print("\n" + "=" * 90)
    print("❌ FATAL SCANNER ERROR")
    print("=" * 90)
    print(type(exc).__name__ + ":", exc)
    print(
        "\nThe scanner stopped before symbol-level scanning because the "
        "benchmark could not establish a valid trading session."
    )
    print(
        "Check VNSTOCK_API_KEY, current vnstock package, network access, "
        "and the benchmark data source."
    )
    raise


DAILY STOCK SCANNER — PRODUCTION
Requested symbols : 242
Started (VN)      : 2026-09-08 21:17:03.905312+07:00
Cache             : /content/data/raw

AS-OF TRADING SESSION: 2026-09-08
MARKET REGIME: STRONG_BULL (score=90.0)

📋 Kết nối tài khoản Google Drive để lưu các thiết lập của dự án.
Dữ liệu phiên làm việc với Colab của bạn sẽ bị xóa nếu không lưu trữ vào Google Drive.



ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.




⚠️ 
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 17 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 17 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia chương trình tài trợ dự án (Sponsor) để sử dụng không gián đoạn.
      Lưu ý: vnstock là công cụ mã nguồn mở giúp tự động hoá kết nối với
      các API công khai mà bạn vốn đã có quyền truy cập hợp lệ, không
      phải nhà cung cấp dữ liệu. Việc tài trợ giúp duy trì
      nghiên cứu - phát triển công cụ và hạ tầng công nghệ phục vụ cộng đồng.

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 request/phút - Community):
     Đăng ký API key miễn phí: https://vnstocks.com/login
   • Gói thành viên tài trợ (180-600 request/phút - Sponsor):
     

TypeError: object of type 'NoneType' has no len()

# PHASE 13.10 — Reading the result correctly

## 🟢 BUY / STRONG BUY

These are **research candidates**, not automatic buy orders.

A good candidate should ideally have:

- strong trend,
- supportive money flow,
- healthy volume,
- acceptable structure,
- relative strength,
- acceptable market regime,
- acceptable exit risk,
- and, when the trained probability model is available, sufficiently high `p_tp`.

## 🟡 WATCH

The stock is interesting but one or more conditions are not strong enough.

Typical examples:

- trend is improving but breakout has not happened,
- pullback is healthy but confirmation is missing,
- market regime is weak,
- probability is borderline,
- flow has not confirmed price.

## 🔴 EXIT / REDUCE

This is specifically designed for the second half of your objective:

> Detect when an existing uptrend is deteriorating.

The strongest warnings are:

- price below MA20/MA50,
- negative MA20 slope,
- MACD deterioration,
- money-flow distribution,
- high-volume down day,
- support breakdown,
- price-flow divergence.

## ⚠ Important limitation

The scanner's hand-built score is a **ranking/risk layer**. It should not be presented as a statistically calibrated probability.

`p_tp` is only meaningful when the ProbabilityModel from the earlier research pipeline has been trained with the same feature contract and then frozen for live use.

If no compatible trained model is available, the scanner intentionally displays `p_tp = NaN` instead of fabricating a probability.

In [ ]:
# ============================================================
# PHASE 13.11 — NOTEBOOK SELF-CHECK
# ============================================================

def phase13_self_check():
    required = [
        "MY_STOCKS",
        "SCAN_CFG",
        "scan_one_symbol",
        "scan_watchlist",
        "print_daily_report",
        "save_daily_report",
        "SCANNER_STORE",
        "_fetch_current_quote",
        "_fetch_benchmark_until_latest",
    ]

    missing = [
        name for name in required
        if name not in globals()
    ]

    if missing:
        raise RuntimeError(
            f"Phase 13 missing objects: {missing}"
        )

    assert isinstance(MY_STOCKS, list)
    assert len(MY_STOCKS) > 0

    print("✓ Phase 13 self-check passed")
    print(f"✓ Watchlist size: {len(MY_STOCKS)}")
    print("✓ Daily Run-All scanner is ready")


phase13_self_check()